## Import data


In [2]:
import os
import getpass
import random
from collections import defaultdict
from itertools import combinations

import numpy as np
import pandas as pd
import dask.dataframe as dd
import seaborn as sns
import matplotlib.pyplot as plt

from sqlalchemy import create_engine

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, silhouette_score
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.neighbors import NearestNeighbors


In [3]:
pd.set_option('display.max_rows', None)  # Show all rows
pd.set_option('display.max_columns', None)  # Show all columns
pd.set_option('display.width', None)  # Avoid truncation of wide output
pd.set_option('display.max_colwidth', None)  # Avoid truncation of column content

In [ ]:
# Connection details
host = "172.20.20.4"
port = "5432"
database = "nos"
user = input("Username: ")
password = getpass.getpass("Insert Password:")

In [4]:
# Create a SQLAlchemy engine
engine = create_engine(f'postgresql+psycopg2://{user}:{password}@{host}:{port}/{database}')

list_1 = [
    'masterdataclients_jan_may',
    'masterdataclients_jun_oct',
    'mastercalls_jan_feb',
    'mastercalls_mar_apr',
    'mastercalls_may_jul',
    'mastercalls_aug_oct',
    'masterdatagcs'
    ]
# Function to fetch data from a specific table
def fetch_table(table_name):
    query = f"SELECT * FROM nos2425.{table_name}  ;"
    return pd.read_sql_query(query, engine)

# Loop through the list, fetch each table, and assign it to a variable with the table name
for table_name in list_1:
    df = fetch_table(table_name)  # Fetch the table data
    
    # Dynamically create a variable with the table name to store the data
    globals()[table_name] = df
    

In [ ]:
client_df = pd.concat([masterdataclients_jan_may, masterdataclients_jun_oct], ignore_index=True)
call_df = pd.concat([mastercalls_jan_feb, mastercalls_mar_apr, mastercalls_may_jul, mastercalls_aug_oct], ignore_index=True)


## Data Cleaning


### GC Cleaning

In [6]:
masterdatagcs.head()

,Column1,RESOURCE_KEY,RESOURCE_ID,LEG_START_TIME,COUNT_CALLS,MEAN_RPC,MEDIAN_RPC,MEAN_FTR,MEAN_TMC,MEDIAN_TMC,TOTAL_OTS,OTS_BY_CALL,COUNT_CALLS_IF,MEAN_RPC_IF,MEDIAN_RPC_IF,MEAN_FTR_IF,MEAN_TMC_IF,MEDIAN_TMC_IF,TOTAL_OTS_IF,OTS_BY_CALL_IF,COUNT_CALLS_OTHER,MEAN_RPC_OTHER,MEDIAN_RPC_OTHER,MEAN_FTR_OTHER,MEAN_TMC_OTHER,MEDIAN_TMC_OTHER,TOTAL_OTS_OTHER,OTS_BY_CALL_OTHER,COUNT_CALLS_VF,MEAN_RPC_VF,MEDIAN_RPC_VF,MEAN_FTR_VF,MEAN_TMC_VF,MEDIAN_TMC_VF,TOTAL_OTS_VF,OTS_BY_CALL_VF,COUNT_CALLS_TV,MEAN_RPC_TV,MEDIAN_RPC_TV,MEAN_FTR_TV,MEAN_TMC_TV,MEDIAN_TMC_TV,TOTAL_OTS_TV,OTS_BY_CALL_TV,COUNT_CALLS_MOVEL,MEAN_RPC_MOVEL,MEDIAN_RPC_MOVEL,MEAN_FTR_MOVEL,MEAN_TMC_MOVEL,MEDIAN_TMC_MOVEL,TOTAL_OTS_MOVEL,OTS_BY_CALL_MOVEL,COUNT_LEGS_14_DAYS,MEAN_DURATION_14_DAYS,MEDIAN_DURATION_14_DAYS,MEAN_RPC_14_DAYS,MEDIAN_RPC_14_DAYS,MEAN_FTR_14_DAYS,TOTAL_OTS_14_DAYS,COUNT_LEGS_30_DAYS,MEAN_DURATION_30_DAYS,MEDIAN_DURATION_30_DAYS,MEAN_RPC_30_DAYS,MEDIAN_RPC_30_DAYS,MEAN_FTR_30_DAYS,TOTAL_OTS_30_DAYS,COUNT_LEGS_IF_14_DAYS,MEAN_DURATION_IF_14_DAYS,MEDIAN_DURATION_IF_14_DAYS,MEAN_RPC_IF_14_DAYS,MEDIAN_RPC_IF_14_DAYS,MEAN_FTR_IF_14_DAYS,TOTAL_OTS_IF_14_DAYS,COUNT_LEGS_MOVEL_14_DAYS,MEAN_DURATION_MOVEL_14_DAYS,MEDIAN_DURATION_MOVEL_14_DAYS,MEAN_RPC_MOVEL_14_DAYS,MEDIAN_RPC_MOVEL_14_DAYS,MEAN_FTR_MOVEL_14_DAYS,TOTAL_OTS_MOVEL_14_DAYS,COUNT_LEGS_TV_14_DAYS,MEAN_DURATION_TV_14_DAYS,MEDIAN_DURATION_TV_14_DAYS,MEAN_RPC_TV_14_DAYS,MEDIAN_RPC_TV_14_DAYS,MEAN_FTR_TV_14_DAYS,TOTAL_OTS_TV_14_DAYS,COUNT_LEGS_VF_14_DAYS,MEAN_DURATION_VF_14_DAYS,MEDIAN_DURATION_VF_14_DAYS,MEAN_RPC_VF_14_DAYS,MEDIAN_RPC_VF_14_DAYS,MEAN_FTR_VF_14_DAYS,TOTAL_OTS_VF_14_DAYS,COUNT_LEGS_IF_30_DAYS,MEAN_DURATION_IF_30_DAYS,MEDIAN_DURATION_IF_30_DAYS,MEAN_RPC_IF_30_DAYS,MEDIAN_RPC_IF_30_DAYS,MEAN_FTR_IF_30_DAYS,TOTAL_OTS_IF_30_DAYS,COUNT_LEGS_MOVEL_30_DAYS,MEAN_DURATION_MOVEL_30_DAYS,MEDIAN_DURATION_MOVEL_30_DAYS,MEAN_RPC_MOVEL_30_DAYS,MEDIAN_RPC_MOVEL_30_DAYS,MEAN_FTR_MOVEL_30_DAYS,TOTAL_OTS_MOVEL_30_DAYS,COUNT_LEGS_TV_30_DAYS,MEAN_DURATION_TV_30_DAYS,MEDIAN_DURATION_TV_30_DAYS,MEAN_RPC_TV_30_DAYS,MEDIAN_RPC_TV_30_DAYS,MEAN_FTR_TV_30_DAYS,TOTAL_OTS_TV_30_DAYS,COUNT_LEGS_VF_30_DAYS,MEAN_DURATION_VF_30_DAYS,MEDIAN_DURATION_VF_30_DAYS,MEAN_RPC_VF_30_DAYS,MEDIAN_RPC_VF_30_DAYS,MEAN_FTR_VF_30_DAYS,TOTAL_OTS_VF_30_DAYS,MEAN_HANDOVERS_14_DAYS,MEDIAN_HANDOVERS_14_DAYS,MEAN_HANDOVERS_30_DAYS,MEDIAN_HANDOVERS_30_DAYS,MEAN_HANDOVERS_IF_14_DAYS,MEDIAN_HANDOVERS_IF_14_DAYS,MEAN_HANDOVERS_MOVEL_14_DAYS,MEDIAN_HANDOVERS_MOVEL_14_DAYS,MEAN_HANDOVERS_TV_14_DAYS,MEDIAN_HANDOVERS_TV_14_DAYS,MEAN_HANDOVERS_VF_14_DAYS,MEDIAN_HANDOVERS_VF_14_DAYS,MEAN_HANDOVERS_IF_30_DAYS,MEDIAN_HANDOVERS_IF_30_DAYS,MEAN_HANDOVERS_MOVEL_30_DAYS,MEDIAN_HANDOVERS_MOVEL_30_DAYS,MEAN_HANDOVERS_TV_30_DAYS,MEDIAN_HANDOVERS_TV_30_DAYS,MEAN_HANDOVERS_VF_30_DAYS,MEDIAN_HANDOVERS_VF_30_DAYS
0,0,11741170013,1174117,2024-01-02,3479,0.681524,1.0,0.164038,591.8416,481.0,566,0.16269,1079.0,0.648919,1.0,0.173,643.4541,552.0,271.0,0.251158,1067.0,0.652124,0.5,0.134144,431.71884,325.0,53.0,0.049672,138.0,0.833575,1.0,0.128019,722.7246,594.0,26.0,0.188406,1005.0,0.755046,1.0,0.177197,705.3393,601.0,215.0,0.21393,190.0,0.532456,0.666667,0.157018,502.54736,399.5,1.0,0.005263,0.0,NaN,NaN,NaN,NaN,NaN,0.0,0.0,NaN,NaN,NaN,NaN,NaN,0.0,0.0,NaN,NaN,NaN,NaN,NaN,0.0,0.0,NaN,NaN,NaN,NaN,NaN,0.0,0.0,NaN,NaN,NaN,NaN,NaN,0.0,0.0,NaN,NaN,NaN,NaN,NaN,0.0,0.0,NaN,NaN,NaN,NaN,NaN,0.0,0.0,NaN,NaN,NaN,NaN,NaN,0.0,0.0,NaN,NaN,NaN,NaN,NaN,0.0,0.0,NaN,NaN,NaN,NaN,NaN,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,1,11741170013,1174117,2024-01-03,3479,0.681524,1.0,0.164038,591.8416,481.0,566,0.16269,1079.0,0.648919,1.0,0.173,643.4541,552.0,271.0,0.251158,1067.0,0.652124,0.5,0.134144,431.71884,325.0,53.0,0.049672,138.0,0.833575,1.0,0.128019,722.7246,594.0,26.0,0.188406,1005.0,0.755046,1.0,0.177197,705.3393,601.0,215.0,0.21393,190.0,0.532456,0.666667,0.157018,502.54736,399.5,1.0,0.005263,16.0,685.25000,560.5,0.635417,0.833333,0.153846,2.0,1

In [7]:
# Step 1: Start from original masterdatagcs
masterdatagcs_filtered = masterdatagcs.copy()

In [8]:
# Step 2: Remove columns with 'MEDIAN' in the name
columns_to_keep = [col for col in masterdatagcs_filtered.columns if 'MEDIAN' not in col]
masterdatagcs_filtered = masterdatagcs_filtered[columns_to_keep]

In [9]:
# Step 3: Drop irrelevant column RESOURCE_ID
if 'RESOURCE_ID' in masterdatagcs_filtered.columns:
    masterdatagcs_filtered = masterdatagcs_filtered.drop(columns=['RESOURCE_ID'])

In [10]:
# Step 4: Revert FTR values (FTR = 1 - FTR)
ftr_columns = [col for col in masterdatagcs_filtered.columns if 'FTR' in col]
for col in ftr_columns:
    masterdatagcs_filtered[col] = 1 - masterdatagcs_filtered[col]

In [11]:
# Remove index
masterdatagcs_filtered = masterdatagcs_filtered.reset_index(drop=True)

In [12]:
# Display the cleaned dataset
masterdatagcs_filtered.head()

,Column1,RESOURCE_KEY,LEG_START_TIME,COUNT_CALLS,MEAN_RPC,MEAN_FTR,MEAN_TMC,TOTAL_OTS,OTS_BY_CALL,COUNT_CALLS_IF,MEAN_RPC_IF,MEAN_FTR_IF,MEAN_TMC_IF,TOTAL_OTS_IF,OTS_BY_CALL_IF,COUNT_CALLS_OTHER,MEAN_RPC_OTHER,MEAN_FTR_OTHER,MEAN_TMC_OTHER,TOTAL_OTS_OTHER,OTS_BY_CALL_OTHER,COUNT_CALLS_VF,MEAN_RPC_VF,MEAN_FTR_VF,MEAN_TMC_VF,TOTAL_OTS_VF,OTS_BY_CALL_VF,COUNT_CALLS_TV,MEAN_RPC_TV,MEAN_FTR_TV,MEAN_TMC_TV,TOTAL_OTS_TV,OTS_BY_CALL_TV,COUNT_CALLS_MOVEL,MEAN_RPC_MOVEL,MEAN_FTR_MOVEL,MEAN_TMC_MOVEL,TOTAL_OTS_MOVEL,OTS_BY_CALL_MOVEL,COUNT_LEGS_14_DAYS,MEAN_DURATION_14_DAYS,MEAN_RPC_14_DAYS,MEAN_FTR_14_DAYS,TOTAL_OTS_14_DAYS,COUNT_LEGS_30_DAYS,MEAN_DURATION_30_DAYS,MEAN_RPC_30_DAYS,MEAN_FTR_30_DAYS,TOTAL_OTS_30_DAYS,COUNT_LEGS_IF_14_DAYS,MEAN_DURATION_IF_14_DAYS,MEAN_RPC_IF_14_DAYS,MEAN_FTR_IF_14_DAYS,TOTAL_OTS_IF_14_DAYS,COUNT_LEGS_MOVEL_14_DAYS,MEAN_DURATION_MOVEL_14_DAYS,MEAN_RPC_MOVEL_14_DAYS,MEAN_FTR_MOVEL_14_DAYS,TOTAL_OTS_MOVEL_14_DAYS,COUNT_LEGS_TV_14_DAYS,MEAN_DURATION_TV_14_DAYS,MEAN_RPC_TV_14_DAYS,MEAN_FTR_TV_14_DAYS,TOTAL_OTS_TV_14_DAYS,COUNT_LEGS_VF_14_DAYS,MEAN_DURATION_VF_14_DAYS,MEAN_RPC_VF_14_DAYS,MEAN_FTR_VF_14_DAYS,TOTAL_OTS_VF_14_DAYS,COUNT_LEGS_IF_30_DAYS,MEAN_DURATION_IF_30_DAYS,MEAN_RPC_IF_30_DAYS,MEAN_FTR_IF_30_DAYS,TOTAL_OTS_IF_30_DAYS,COUNT_LEGS_MOVEL_30_DAYS,MEAN_DURATION_MOVEL_30_DAYS,MEAN_RPC_MOVEL_30_DAYS,MEAN_FTR_MOVEL_30_DAYS,TOTAL_OTS_MOVEL_30_DAYS,COUNT_LEGS_TV_30_DAYS,MEAN_DURATION_TV_30_DAYS,MEAN_RPC_TV_30_DAYS,MEAN_FTR_TV_30_DAYS,TOTAL_OTS_TV_30_DAYS,COUNT_LEGS_VF_30_DAYS,MEAN_DURATION_VF_30_DAYS,MEAN_RPC_VF_30_DAYS,MEAN_FTR_VF_30_DAYS,TOTAL_OTS_VF_30_DAYS,MEAN_HANDOVERS_14_DAYS,MEAN_HANDOVERS_30_DAYS,MEAN_HANDOVERS_IF_14_DAYS,MEAN_HANDOVERS_MOVEL_14_DAYS,MEAN_HANDOVERS_TV_14_DAYS,MEAN_HANDOVERS_VF_14_DAYS,MEAN_HANDOVERS_IF_30_DAYS,MEAN_HANDOVERS_MOVEL_30_DAYS,MEAN_HANDOVERS_TV_30_DAYS,MEAN_HANDOVERS_VF_30_DAYS
0,0,11741170013,2024-01-02,3479,0.681524,0.835962,591.8416,566,0.16269,1079.0,0.648919,0.827,643.4541,271.0,0.251158,1067.0,0.652124,0.865856,431.71884,53.0,0.049672,138.0,0.833575,0.871981,722.7246,26.0,0.188406,1005.0,0.755046,0.822803,705.3393,215.0,0.21393,190.0,0.532456,0.842982,502.54736,1.0,0.005263,0.0,NaN,NaN,NaN,0.0,0.0,NaN,NaN,NaN,0.0,0.0,NaN,NaN,NaN,0.0,0.0,NaN,NaN,NaN,0.0,0.0,NaN,NaN,NaN,0.0,0.0,NaN,NaN,NaN,0.0,0.0,NaN,NaN,NaN,0.0,0.0,NaN,NaN,NaN,0.0,0.0,NaN,NaN,NaN,0.0,0.0,NaN,NaN,NaN,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,1,11741170013,2024-01-03,3479,0.681524,0.835962,591.8416,566,0.16269,1079.0,0.648919,0.827,643.4541,271.0,0.251158,1067.0,0.652124,0.865856,431.71884,53.0,0.049672,138.0,0.833575,0.871981,722.7246,26.0,0.188406,1005.0,0.755046,0.822803,705.3393,215.0,0.21393,190.0,0.532456,0.842982,502.54736,1.0,0.005263,16.0,685.25000,0.635417,0.846154,2.0,16.0,685.25000,0.635417,0.846154,2.0,4.0,864.25000,0.750000,0.750000,1.0,2.0,311.50000,0.500000,0.500000,0.0,2.0,844.5000,0.500000,1.000000,1.0,1.0,443.0,0.50,1.0,0.0,4.0,864.25000,0.750000,0.750000,1.0,2.0,311.50000,0.500000,0.500000,0.0,2.0,844.5000,0.500000,1.000000,1.0,1.0,443.0,0.50,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,2,11741170013,2024-01-04,3479,0.681524,0.835962,591.8416,566,0.16269,1079.0,0.648919,0.827,643.4541,271.0,0.251158,1067.0,0.652124,0.865856,431.71884,53.0,0.049672,138.0,0.833575,0.871981,722.7246,26.0,0.188406,1005.0,0.755046,0.822803,705.3393,215.0,0.21393,190.0,0.532456,0.842982,502.54736,1.0,0.005263,24.0,727.45830,0.656944,0.750000,3.0,24.0,727.45830,0.656944,0.750000,3.0,5.0,964.20000,0.600000,0.800000,1.0,NaN,NaN,NaN,NaN,NaN,5.0,770.4000,0.600000,0.600000,1.0,2.0,720.0,0.55,1.0,1.0,5.0,964.20000,0.600000,0.800000,1.0,NaN,NaN,NaN,NaN,NaN,5.0,770.4000,0.600000,0.600000,1.0,2.0,720.0,0.55,1.0,1.0,0.0,0.0,0.0,NaN,0.0,0.0,0.0,NaN,0.0,0.0
3,3,11741170013,2024-01-05,3479,0.681524,0.835962,591.8416,566,0.16269,1079.0,0.648919,0.827,643.4541,271.0,0.251158,1067.0,0.652124,0.865856,431.71884,53.0,0.049672,138.0,0.833575,0.871981,722.7246,26.0,0.188406,1005.0,0.755046,0.822803,705.3393,215.

In [ ]:
# Count NaNs in each column
na_counts = masterdatagcs_filtered.isna().sum()

# Display only columns with at least one NaN
na_counts = na_counts[na_counts > 0]
print(na_counts)

MEAN_FTR                            1
COUNT_CALLS_IF                      9
MEAN_RPC_IF                         9
MEAN_FTR_IF                         9
MEAN_TMC_IF                         9
TOTAL_OTS_IF                        9
OTS_BY_CALL_IF                      9
COUNT_CALLS_OTHER                   1
MEAN_RPC_OTHER                      1
MEAN_FTR_OTHER                     22
MEAN_TMC_OTHER                      1
TOTAL_OTS_OTHER                     1
OTS_BY_CALL_OTHER                   1
COUNT_CALLS_VF                    107
MEAN_RPC_VF                       107
MEAN_FTR_VF                       107
MEAN_TMC_VF                       107
TOTAL_OTS_VF                      107
OTS_BY_CALL_VF                    107
COUNT_CALLS_TV                      8
MEAN_RPC_TV                         8
MEAN_FTR_TV                         8
MEAN_TMC_TV                         8
TOTAL_OTS_TV                        8
OTS_BY_CALL_TV                      8
COUNT_CALLS_MOVEL                 155
MEAN_RPC_MOV

#### Create GC Unique Table

The output of this cleaning is the gcs_unique table which contains one row per GC, representing their overall performance. It includes key metrics like FTR, TMC, and call counts across categories.

In [14]:
# Ensure LEG_START_TIME is datetime and extract date part
masterdatagcs_filtered['LEG_START_TIME'] = pd.to_datetime(masterdatagcs_filtered['LEG_START_TIME'])
masterdatagcs_filtered['LEG_START_DATE'] = masterdatagcs_filtered['LEG_START_TIME'].dt.date


In [15]:
# Calculate DAYS_ACTIVE per RESOURCE_KEY
masterdatagcs_filtered['DAYS_ACTIVE'] = masterdatagcs_filtered.groupby('RESOURCE_KEY')['LEG_START_DATE'].transform('nunique')


In [16]:
# Define the columns to keep (exclude LEG_START_TIME)
columns_to_keep = [
    "RESOURCE_KEY", "COUNT_CALLS", "DAYS_ACTIVE",
    "MEAN_RPC", "MEAN_FTR", "MEAN_TMC", "TOTAL_OTS", "OTS_BY_CALL",
    "COUNT_CALLS_IF", "MEAN_RPC_IF", "MEAN_FTR_IF", "MEAN_TMC_IF", "TOTAL_OTS_IF", "OTS_BY_CALL_IF",
    "COUNT_CALLS_OTHER", "MEAN_RPC_OTHER", "MEAN_FTR_OTHER", "MEAN_TMC_OTHER", "TOTAL_OTS_OTHER", "OTS_BY_CALL_OTHER",
    "COUNT_CALLS_VF", "MEAN_RPC_VF", "MEAN_FTR_VF", "MEAN_TMC_VF", "TOTAL_OTS_VF", "OTS_BY_CALL_VF",
    "COUNT_CALLS_TV", "MEAN_RPC_TV", "MEAN_FTR_TV", "MEAN_TMC_TV", "TOTAL_OTS_TV", "OTS_BY_CALL_TV",
    "COUNT_CALLS_MOVEL", "MEAN_RPC_MOVEL", "MEAN_FTR_MOVEL", "MEAN_TMC_MOVEL", "TOTAL_OTS_MOVEL", "OTS_BY_CALL_MOVEL"
]


In [17]:
# Drop RESOURCE_KEY from columns_to_keep if it's there
columns_to_keep_no_key = [col for col in columns_to_keep if col != "RESOURCE_KEY"]


In [18]:
def mode_agg(group):
    return group.mode().iloc[0] if not group.mode().empty else group.iloc[0]


In [19]:
# Group and aggregate
gcs_unique = (
    masterdatagcs_filtered
    .groupby("RESOURCE_KEY")[columns_to_keep_no_key]
    .agg(mode_agg)
    .reset_index()  # now this works safely — RESOURCE_KEY comes back as a column
)


In [20]:
# Output shape and preview
print("Shape of the resulting DataFrame:", gcs_unique.shape)

gcs_unique.head()

Shape of the resulting DataFrame: (653, 38)


,RESOURCE_KEY,COUNT_CALLS,DAYS_ACTIVE,MEAN_RPC,MEAN_FTR,MEAN_TMC,TOTAL_OTS,OTS_BY_CALL,COUNT_CALLS_IF,MEAN_RPC_IF,MEAN_FTR_IF,MEAN_TMC_IF,TOTAL_OTS_IF,OTS_BY_CALL_IF,COUNT_CALLS_OTHER,MEAN_RPC_OTHER,MEAN_FTR_OTHER,MEAN_TMC_OTHER,TOTAL_OTS_OTHER,OTS_BY_CALL_OTHER,COUNT_CALLS_VF,MEAN_RPC_VF,MEAN_FTR_VF,MEAN_TMC_VF,TOTAL_OTS_VF,OTS_BY_CALL_VF,COUNT_CALLS_TV,MEAN_RPC_TV,MEAN_FTR_TV,MEAN_TMC_TV,TOTAL_OTS_TV,OTS_BY_CALL_TV,COUNT_CALLS_MOVEL,MEAN_RPC_MOVEL,MEAN_FTR_MOVEL,MEAN_TMC_MOVEL,TOTAL_OTS_MOVEL,OTS_BY_CALL_MOVEL
0,11741170013,3479,178,0.681524,0.835962,591.84160,566,0.162690,1079.0,0.648919,0.827000,643.45410,271.0,0.251158,1067.0,0.652124,0.865856,431.71884,53.0,0.049672,138.0,0.833575,0.871981,722.72460,26.0,0.188406,1005.0,0.755046,0.822803,705.3393,215.0,0.21393,190.0,0.532456,0.842982,502.54736,1.0,0.005263
1,11741900049,1876,113,0.643456,0.832462,666.62210,419,0.223348,601.0,0.600751,0.814254,709.27454,167.0,0.277870,474.0,0.706495,0.848073,549.05273,21.0,0.044304,72.0,0.813657,0.925926,700.48610,8.0,0.111111,604.0,0.591075,0.817993,732.5844,222.0,0.36755,125.0,0.764800,0.896000,569.13600,1.0,0.008000
2,11741920010,2662,177,0.712828,0.825540,777.22955,372,0.139745,908.0,0.685738,0.809068,816.91960,161.0,0.177313,641.0,0.710045,0.836207,632.10767,21.0,0.032761,118.0,0.809322,0.827684,778.20337,15.0,0.127119,837.0,0.753196,0.831243,852.9068,175.0,0.20908,158.0,0.593882,0.857068,736.26580,0.0,0.000000
3,11743190006,163,12,0.625256,0.883463,679.68097,32,0.196319,37.0,0.677928,0.896396,829.35140,12.0,0.324324,60.0,0.633333,0.923077,522.31665,5.0,0.083333,8.0,0.687500,0.875000,647.37500,1.0,0.125000,50.0,0.626667,0.836000,743.9800,14.0,0.28000,8.0,0.250000,1.000000,798.12500,0.0,0.000000
4,11750000013,3295,173,0.692244,0.837657,587.90656,570,0.172989,1098.0,0.663420,0.832878,649.13570,259.0,0.235883,908.0,0.697690,0.854841,389.85352,54.0,0.059471,143.0,0.761655,0.846154,631.88810,29.0,0.202797,947.0,0.740589,0.833685,707.1098,228.0,0.24076,199.0,0.546482,0.832496,554.87940,0.0,0.000000


In [21]:
# Define columns grouped by category
category_columns = {
    "IF": ["COUNT_CALLS_IF", "MEAN_RPC_IF", "MEAN_FTR_IF", "MEAN_TMC_IF", "TOTAL_OTS_IF", "OTS_BY_CALL_IF"],
    "OTHER": ["COUNT_CALLS_OTHER", "MEAN_RPC_OTHER", "MEAN_FTR_OTHER", "MEAN_TMC_OTHER", "TOTAL_OTS_OTHER", "OTS_BY_CALL_OTHER"],
    "VF": ["COUNT_CALLS_VF", "MEAN_RPC_VF", "MEAN_FTR_VF", "MEAN_TMC_VF", "TOTAL_OTS_VF", "OTS_BY_CALL_VF"],
    "TV": ["COUNT_CALLS_TV", "MEAN_RPC_TV", "MEAN_FTR_TV", "MEAN_TMC_TV", "TOTAL_OTS_TV", "OTS_BY_CALL_TV"],
    "MOVEL": ["COUNT_CALLS_MOVEL", "MEAN_RPC_MOVEL", "MEAN_FTR_MOVEL", "MEAN_TMC_MOVEL", "TOTAL_OTS_MOVEL", "OTS_BY_CALL_MOVEL"]
}

# Loop over each category and create binary flags, then fill NaNs with 0
for cat, cols in category_columns.items():
    # Create binary column: 0 if all values in category are NaN, 1 otherwise
    gcs_unique[f"CALLS_{cat}_BINARY"] = gcs_unique[cols].notna().any(axis=1).astype(int)
    
# Replace all NaNs in the DataFrame with 0
gcs_unique = gcs_unique.fillna(0)

# Optional: check if any NaNs remain
print("Remaining NaNs:\n", gcs_unique.isna().sum()[gcs_unique.isna().sum() > 0])

Remaining NaNs:
 Series([], dtype: int64)


In [22]:
# Count NaNs per column in gcs_unique
na_counts = gcs_unique.isna().sum()
na_counts = na_counts[na_counts > 0]  # Show only columns with at least one NaN
na_counts

Series([], dtype: int64)

In [ ]:
# Count number of zeros in each column of gcs_unique
zero_counts = (gcs_unique == 0).sum()
zero_counts = zero_counts[zero_counts > 0]  # Show only columns with at least one zero
zero_counts

MEAN_FTR                2
TOTAL_OTS               3
OTS_BY_CALL             3
COUNT_CALLS_IF          2
MEAN_RPC_IF             5
MEAN_FTR_IF             2
MEAN_TMC_IF             2
TOTAL_OTS_IF           18
OTS_BY_CALL_IF         18
COUNT_CALLS_OTHER       1
MEAN_RPC_OTHER          1
MEAN_FTR_OTHER          8
MEAN_TMC_OTHER          1
TOTAL_OTS_OTHER        59
OTS_BY_CALL_OTHER      59
COUNT_CALLS_VF         23
MEAN_RPC_VF            27
MEAN_FTR_VF            23
MEAN_TMC_VF            23
TOTAL_OTS_VF           75
OTS_BY_CALL_VF         75
COUNT_CALLS_TV          3
MEAN_RPC_TV             6
MEAN_FTR_TV             4
MEAN_TMC_TV             3
TOTAL_OTS_TV           10
OTS_BY_CALL_TV         10
COUNT_CALLS_MOVEL      24
MEAN_RPC_MOVEL         31
MEAN_FTR_MOVEL         28
MEAN_TMC_MOVEL         24
TOTAL_OTS_MOVEL       535
OTS_BY_CALL_MOVEL     535
CALLS_IF_BINARY         2
CALLS_OTHER_BINARY      1
CALLS_VF_BINARY        23
CALLS_TV_BINARY         3
CALLS_MOVEL_BINARY     24
dtype: int64

### Clients Cleaning

In [ ]:
# Check the new client dataset for missing values
missing_values = client_df.isnull().sum()
missing_values = missing_values[missing_values > 0]
missing_values = missing_values.sort_values(ascending=False)
missing_values

FTR_CALCULATED_MEAN_180_ASSUNTO_VF       1125799
PROP_OTS_VF_180                          1125791
LEG_DURATION_MEAN_180_ASSUNTO_VF         1125791
RPC_MEAN_180_ASSUNTO_VF                  1125791
FTR_CALCULATED_MEAN_365_ASSUNTO_VF       1120865
LEG_DURATION_MEAN_365_ASSUNTO_VF         1120857
PROP_OTS_VF_365                          1120857
RPC_MEAN_365_ASSUNTO_VF                  1120857
LEG_DURATION_MEAN_180_ASSUNTO_MOVEL      1120821
RPC_MEAN_180_ASSUNTO_MOVEL               1120821
FTR_CALCULATED_MEAN_180_ASSUNTO_MOVEL    1120821
PROP_OTS_MOVEL_180                       1120821
PROP_OTS_MOVEL_365                       1115715
LEG_DURATION_MEAN_365_ASSUNTO_MOVEL      1115715
FTR_CALCULATED_MEAN_365_ASSUNTO_MOVEL    1115715
RPC_MEAN_365_ASSUNTO_MOVEL               1115715
FTR_CALCULATED_MEAN_180_ASSUNTO_IF        900788
LEG_DURATION_MEAN_180_ASSUNTO_IF          900727
RPC_MEAN_180_ASSUNTO_IF                   900727
PROP_OTS_IF_180                           900727
FTR_CALCULATED_MEAN_

#### Age Cleaning

In [25]:
# Investigate Age
missing_percentage = client_df['CLIENT_AGE_YEARS'].isnull().mean() * 100
print(f'Missing age percentage: {missing_percentage:.2f}%')

Missing age percentage: 17.49%


In [26]:
# Ensure -1 and already missing (NaN) values are all treated as missing
client_df['Age_missing'] = ((client_df['CLIENT_AGE_YEARS'] == -1) | 
                            (client_df['CLIENT_AGE_YEARS'].isna())).astype(int)

# Replace -1 explicitly with np.nan
client_df['CLIENT_AGE_YEARS'] = client_df['CLIENT_AGE_YEARS'].replace(-1, np.nan)

client_df['Age_missing'].value_counts(normalize=True)

Age_missing
0    0.825119
1    0.174881
Name: proportion, dtype: float64

In [27]:
# Check for each PERSON_ID how many distinct age values (excluding NaNs) exist
age_counts = client_df.groupby('PERSON_ID')['CLIENT_AGE_YEARS'].nunique(dropna=True)

# Identify PERSON_IDs with at least one valid age value and at least one missing age value
persons_with_mixed_age = client_df.groupby('PERSON_ID').filter(
    lambda x: x['CLIENT_AGE_YEARS'].notna().any() and x['CLIENT_AGE_YEARS'].isna().any()
)['PERSON_ID'].nunique()

print(f"Number of PERSON_IDs with mixed missing/non-missing age values: {persons_with_mixed_age}")


Number of PERSON_IDs with mixed missing/non-missing age values: 35


In [28]:
# Number of rows with missing age that could potentially be filled from another row
missing_age_rows_potentially_fillable = client_df[
    client_df['CLIENT_AGE_YEARS'].isna() &
    client_df['PERSON_ID'].isin(
        client_df.loc[client_df['CLIENT_AGE_YEARS'].notna(), 'PERSON_ID']
    )
].shape[0]

total_missing = client_df['CLIENT_AGE_YEARS'].isna().sum()

print(f"Rows with missing age potentially fillable: {missing_age_rows_potentially_fillable}")
print(f"Total rows with missing age: {total_missing}")
print(f"Percentage of potentially fillable rows: {100 * missing_age_rows_potentially_fillable / total_missing:.2f}%")


Rows with missing age potentially fillable: 55
Total rows with missing age: 205175
Percentage of potentially fillable rows: 0.03%


In [29]:
# Fill rows
def fill_age(group):
    if group.dropna().empty:
        # If the group has no valid age values, just return the group unchanged.
        return group
    else:
        # Otherwise, fill missing values with the group's median.
        return group.fillna(group.median())

client_df['CLIENT_AGE_YEARS'] = client_df.groupby('PERSON_ID')['CLIENT_AGE_YEARS'].transform(fill_age)


In [30]:
print("Remaining missing ages after transfer:", client_df['CLIENT_AGE_YEARS'].isnull().sum())


Remaining missing ages after transfer: 205120


In [31]:
# Select only numerical columns from the DataFrame
numerical_client_df = client_df.select_dtypes(include=['number'])

# Compute the correlation matrix for numerical columns
corr_matrix = numerical_client_df.corr()

# Sort the correlations with respect to 'CLIENT_AGE_YEARS' in descending order
corr_with_missing_age = corr_matrix['Age_missing'].sort_values(ascending=False)

# Display the result
corr_with_missing_age

Age_missing                                   1.000000
FLG_EMPRESARIAL                               0.447765
FLG_BOX_2_HD_CABO                             0.203044
ARPU_CALCULATED                               0.141987
IF_UPLOAD_SPEED_KBPS_QTY                      0.137074
IF_DOWNLOAD_SPEED_MBPS_QTY                    0.131246
TV_SPTV_FLG                                   0.119504
TV_11SPORTS_FLG                               0.107731
TV_BTV_FLG                                    0.101281
PROP_CALLS_OTHER_180                          0.090393
PROP_CALLS_OTHER_365                          0.089282
IM_PROD_FLG                                   0.085839
PF_CLIENT_MONTHS                              0.082845
FLG_UNKNOWN                                   0.081685
NR_SAS                                        0.078695
SERVICES_QTY                                  0.072784
PROP_CALLS_MOVEL_365                          0.066945
PROP_CALLS_MOVEL_180                          0.064906
FLG_BOX_3_

Missing Age is highly correlating with B2B customers, which makes sense. 44% of empreserial consumers / or at least rows with those consumers, have missing ages. Therefore, going forward, we will impute the missing ages based on the medians per FLG_EMPRESERIAL group.

In [32]:
# Just checking how many unique customers we have per group
# Count unique customers per FLG_EMPRESARIAL
unique_customers_per_flag = client_df.groupby('FLG_EMPRESARIAL')['PERSON_SK'].nunique()

# Display the result
print(unique_customers_per_flag)

FLG_EMPRESARIAL
0.0    511177
1.0     43070
Name: PERSON_SK, dtype: int64


In [33]:
# Calculate median ages per group
median_age_b2b = client_df[client_df['FLG_EMPRESARIAL'] == 1]['CLIENT_AGE_YEARS'].median()
median_age_consumer = client_df[client_df['FLG_EMPRESARIAL'] == 0]['CLIENT_AGE_YEARS'].median()

In [34]:
# B2B customers (FLG_EMPRESARIAL == 1)
client_df['CLIENT_AGE_YEARS'] = np.where(
    (client_df['CLIENT_AGE_YEARS'].isna()) & (client_df['FLG_EMPRESARIAL'] == 1),
    median_age_b2b,
    client_df['CLIENT_AGE_YEARS']
)

# Consumer customers (FLG_EMPRESARIAL == 0)
client_df['CLIENT_AGE_YEARS'] = np.where(
    (client_df['CLIENT_AGE_YEARS'].isna()) & (client_df['FLG_EMPRESARIAL'] == 0),
    median_age_consumer,
    client_df['CLIENT_AGE_YEARS']
)


In [35]:
print("Remaining missing ages:", client_df['CLIENT_AGE_YEARS'].isnull().sum())  # Should print 0


Remaining missing ages: 56495


In [36]:
print(client_df['FLG_EMPRESARIAL'].value_counts(dropna=False))


FLG_EMPRESARIAL
0.0    1005718
1.0      87244
NaN      80264
Name: count, dtype: int64


In [37]:
# For customers with no flag, imputing global median age
global_median_age = client_df['CLIENT_AGE_YEARS'].median()

client_df['CLIENT_AGE_YEARS'] = np.where(
    client_df['CLIENT_AGE_YEARS'].isna(),
    global_median_age,
    client_df['CLIENT_AGE_YEARS']
)

In [ ]:
print("Remaining missing ages:", client_df['CLIENT_AGE_YEARS'].isnull().sum())  # Should print 0


Remaining missing ages: 0


#### Other Metrics

In [39]:
# Check the new client dataset for missing values
missing_values = client_df.isnull().sum()
missing_values = missing_values[missing_values > 0]
missing_values = missing_values.sort_values(ascending=False)
missing_values

FTR_CALCULATED_MEAN_180_ASSUNTO_VF       1125799
LEG_DURATION_MEAN_180_ASSUNTO_VF         1125791
RPC_MEAN_180_ASSUNTO_VF                  1125791
PROP_OTS_VF_180                          1125791
FTR_CALCULATED_MEAN_365_ASSUNTO_VF       1120865
PROP_OTS_VF_365                          1120857
RPC_MEAN_365_ASSUNTO_VF                  1120857
LEG_DURATION_MEAN_365_ASSUNTO_VF         1120857
RPC_MEAN_180_ASSUNTO_MOVEL               1120821
FTR_CALCULATED_MEAN_180_ASSUNTO_MOVEL    1120821
PROP_OTS_MOVEL_180                       1120821
LEG_DURATION_MEAN_180_ASSUNTO_MOVEL      1120821
PROP_OTS_MOVEL_365                       1115715
FTR_CALCULATED_MEAN_365_ASSUNTO_MOVEL    1115715
RPC_MEAN_365_ASSUNTO_MOVEL               1115715
LEG_DURATION_MEAN_365_ASSUNTO_MOVEL      1115715
FTR_CALCULATED_MEAN_180_ASSUNTO_IF        900788
RPC_MEAN_180_ASSUNTO_IF                   900727
PROP_OTS_IF_180                           900727
LEG_DURATION_MEAN_180_ASSUNTO_IF          900727
FTR_CALCULATED_MEAN_

In [40]:
# ---------------------------
# Step 1. Define Your Metric Lists
# ---------------------------
# These lists include the columns you want to clean.
# For the 180-day (shorter-term) metrics:
metrics_180 = [
    'LEG_DURATION_MEAN_180_ASSUNTO_IF',
    'LEG_DURATION_MEAN_180_ASSUNTO_MOVEL',
    'LEG_DURATION_MEAN_180_ASSUNTO_OTHER',
    'LEG_DURATION_MEAN_180_ASSUNTO_TV',
    'LEG_DURATION_MEAN_180_ASSUNTO_VF',
    'LEG_DURATION_MEAN_180_TOTAL',
    'RPC_MEAN_180_ASSUNTO_IF',
    'RPC_MEAN_180_ASSUNTO_MOVEL',
    'RPC_MEAN_180_ASSUNTO_OTHER',
    'RPC_MEAN_180_ASSUNTO_TV',
    'RPC_MEAN_180_ASSUNTO_VF',
    'RPC_MEAN_180_TOTAL',
    'FTR_CALCULATED_MEAN_180_ASSUNTO_IF',
    'FTR_CALCULATED_MEAN_180_ASSUNTO_MOVEL',
    'FTR_CALCULATED_MEAN_180_ASSUNTO_OTHER',
    'FTR_CALCULATED_MEAN_180_ASSUNTO_TV',
    'FTR_CALCULATED_MEAN_180_ASSUNTO_VF',
    'FTR_CALCULATED_MEAN_180_TOTAL'
]

# For the 365-day (longer-term) metrics:
metrics_365 = [
    'LEG_DURATION_MEAN_365_ASSUNTO_IF',
    'LEG_DURATION_MEAN_365_ASSUNTO_MOVEL',
    'LEG_DURATION_MEAN_365_ASSUNTO_OTHER',
    'LEG_DURATION_MEAN_365_ASSUNTO_TV',
    'LEG_DURATION_MEAN_365_ASSUNTO_VF',
    'LEG_DURATION_MEAN_365_TOTAL',
    'RPC_MEAN_365_ASSUNTO_IF',
    'RPC_MEAN_365_ASSUNTO_MOVEL',
    'RPC_MEAN_365_ASSUNTO_OTHER',
    'RPC_MEAN_365_ASSUNTO_TV',
    'RPC_MEAN_365_ASSUNTO_VF',
    'RPC_MEAN_365_TOTAL',
    'FTR_CALCULATED_MEAN_365_ASSUNTO_IF',
    'FTR_CALCULATED_MEAN_365_ASSUNTO_MOVEL',
    'FTR_CALCULATED_MEAN_365_ASSUNTO_OTHER',
    'FTR_CALCULATED_MEAN_365_ASSUNTO_TV',
    'FTR_CALCULATED_MEAN_365_ASSUNTO_VF',
    'FTR_CALCULATED_MEAN_365_TOTAL'
]


In [41]:
# ---------------------------
# Step 2. Sort the DataFrame
# ---------------------------
# Sort by PERSON_ID and the date column (START_DATE) so that ffill and bfill work correctly.
client_df = client_df.sort_values(['PERSON_ID', 'START_DATE'])


In [42]:
# ---------------------------
# Step 3. Define a Vectorized Fill Function
# ---------------------------
# This function uses ffill() followed by bfill() for rows where the metric is missing
# and the call count is 0. We use groupby().transform() to do this in a vectorized manner.
def fill_metric_vectorized(df, metric, count_col):
    # Create a mask for rows where the metric is missing and the count is 0.
    mask = (df[count_col] == 0) & (df[metric].isna())
    if mask.any():
        # Use groupby transform to generate a filled series using forward fill and backward fill.
        filled_series = df.groupby('PERSON_ID')[metric].transform(lambda x: x.ffill())
        # Update only the rows where the mask is True.
        df.loc[mask, metric] = filled_series[mask]
    return df

In [43]:
# ---------------------------
# Step 4. Apply the Fill Function for Each Group of Metrics
# ---------------------------
# For 180-day metrics, we use the call count column 'LEG_IF_ID_COUNT_180_TOTAL'.
for metric in metrics_180:
    client_df = fill_metric_vectorized(client_df, metric, 'LEG_IF_ID_COUNT_180_TOTAL')

# For 365-day metrics, we use the call count column 'LEG_IF_ID_COUNT_365_TOTAL'.
for metric in metrics_365:
    client_df = fill_metric_vectorized(client_df, metric, 'LEG_IF_ID_COUNT_365_TOTAL')

# ---------------------------

In [44]:
# ---------------------------
# Step 5. Validate the Imputation
# ---------------------------
# Check the number of missing values in one of the metrics, for example:
print("Missing in LEG_DURATION_MEAN_180_ASSUNTO_IF:",
      client_df['LEG_DURATION_MEAN_180_ASSUNTO_IF'].isna().sum())


Missing in LEG_DURATION_MEAN_180_ASSUNTO_IF: 899623


In [45]:
# And inspect a sample of rows for a given client:
sample_client = client_df[client_df['PERSON_ID'] == client_df['PERSON_ID'].iloc[0]]
print(sample_client[['START_DATE', 'LEG_DURATION_MEAN_180_ASSUNTO_IF', 'LEG_IF_ID_COUNT_180_TOTAL']].head())


                     START_DATE  LEG_DURATION_MEAN_180_ASSUNTO_IF  \
376658  2024-01-31 00:00:00.000                               NaN   

        LEG_IF_ID_COUNT_180_TOTAL  
376658                        0.0  


In [46]:
client_df.head()

,Column1,PERSON_SK,FISCAL_NUM,START_DATE,PERSON_ID,CA_COD,SA_COD,LEG_START_TIME_DAY,DAT_CRI_SA,DT_NASC,CLIENT_AGE_YEARS,CLIENT_ANTIQUITY_YEARS,NR_SAS,ARPU_CALCULATED,SERVICES_QTY,TV_PROD_FLG,IF_PROD_FLG,VF_PROD_FLG,VM_PROD_FLG,IM_PROD_FLG,TECH_COPPER_FLG,TECH_CABLE_FLG,TECH_FTTH_FLG,TECH_DTH_FLG,TECH_GSM_FLG,TV_SPTV_FLG,TV_BTV_FLG,TV_11SPORTS_FLG,TV_TVCINE_FLG,IF_DOWNLOAD_SPEED_MBPS_QTY,IF_UPLOAD_SPEED_KBPS_QTY,FLG_BOX_1_HD_SAT,FLG_BOX_1_HD_PLUS_DVR_SAT_TDT,FLG_BOX_2_HD_CABO,FLG_BOX_2_HD_PLUS_DVR_CABO,FLG_BOX_3_ULTRA_HD,FLG_NO_BOX,FLG_UNKNOWN,FLG_CONSUMER,FLG_EMPRESARIAL,FLG_ILHAS,FLG_INDEFINIDO,ARPU_CLIENT,PF_CLIENT_MONTHS,LEG_DURATION_MEAN_180_ASSUNTO_IF,LEG_DURATION_MEAN_180_ASSUNTO_MOVEL,LEG_DURATION_MEAN_180_ASSUNTO_OTHER,LEG_DURATION_MEAN_180_ASSUNTO_TV,LEG_DURATION_MEAN_180_ASSUNTO_VF,LEG_DURATION_MEAN_180_TOTAL,LEG_DURATION_MEAN_365_ASSUNTO_IF,LEG_DURATION_MEAN_365_ASSUNTO_MOVEL,LEG_DURATION_MEAN_365_ASSUNTO_OTHER,LEG_DURATION_MEAN_365_ASSUNTO_TV,LEG_DURATION_MEAN_365_ASSUNTO_VF,LEG_DURATION_MEAN_365_TOTAL,RPC_MEAN_180_ASSUNTO_IF,RPC_MEAN_180_ASSUNTO_MOVEL,RPC_MEAN_180_ASSUNTO_OTHER,RPC_MEAN_180_ASSUNTO_TV,RPC_MEAN_180_ASSUNTO_VF,RPC_MEAN_180_TOTAL,RPC_MEAN_365_ASSUNTO_IF,RPC_MEAN_365_ASSUNTO_MOVEL,RPC_MEAN_365_ASSUNTO_OTHER,RPC_MEAN_365_ASSUNTO_TV,RPC_MEAN_365_ASSUNTO_VF,RPC_MEAN_365_TOTAL,FTR_CALCULATED_MEAN_180_ASSUNTO_IF,FTR_CALCULATED_MEAN_180_ASSUNTO_MOVEL,FTR_CALCULATED_MEAN_180_ASSUNTO_OTHER,FTR_CALCULATED_MEAN_180_ASSUNTO_TV,FTR_CALCULATED_MEAN_180_ASSUNTO_VF,FTR_CALCULATED_MEAN_180_TOTAL,FTR_CALCULATED_MEAN_365_ASSUNTO_IF,FTR_CALCULATED_MEAN_365_ASSUNTO_MOVEL,FTR_CALCULATED_MEAN_365_ASSUNTO_OTHER,FTR_CALCULATED_MEAN_365_ASSUNTO_TV,FTR_CALCULATED_MEAN_365_ASSUNTO_VF,FTR_CALCULATED_MEAN_365_TOTAL,LEG_IF_ID_COUNT_180_ASSUNTO_IF,LEG_IF_ID_COUNT_180_ASSUNTO_MOVEL,LEG_IF_ID_COUNT_180_ASSUNTO_OTHER,LEG_IF_ID_COUNT_180_ASSUNTO_TV,LEG_IF_ID_COUNT_180_ASSUNTO_VF,LEG_IF_ID_COUNT_180_TOTAL,LEG_IF_ID_COUNT_365_ASSUNTO_IF,LEG_IF_ID_COUNT_365_ASSUNTO_MOVEL,LEG_IF_ID_COUNT_365_ASSUNTO_OTHER,LEG_IF_ID_COUNT_365_ASSUNTO_TV,LEG_IF_ID_COUNT_365_ASSUNTO_VF,LEG_IF_ID_COUNT_365_TOTAL,NR_INCIDENTS_PER_LEG_SUM_180_ASSUNTO_IF,NR_INCIDENTS_PER_LEG_SUM_180_ASSUNTO_MOVEL,NR_INCIDENTS_PER_LEG_SUM_180_ASSUNTO_OTHER,NR_INCIDENTS_PER_LEG_SUM_180_ASSUNTO_TV,NR_INCIDENTS_PER_LEG_SUM_180_ASSUNTO_VF,NR_INCIDENTS_PER_LEG_SUM_180_TOTAL,NR_INCIDENTS_PER_LEG_SUM_365_ASSUNTO_IF,NR_INCIDENTS_PER_LEG_SUM_365_ASSUNTO_MOVEL,NR_INCIDENTS_PER_LEG_SUM_365_ASSUNTO_OTHER,NR_INCIDENTS_PER_LEG_SUM_365_ASSUNTO_TV,NR_INCIDENTS_PER_LEG_SUM_365_ASSUNTO_VF,NR_INCIDENTS_PER_LEG_SUM_365_TOTAL,CALL_IF_ID_NUNIQUE_180_ASSUNTO_IF,CALL_IF_ID_NUNIQUE_180_ASSUNTO_MOVEL,CALL_IF_ID_NUNIQUE_180_ASSUNTO_OTHER,CALL_IF_ID_NUNIQUE_180_ASSUNTO_TV,CALL_IF_ID_NUNIQUE_180_ASSUNTO_VF,CALL_IF_ID_NUNIQUE_180_TOTAL,CALL_IF_ID_NUNIQUE_365_ASSUNTO_IF,CALL_IF_ID_NUNIQUE_365_ASSUNTO_MOVEL,CALL_IF_ID_NUNIQUE_365_ASSUNTO_OTHER,CALL_IF_ID_NUNIQUE_365_ASSUNTO_TV,CALL_IF_ID_NUNIQUE_365_ASSUNTO_VF,CALL_IF_ID_NUNIQUE_365_TOTAL,FLAG_OT_SUM_180_ASSUNTO_IF,FLAG_OT_SUM_180_ASSUNTO_MOVEL,FLAG_OT_SUM_180_ASSUNTO_OTHER,FLAG_OT_SUM_180_ASSUNTO_TV,FLAG_OT_SUM_180_ASSUNTO_VF,FLAG_OT_SUM_180_TOTAL,FLAG_OT_SUM_365_ASSUNTO_IF,FLAG_OT_SUM_365_ASSUNTO_MOVEL,FLAG_OT_SUM_365_ASSUNTO_OTHER,FLAG_OT_SUM_365_ASSUNTO_TV,FLAG_OT_SUM_365_ASSUNTO_VF,FLAG_OT_SUM_365_TOTAL,PROP_CALLS_IF_180,PROP_CALLS_IF_365,PROP_OTS_IF_180,PROP_OTS_IF_365,PROP_CALLS_OTHER_180,PROP_CALLS_OTHER_365,PROP_OTS_OTHER_180,PROP_OTS_OTHER_365,PROP_CALLS_TV_180,PROP_CALLS_TV_365,PROP_OTS_TV_180,PROP_OTS_TV_365,PROP_CALLS_VF_180,PROP_CALLS_VF_365,PROP_OTS_VF_180,PROP_OTS_VF_365,PROP_CALLS_MOVEL_180,PROP_CALLS_MOVEL_365,PROP_OTS_MOVEL_180,PROP_OTS_MOVEL_365,PROP_OTS_TOTAL_180,PROP_OTS_TOTAL_365,Age_missing
376658,139635,07+/iQGuL6MTIDgkWV8Yuy2QyUOHPWm8dmYrTzjnNsYM=,0J9g5l9ZgsFKdD9NvdVPaA8XvlO2pdpkJLygLF7sA8VM=,2024-01-31 00:00:00.000,0++/55V3HyceNFIOE5CoxKCpQsOP3du/tNKvRpHte8og=,0SYkJCS2h4FIXckZufMOOdLUR8jyqVtQo08dypU21c0A=,0SYkJCS2h4FIXckZufMOOdLUR8jyqV

In [47]:
# Check to see how much missing data we have in the client dataset
missing_values = client_df.isnull().sum()
missing_values = missing_values[missing_values > 0]
missing_values = missing_values.sort_values(ascending=False)
missing_values

PROP_OTS_VF_180                          1125791
FTR_CALCULATED_MEAN_180_ASSUNTO_VF       1125628
RPC_MEAN_180_ASSUNTO_VF                  1125620
LEG_DURATION_MEAN_180_ASSUNTO_VF         1125620
FTR_CALCULATED_MEAN_365_ASSUNTO_VF       1120865
PROP_OTS_VF_365                          1120857
RPC_MEAN_365_ASSUNTO_VF                  1120857
LEG_DURATION_MEAN_365_ASSUNTO_VF         1120857
PROP_OTS_MOVEL_180                       1120821
FTR_CALCULATED_MEAN_180_ASSUNTO_MOVEL    1120668
RPC_MEAN_180_ASSUNTO_MOVEL               1120668
LEG_DURATION_MEAN_180_ASSUNTO_MOVEL      1120668
PROP_OTS_MOVEL_365                       1115715
FTR_CALCULATED_MEAN_365_ASSUNTO_MOVEL    1115715
RPC_MEAN_365_ASSUNTO_MOVEL               1115715
LEG_DURATION_MEAN_365_ASSUNTO_MOVEL      1115715
PROP_OTS_IF_180                           900727
FTR_CALCULATED_MEAN_180_ASSUNTO_IF        899684
RPC_MEAN_180_ASSUNTO_IF                   899623
LEG_DURATION_MEAN_180_ASSUNTO_IF          899623
PROP_OTS_TV_180     

In [48]:
# Check for negative values in antiquity
negative_antiquity = client_df[client_df['CLIENT_ANTIQUITY_YEARS'] < 0]
negative_antiquity[['PERSON_ID', 'CLIENT_ANTIQUITY_YEARS']].head(10)
negative_antiquity_count = negative_antiquity.shape[0]
negative_antiquity_count

1699

In [49]:
missing_antiquity = client_df[client_df['CLIENT_ANTIQUITY_YEARS'].isnull()]
missing_antiquity_count = missing_antiquity.shape[0]
missing_antiquity_count

54152

In [50]:
client_df.shape

(1173226, 151)

In [51]:
# Check missing values in client_df
missing_values = client_df.isnull().sum()
missing_values = missing_values[missing_values > 0]
missing_values = missing_values.sort_values(ascending=False)
missing_values

PROP_OTS_VF_180                          1125791
FTR_CALCULATED_MEAN_180_ASSUNTO_VF       1125628
RPC_MEAN_180_ASSUNTO_VF                  1125620
LEG_DURATION_MEAN_180_ASSUNTO_VF         1125620
FTR_CALCULATED_MEAN_365_ASSUNTO_VF       1120865
PROP_OTS_VF_365                          1120857
RPC_MEAN_365_ASSUNTO_VF                  1120857
LEG_DURATION_MEAN_365_ASSUNTO_VF         1120857
PROP_OTS_MOVEL_180                       1120821
FTR_CALCULATED_MEAN_180_ASSUNTO_MOVEL    1120668
RPC_MEAN_180_ASSUNTO_MOVEL               1120668
LEG_DURATION_MEAN_180_ASSUNTO_MOVEL      1120668
PROP_OTS_MOVEL_365                       1115715
FTR_CALCULATED_MEAN_365_ASSUNTO_MOVEL    1115715
RPC_MEAN_365_ASSUNTO_MOVEL               1115715
LEG_DURATION_MEAN_365_ASSUNTO_MOVEL      1115715
PROP_OTS_IF_180                           900727
FTR_CALCULATED_MEAN_180_ASSUNTO_IF        899684
RPC_MEAN_180_ASSUNTO_IF                   899623
LEG_DURATION_MEAN_180_ASSUNTO_IF          899623
PROP_OTS_TV_180     

In [52]:
call_count_columns = [
    "CALL_IF_ID_NUNIQUE_180_ASSUNTO_IF",
    "CALL_IF_ID_NUNIQUE_180_ASSUNTO_MOVEL",
    "CALL_IF_ID_NUNIQUE_180_ASSUNTO_OTHER",
    "CALL_IF_ID_NUNIQUE_180_ASSUNTO_TV",
    "CALL_IF_ID_NUNIQUE_180_ASSUNTO_VF",
    "CALL_IF_ID_NUNIQUE_180_TOTAL",
    "CALL_IF_ID_NUNIQUE_365_ASSUNTO_IF",
    "CALL_IF_ID_NUNIQUE_365_ASSUNTO_MOVEL",
    "CALL_IF_ID_NUNIQUE_365_ASSUNTO_OTHER",
    "CALL_IF_ID_NUNIQUE_365_ASSUNTO_TV",
    "CALL_IF_ID_NUNIQUE_365_ASSUNTO_VF",
    "CALL_IF_ID_NUNIQUE_365_TOTAL"
]

# For each metric in call_count_columns, Create a new flag column, where 0 indicates a missing value and 1 indicates a non-missing value
for col in call_count_columns:
    flag_col = f"Flag_{col}"
    client_df[flag_col] = np.where((client_df[col].isnull()) | (client_df[col] == 0), 0, 1) 



In [53]:
# --- Ensure relevant columns exist and are numeric (0/1) ---
subscription_cols_individual = ['IF_PROD_FLG', 'TV_PROD_FLG', 'VF_PROD_FLG']
subscription_cols_mobile = ['VM_PROD_FLG', 'IM_PROD_FLG']
all_subscription_cols_needed = subscription_cols_individual + subscription_cols_mobile

print("Checking and converting subscription columns...")
for col in all_subscription_cols_needed:
    if col in client_df.columns:
        # Ensure boolean True/False become 1/0, handle NaNs
        if client_df[col].dtype == 'bool' or client_df[col].dtype == 'object':
             # Convert potential string 'True'/'False' or boolean True/False
             client_df[col] = client_df[col].replace({'True': 1, 'False': 0, True: 1, False: 0})
        # Fill NaNs with 0 and ensure integer type
        client_df[col] = pd.to_numeric(client_df[col], errors='coerce').fillna(0).astype(int)
        print(f"Processed column: {col}")
    else:
        print(f"Warning: Subscription column {col} not found.")

# --- Calculate Total Subscribed Service Categories (Denominator) ---
print("\nCalculating total subscribed service categories...")

# Start with non-mobile categories that exist
active_individual_subs = [col for col in subscription_cols_individual if col in client_df.columns]
client_df['Num_Subscribed_Service_Categories'] = client_df[active_individual_subs].sum(axis=1)

# Check which mobile columns exist
has_vm = 'VM_PROD_FLG' in client_df.columns
has_im = 'IM_PROD_FLG' in client_df.columns

# Create a boolean Series indicating if *any* mobile service is subscribed to
subscribes_to_mobile = pd.Series(False, index=client_df.index)
if has_vm:
    subscribes_to_mobile = subscribes_to_mobile | (client_df['VM_PROD_FLG'] == 1)
if has_im:
    subscribes_to_mobile = subscribes_to_mobile | (client_df['IM_PROD_FLG'] == 1)

# Add 1 to the count if the customer subscribes to the Mobile category
client_df['Num_Subscribed_Service_Categories'] += subscribes_to_mobile.astype(int)
print("Denominator 'Num_Subscribed_Service_Categories' calculated.")
# print(client_df['Num_Subscribed_Service_Categories'].value_counts())


# --- Calculate Number of Subscribed Categories with Issues (Numerator) ---
print("\nCalculating number of subscribed categories with issues...")

# Define mappings and check flag existence
service_map_individual_180 = {
    'IF_PROD_FLG': 'Flag_CALL_IF_ID_NUNIQUE_180_ASSUNTO_IF',
    'TV_PROD_FLG': 'Flag_CALL_IF_ID_NUNIQUE_180_ASSUNTO_TV',
    'VF_PROD_FLG': 'Flag_CALL_IF_ID_NUNIQUE_180_ASSUNTO_VF',
}
mobile_call_flag_180 = 'Flag_CALL_IF_ID_NUNIQUE_180_ASSUNTO_MOVEL'
active_service_map_individual_180 = {k: v for k, v in service_map_individual_180.items() if k in client_df.columns and v in client_df.columns}
mobile_call_flag_180_exists = mobile_call_flag_180 in client_df.columns

service_map_individual_365 = {
    'IF_PROD_FLG': 'Flag_CALL_IF_ID_NUNIQUE_365_ASSUNTO_IF',
    'TV_PROD_FLG': 'Flag_CALL_IF_ID_NUNIQUE_365_ASSUNTO_TV',
    'VF_PROD_FLG': 'Flag_CALL_IF_ID_NUNIQUE_365_ASSUNTO_VF',
}
mobile_call_flag_365 = 'Flag_CALL_IF_ID_NUNIQUE_365_ASSUNTO_MOVEL'
active_service_map_individual_365 = {k: v for k, v in service_map_individual_365.items() if k in client_df.columns and v in client_df.columns}
mobile_call_flag_365_exists = mobile_call_flag_365 in client_df.columns

# --- 180 Days Numerator ---
client_df['Num_Issue_Service_Categories_180'] = 0
# Add issues for individual service categories
for sub_col, call_flag_col in active_service_map_individual_180.items():
    client_df['Num_Issue_Service_Categories_180'] += (client_df[sub_col] == 1) & (client_df[call_flag_col] == 1)

# Add issue for Mobile category (if subscribed to ANY mobile AND called about mobile)
if mobile_call_flag_180_exists:
    client_df['Num_Issue_Service_Categories_180'] += (subscribes_to_mobile & (client_df[mobile_call_flag_180] == 1))
# Convert boolean result back to integer for the column
client_df['Num_Issue_Service_Categories_180'] = client_df['Num_Issue_Service_Categories_180'].astype(int)
print("Numerator 'Num_Issue_Service_Categories_180' calculated.")
# print(client_df['Num_Issue_Service_Categories_180'].value_counts())

# --- 365 Days Numerator ---
client_df['Num_Issue_Service_Categories_365'] = 0
# Add issues for individual service categories
for sub_col, call_flag_col in active_service_map_individual_365.items():
    client_df['Num_Issue_Service_Categories_365'] += (client_df[sub_col] == 1) & (client_df[call_flag_col] == 1)

# Add issue for Mobile category (if subscribed to ANY mobile AND called about mobile)
if mobile_call_flag_365_exists:
    client_df['Num_Issue_Service_Categories_365'] += (subscribes_to_mobile & (client_df[mobile_call_flag_365] == 1))
# Convert boolean result back to integer for the column
client_df['Num_Issue_Service_Categories_365'] = client_df['Num_Issue_Service_Categories_365'].astype(int)
print("Numerator 'Num_Issue_Service_Categories_365' calculated.")
# print(client_df['Num_Issue_Service_Categories_365'].value_counts())


Checking and converting subscription columns...
Processed column: IF_PROD_FLG
Processed column: TV_PROD_FLG
Processed column: VF_PROD_FLG
Processed column: VM_PROD_FLG
Processed column: IM_PROD_FLG

Calculating total subscribed service categories...
Denominator 'Num_Subscribed_Service_Categories' calculated.

Calculating number of subscribed categories with issues...
Numerator 'Num_Issue_Service_Categories_180' calculated.
Numerator 'Num_Issue_Service_Categories_365' calculated.


In [54]:
# --- Ensure relevant columns exist and are numeric (0/1) ---
subscription_cols_individual = ['IF_PROD_FLG', 'TV_PROD_FLG', 'VF_PROD_FLG']
subscription_cols_mobile = ['VM_PROD_FLG', 'IM_PROD_FLG']
all_subscription_cols_needed = subscription_cols_individual + subscription_cols_mobile

print("Checking and converting subscription columns...")
for col in all_subscription_cols_needed:
    if col in client_df.columns:
        # Ensure boolean True/False become 1/0, handle NaNs
        if client_df[col].dtype == 'bool' or client_df[col].dtype == 'object':
             # Convert potential string 'True'/'False' or boolean True/False
             client_df[col] = client_df[col].replace({'True': 1, 'False': 0, True: 1, False: 0})
        # Fill NaNs with 0 and ensure integer type
        client_df[col] = pd.to_numeric(client_df[col], errors='coerce').fillna(0).astype(int)
        print(f"Processed column: {col}")
    else:
        print(f"Warning: Subscription column {col} not found.")

# --- Calculate Total Subscribed Service Categories (Denominator) ---
print("\nCalculating total subscribed service categories...")

# Start with non-mobile categories that exist
active_individual_subs = [col for col in subscription_cols_individual if col in client_df.columns]
client_df['Num_Subscribed_Service_Categories'] = client_df[active_individual_subs].sum(axis=1)

# Check which mobile columns exist
has_vm = 'VM_PROD_FLG' in client_df.columns
has_im = 'IM_PROD_FLG' in client_df.columns

# Create a boolean Series indicating if *any* mobile service is subscribed to
subscribes_to_mobile = pd.Series(False, index=client_df.index)
if has_vm:
    subscribes_to_mobile = subscribes_to_mobile | (client_df['VM_PROD_FLG'] == 1)
if has_im:
    subscribes_to_mobile = subscribes_to_mobile | (client_df['IM_PROD_FLG'] == 1)

# Add 1 to the count if the customer subscribes to the Mobile category
client_df['Num_Subscribed_Service_Categories'] += subscribes_to_mobile.astype(int)
print("Denominator 'Num_Subscribed_Service_Categories' calculated.")
# print(client_df['Num_Subscribed_Service_Categories'].value_counts())


# --- Calculate Number of Subscribed Categories with Issues (Numerator) ---
print("\nCalculating number of subscribed categories with issues...")

# Define mappings and check flag existence
service_map_individual_180 = {
    'IF_PROD_FLG': 'Flag_CALL_IF_ID_NUNIQUE_180_ASSUNTO_IF',
    'TV_PROD_FLG': 'Flag_CALL_IF_ID_NUNIQUE_180_ASSUNTO_TV',
    'VF_PROD_FLG': 'Flag_CALL_IF_ID_NUNIQUE_180_ASSUNTO_VF',
}
mobile_call_flag_180 = 'Flag_CALL_IF_ID_NUNIQUE_180_ASSUNTO_MOVEL'
active_service_map_individual_180 = {k: v for k, v in service_map_individual_180.items() if k in client_df.columns and v in client_df.columns}
mobile_call_flag_180_exists = mobile_call_flag_180 in client_df.columns

service_map_individual_365 = {
    'IF_PROD_FLG': 'Flag_CALL_IF_ID_NUNIQUE_365_ASSUNTO_IF',
    'TV_PROD_FLG': 'Flag_CALL_IF_ID_NUNIQUE_365_ASSUNTO_TV',
    'VF_PROD_FLG': 'Flag_CALL_IF_ID_NUNIQUE_365_ASSUNTO_VF',
}
mobile_call_flag_365 = 'Flag_CALL_IF_ID_NUNIQUE_365_ASSUNTO_MOVEL'
active_service_map_individual_365 = {k: v for k, v in service_map_individual_365.items() if k in client_df.columns and v in client_df.columns}
mobile_call_flag_365_exists = mobile_call_flag_365 in client_df.columns

# --- 180 Days Numerator ---
client_df['Num_Issue_Service_Categories_180'] = 0
# Add issues for individual service categories
for sub_col, call_flag_col in active_service_map_individual_180.items():
    client_df['Num_Issue_Service_Categories_180'] += (client_df[sub_col] == 1) & (client_df[call_flag_col] == 1)

# Add issue for Mobile category (if subscribed to ANY mobile AND called about mobile)
if mobile_call_flag_180_exists:
    client_df['Num_Issue_Service_Categories_180'] += (subscribes_to_mobile & (client_df[mobile_call_flag_180] == 1))
# Convert boolean result back to integer for the column
client_df['Num_Issue_Service_Categories_180'] = client_df['Num_Issue_Service_Categories_180'].astype(int)
print("Numerator 'Num_Issue_Service_Categories_180' calculated.")
# print(client_df['Num_Issue_Service_Categories_180'].value_counts())

# --- 365 Days Numerator ---
client_df['Num_Issue_Service_Categories_365'] = 0
# Add issues for individual service categories
for sub_col, call_flag_col in active_service_map_individual_365.items():
    client_df['Num_Issue_Service_Categories_365'] += (client_df[sub_col] == 1) & (client_df[call_flag_col] == 1)

# Add issue for Mobile category (if subscribed to ANY mobile AND called about mobile)
if mobile_call_flag_365_exists:
    client_df['Num_Issue_Service_Categories_365'] += (subscribes_to_mobile & (client_df[mobile_call_flag_365] == 1))
# Convert boolean result back to integer for the column
client_df['Num_Issue_Service_Categories_365'] = client_df['Num_Issue_Service_Categories_365'].astype(int)
print("Numerator 'Num_Issue_Service_Categories_365' calculated.")
# print(client_df['Num_Issue_Service_Categories_365'].value_counts())


Checking and converting subscription columns...
Processed column: IF_PROD_FLG
Processed column: TV_PROD_FLG
Processed column: VF_PROD_FLG
Processed column: VM_PROD_FLG
Processed column: IM_PROD_FLG

Calculating total subscribed service categories...
Denominator 'Num_Subscribed_Service_Categories' calculated.

Calculating number of subscribed categories with issues...
Numerator 'Num_Issue_Service_Categories_180' calculated.
Numerator 'Num_Issue_Service_Categories_365' calculated.


In [55]:
client_df.head()

,Column1,PERSON_SK,FISCAL_NUM,START_DATE,PERSON_ID,CA_COD,SA_COD,LEG_START_TIME_DAY,DAT_CRI_SA,DT_NASC,CLIENT_AGE_YEARS,CLIENT_ANTIQUITY_YEARS,NR_SAS,ARPU_CALCULATED,SERVICES_QTY,TV_PROD_FLG,IF_PROD_FLG,VF_PROD_FLG,VM_PROD_FLG,IM_PROD_FLG,TECH_COPPER_FLG,TECH_CABLE_FLG,TECH_FTTH_FLG,TECH_DTH_FLG,TECH_GSM_FLG,TV_SPTV_FLG,TV_BTV_FLG,TV_11SPORTS_FLG,TV_TVCINE_FLG,IF_DOWNLOAD_SPEED_MBPS_QTY,IF_UPLOAD_SPEED_KBPS_QTY,FLG_BOX_1_HD_SAT,FLG_BOX_1_HD_PLUS_DVR_SAT_TDT,FLG_BOX_2_HD_CABO,FLG_BOX_2_HD_PLUS_DVR_CABO,FLG_BOX_3_ULTRA_HD,FLG_NO_BOX,FLG_UNKNOWN,FLG_CONSUMER,FLG_EMPRESARIAL,FLG_ILHAS,FLG_INDEFINIDO,ARPU_CLIENT,PF_CLIENT_MONTHS,LEG_DURATION_MEAN_180_ASSUNTO_IF,LEG_DURATION_MEAN_180_ASSUNTO_MOVEL,LEG_DURATION_MEAN_180_ASSUNTO_OTHER,LEG_DURATION_MEAN_180_ASSUNTO_TV,LEG_DURATION_MEAN_180_ASSUNTO_VF,LEG_DURATION_MEAN_180_TOTAL,LEG_DURATION_MEAN_365_ASSUNTO_IF,LEG_DURATION_MEAN_365_ASSUNTO_MOVEL,LEG_DURATION_MEAN_365_ASSUNTO_OTHER,LEG_DURATION_MEAN_365_ASSUNTO_TV,LEG_DURATION_MEAN_365_ASSUNTO_VF,LEG_DURATION_MEAN_365_TOTAL,RPC_MEAN_180_ASSUNTO_IF,RPC_MEAN_180_ASSUNTO_MOVEL,RPC_MEAN_180_ASSUNTO_OTHER,RPC_MEAN_180_ASSUNTO_TV,RPC_MEAN_180_ASSUNTO_VF,RPC_MEAN_180_TOTAL,RPC_MEAN_365_ASSUNTO_IF,RPC_MEAN_365_ASSUNTO_MOVEL,RPC_MEAN_365_ASSUNTO_OTHER,RPC_MEAN_365_ASSUNTO_TV,RPC_MEAN_365_ASSUNTO_VF,RPC_MEAN_365_TOTAL,FTR_CALCULATED_MEAN_180_ASSUNTO_IF,FTR_CALCULATED_MEAN_180_ASSUNTO_MOVEL,FTR_CALCULATED_MEAN_180_ASSUNTO_OTHER,FTR_CALCULATED_MEAN_180_ASSUNTO_TV,FTR_CALCULATED_MEAN_180_ASSUNTO_VF,FTR_CALCULATED_MEAN_180_TOTAL,FTR_CALCULATED_MEAN_365_ASSUNTO_IF,FTR_CALCULATED_MEAN_365_ASSUNTO_MOVEL,FTR_CALCULATED_MEAN_365_ASSUNTO_OTHER,FTR_CALCULATED_MEAN_365_ASSUNTO_TV,FTR_CALCULATED_MEAN_365_ASSUNTO_VF,FTR_CALCULATED_MEAN_365_TOTAL,LEG_IF_ID_COUNT_180_ASSUNTO_IF,LEG_IF_ID_COUNT_180_ASSUNTO_MOVEL,LEG_IF_ID_COUNT_180_ASSUNTO_OTHER,LEG_IF_ID_COUNT_180_ASSUNTO_TV,LEG_IF_ID_COUNT_180_ASSUNTO_VF,LEG_IF_ID_COUNT_180_TOTAL,LEG_IF_ID_COUNT_365_ASSUNTO_IF,LEG_IF_ID_COUNT_365_ASSUNTO_MOVEL,LEG_IF_ID_COUNT_365_ASSUNTO_OTHER,LEG_IF_ID_COUNT_365_ASSUNTO_TV,LEG_IF_ID_COUNT_365_ASSUNTO_VF,LEG_IF_ID_COUNT_365_TOTAL,NR_INCIDENTS_PER_LEG_SUM_180_ASSUNTO_IF,NR_INCIDENTS_PER_LEG_SUM_180_ASSUNTO_MOVEL,NR_INCIDENTS_PER_LEG_SUM_180_ASSUNTO_OTHER,NR_INCIDENTS_PER_LEG_SUM_180_ASSUNTO_TV,NR_INCIDENTS_PER_LEG_SUM_180_ASSUNTO_VF,NR_INCIDENTS_PER_LEG_SUM_180_TOTAL,NR_INCIDENTS_PER_LEG_SUM_365_ASSUNTO_IF,NR_INCIDENTS_PER_LEG_SUM_365_ASSUNTO_MOVEL,NR_INCIDENTS_PER_LEG_SUM_365_ASSUNTO_OTHER,NR_INCIDENTS_PER_LEG_SUM_365_ASSUNTO_TV,NR_INCIDENTS_PER_LEG_SUM_365_ASSUNTO_VF,NR_INCIDENTS_PER_LEG_SUM_365_TOTAL,CALL_IF_ID_NUNIQUE_180_ASSUNTO_IF,CALL_IF_ID_NUNIQUE_180_ASSUNTO_MOVEL,CALL_IF_ID_NUNIQUE_180_ASSUNTO_OTHER,CALL_IF_ID_NUNIQUE_180_ASSUNTO_TV,CALL_IF_ID_NUNIQUE_180_ASSUNTO_VF,CALL_IF_ID_NUNIQUE_180_TOTAL,CALL_IF_ID_NUNIQUE_365_ASSUNTO_IF,CALL_IF_ID_NUNIQUE_365_ASSUNTO_MOVEL,CALL_IF_ID_NUNIQUE_365_ASSUNTO_OTHER,CALL_IF_ID_NUNIQUE_365_ASSUNTO_TV,CALL_IF_ID_NUNIQUE_365_ASSUNTO_VF,CALL_IF_ID_NUNIQUE_365_TOTAL,FLAG_OT_SUM_180_ASSUNTO_IF,FLAG_OT_SUM_180_ASSUNTO_MOVEL,FLAG_OT_SUM_180_ASSUNTO_OTHER,FLAG_OT_SUM_180_ASSUNTO_TV,FLAG_OT_SUM_180_ASSUNTO_VF,FLAG_OT_SUM_180_TOTAL,FLAG_OT_SUM_365_ASSUNTO_IF,FLAG_OT_SUM_365_ASSUNTO_MOVEL,FLAG_OT_SUM_365_ASSUNTO_OTHER,FLAG_OT_SUM_365_ASSUNTO_TV,FLAG_OT_SUM_365_ASSUNTO_VF,FLAG_OT_SUM_365_TOTAL,PROP_CALLS_IF_180,PROP_CALLS_IF_365,PROP_OTS_IF_180,PROP_OTS_IF_365,PROP_CALLS_OTHER_180,PROP_CALLS_OTHER_365,PROP_OTS_OTHER_180,PROP_OTS_OTHER_365,PROP_CALLS_TV_180,PROP_CALLS_TV_365,PROP_OTS_TV_180,PROP_OTS_TV_365,PROP_CALLS_VF_180,PROP_CALLS_VF_365,PROP_OTS_VF_180,PROP_OTS_VF_365,PROP_CALLS_MOVEL_180,PROP_CALLS_MOVEL_365,PROP_OTS_MOVEL_180,PROP_OTS_MOVEL_365,PROP_OTS_TOTAL_180,PROP_OTS_TOTAL_365,Age_missing,Flag_CALL_IF_ID_NUNIQUE_180_ASSUNTO_IF,Flag_CALL_IF_ID_NUNIQUE_180_ASSUNTO_MOVEL,Flag_CALL_IF_ID_NUNIQUE_180_ASSUNTO_OTHER,Flag_CALL_IF_ID_NUNIQUE_180_ASSUNTO_TV,Flag_CALL_IF_ID_NUNIQUE_180_ASSUNTO_VF,Flag_CALL_IF_ID_NUNIQUE_180_TOTAL,Flag_CALL_IF_ID_N

In [56]:
time_frames = ['180', '365']
suffixes = ['IF', 'MOVEL', 'OTHER', 'TV', 'VF']

for tf in time_frames:
    # Fill TOTAL calls as the sum of all CALL_*_ASSUNTO_X columns for the time frame
    call_cols = [f'CALL_IF_ID_NUNIQUE_{tf}_ASSUNTO_{suffix}' for suffix in suffixes]
    total_col = f'CALL_IF_ID_NUNIQUE_{tf}_TOTAL'
    
    client_df[total_col] = client_df[call_cols].sum(axis=1)

    # Fill PROP_CALLS using the respective call column and total column
    for suffix in suffixes:
        call_col = f'CALL_IF_ID_NUNIQUE_{tf}_ASSUNTO_{suffix}'
        prop_col = f'PROP_CALLS_{suffix}_{tf}'
        
        if call_col in client_df.columns:
            client_df[prop_col] = np.where(
                client_df[call_col] > 0,
                client_df[call_col] / client_df[total_col],
                client_df[prop_col]  # keeps original value if condition not met
            )


In [57]:
suffixes = ['IF', 'MOVEL', 'OTHER', 'TV', 'VF']

for suffix in suffixes:
    prop_col = f'PROP_CALLS_{suffix}_180'
    call_col = f'CALL_IF_ID_NUNIQUE_180_ASSUNTO_{suffix}'
    flag_col = f'Flag_{call_col}'

    df_to_check = client_df[client_df[prop_col].isna()]

    print(f"--- {suffix} ---")
    print(f"Shape of df_to_check: {df_to_check.shape}")
    print(f"Missing in {call_col}: {df_to_check[call_col].isna().sum()}")
    print(f"Missing in {flag_col}: {df_to_check[flag_col].isna().sum()}\n")


--- IF ---
Shape of df_to_check: (387312, 166)
Missing in CALL_IF_ID_NUNIQUE_180_ASSUNTO_IF: 0
Missing in Flag_CALL_IF_ID_NUNIQUE_180_ASSUNTO_IF: 0

--- MOVEL ---
Shape of df_to_check: (387433, 166)
Missing in CALL_IF_ID_NUNIQUE_180_ASSUNTO_MOVEL: 0
Missing in Flag_CALL_IF_ID_NUNIQUE_180_ASSUNTO_MOVEL: 0

--- OTHER ---
Shape of df_to_check: (387110, 166)
Missing in CALL_IF_ID_NUNIQUE_180_ASSUNTO_OTHER: 0
Missing in Flag_CALL_IF_ID_NUNIQUE_180_ASSUNTO_OTHER: 0

--- TV ---
Shape of df_to_check: (387314, 166)
Missing in CALL_IF_ID_NUNIQUE_180_ASSUNTO_TV: 0
Missing in Flag_CALL_IF_ID_NUNIQUE_180_ASSUNTO_TV: 0

--- VF ---
Shape of df_to_check: (387430, 166)
Missing in CALL_IF_ID_NUNIQUE_180_ASSUNTO_VF: 0
Missing in Flag_CALL_IF_ID_NUNIQUE_180_ASSUNTO_VF: 0



In [ ]:
# Check missing values in client_df
missing_values = client_df.isnull().sum()
missing_values = missing_values[missing_values > 0]
missing_values = missing_values.sort_values(ascending=False)
missing_values

PROP_OTS_VF_180                          1125791
FTR_CALCULATED_MEAN_180_ASSUNTO_VF       1125628
LEG_DURATION_MEAN_180_ASSUNTO_VF         1125620
RPC_MEAN_180_ASSUNTO_VF                  1125620
FTR_CALCULATED_MEAN_365_ASSUNTO_VF       1120865
LEG_DURATION_MEAN_365_ASSUNTO_VF         1120857
PROP_OTS_VF_365                          1120857
RPC_MEAN_365_ASSUNTO_VF                  1120857
PROP_OTS_MOVEL_180                       1120821
LEG_DURATION_MEAN_180_ASSUNTO_MOVEL      1120668
FTR_CALCULATED_MEAN_180_ASSUNTO_MOVEL    1120668
RPC_MEAN_180_ASSUNTO_MOVEL               1120668
PROP_OTS_MOVEL_365                       1115715
FTR_CALCULATED_MEAN_365_ASSUNTO_MOVEL    1115715
RPC_MEAN_365_ASSUNTO_MOVEL               1115715
LEG_DURATION_MEAN_365_ASSUNTO_MOVEL      1115715
PROP_OTS_IF_180                           900727
FTR_CALCULATED_MEAN_180_ASSUNTO_IF        899684
RPC_MEAN_180_ASSUNTO_IF                   899623
LEG_DURATION_MEAN_180_ASSUNTO_IF          899623
PROP_OTS_TV_180     

#### OTs

In [59]:
metrics_180 = [
    'LEG_DURATION_MEAN_180_ASSUNTO_IF',
    'LEG_DURATION_MEAN_180_ASSUNTO_MOVEL',
    'LEG_DURATION_MEAN_180_ASSUNTO_OTHER',
    'LEG_DURATION_MEAN_180_ASSUNTO_TV',
    'LEG_DURATION_MEAN_180_ASSUNTO_VF',
    'LEG_DURATION_MEAN_180_TOTAL',
    'RPC_MEAN_180_ASSUNTO_IF',
    'RPC_MEAN_180_ASSUNTO_MOVEL',
    'RPC_MEAN_180_ASSUNTO_OTHER',
    'RPC_MEAN_180_ASSUNTO_TV',
    'RPC_MEAN_180_ASSUNTO_VF',
    'RPC_MEAN_180_TOTAL',
    'FTR_CALCULATED_MEAN_180_ASSUNTO_IF',
    'FTR_CALCULATED_MEAN_180_ASSUNTO_MOVEL',
    'FTR_CALCULATED_MEAN_180_ASSUNTO_OTHER',
    'FTR_CALCULATED_MEAN_180_ASSUNTO_TV',
    'FTR_CALCULATED_MEAN_180_ASSUNTO_VF',
    'FTR_CALCULATED_MEAN_180_TOTAL'
]

# For the 365-day (longer-term) metrics:
metrics_365 = [
    'LEG_DURATION_MEAN_365_ASSUNTO_IF',
    'LEG_DURATION_MEAN_365_ASSUNTO_MOVEL',
    'LEG_DURATION_MEAN_365_ASSUNTO_OTHER',
    'LEG_DURATION_MEAN_365_ASSUNTO_TV',
    'LEG_DURATION_MEAN_365_ASSUNTO_VF',
    'LEG_DURATION_MEAN_365_TOTAL',
    'RPC_MEAN_365_ASSUNTO_IF',
    'RPC_MEAN_365_ASSUNTO_MOVEL',
    'RPC_MEAN_365_ASSUNTO_OTHER',
    'RPC_MEAN_365_ASSUNTO_TV',
    'RPC_MEAN_365_ASSUNTO_VF',
    'RPC_MEAN_365_TOTAL',
    'FTR_CALCULATED_MEAN_365_ASSUNTO_IF',
    'FTR_CALCULATED_MEAN_365_ASSUNTO_MOVEL',
    'FTR_CALCULATED_MEAN_365_ASSUNTO_OTHER',
    'FTR_CALCULATED_MEAN_365_ASSUNTO_TV',
    'FTR_CALCULATED_MEAN_365_ASSUNTO_VF',
    'FTR_CALCULATED_MEAN_365_TOTAL'
]

additional_metrics = [
    'PROP_CALLS_IF_180', 'PROP_CALLS_IF_365',
    'PROP_OTS_IF_180', 'PROP_OTS_IF_365',
    'PROP_CALLS_OTHER_180', 'PROP_CALLS_OTHER_365',
    'PROP_OTS_OTHER_180', 'PROP_OTS_OTHER_365',
    'PROP_CALLS_TV_180', 'PROP_CALLS_TV_365',
    'PROP_OTS_TV_180', 'PROP_OTS_TV_365',
    'PROP_CALLS_VF_180', 'PROP_CALLS_VF_365',
    'PROP_OTS_VF_180', 'PROP_OTS_VF_365',
    'PROP_CALLS_MOVEL_180', 'PROP_CALLS_MOVEL_365',
    'PROP_OTS_MOVEL_180', 'PROP_OTS_MOVEL_365',
    'PROP_OTS_TOTAL_180', 'PROP_OTS_TOTAL_365'
]


In [60]:
# --- NEW STEP: Fill Remaining NaNs with 0 or -1 ---

print("\nCreating copies for final NaN filling...")
# Create deep copies to avoid modifying the original or each other
client_df_filled_zero = client_df.copy(deep=True)
client_df_filled_minus_one = client_df.copy(deep=True)

# Combine all metric columns that need final filling
all_metrics_to_fill = metrics_180 + metrics_365 + additional_metrics

# Filter the list to only include columns that actually exist in the DataFrame
existing_metrics_to_fill = [col for col in all_metrics_to_fill if col in client_df.columns]
missing_metrics = [col for col in all_metrics_to_fill if col not in client_df.columns]
if missing_metrics:
    print(f"Warning: The following metric columns were not found and will be skipped: {missing_metrics}")


print(f"\nFilling remaining NaNs in {len(existing_metrics_to_fill)} metric columns with 0...")
# Fill remaining NaNs with 0 in the first copy
client_df_filled_zero[existing_metrics_to_fill] = client_df_filled_zero[existing_metrics_to_fill].fillna(0)
print("Filling with 0 complete.")

print(f"\nFilling remaining NaNs in {len(existing_metrics_to_fill)} metric columns with -1...")
# Fill remaining NaNs with -1 in the second copy
client_df_filled_minus_one[existing_metrics_to_fill] = client_df_filled_minus_one[existing_metrics_to_fill].fillna(-1)
print("Filling with -1 complete.")


# --- Verification (Optional but Recommended) ---
print("\nVerifying NaN counts after final filling...")

# Check a sample of columns in both DataFrames
sample_cols_to_check = [
    'LEG_DURATION_MEAN_180_ASSUNTO_IF',
    'RPC_MEAN_180_TOTAL',
    'FTR_CALCULATED_MEAN_365_ASSUNTO_TV',
    'LEG_DURATION_MEAN_365_TOTAL'
]

for col in sample_cols_to_check:
    if col in client_df_filled_zero.columns:
        nan_count_zero = client_df_filled_zero[col].isna().sum()
        nan_count_minus_one = client_df_filled_minus_one[col].isna().sum()
        print(f"NaNs in '{col}' (Zero DF): {nan_count_zero}")
        print(f"NaNs in '{col}' (Minus One DF): {nan_count_minus_one}")
        if nan_count_zero != 0 or nan_count_minus_one != 0:
             print(f"!! Potential issue: NaNs still present in {col} after filling.")
    else:
        print(f"Skipping verification for '{col}' as it was not found.")


# Display sample data from the new dataframes
print("\nSample data from client_df_filled_zero:")
print(client_df_filled_zero[sample_cols_to_check].head())

print("\nSample data from client_df_filled_minus_one:")
print(client_df_filled_minus_one[sample_cols_to_check].head())


Creating copies for final NaN filling...

Filling remaining NaNs in 58 metric columns with 0...
Filling with 0 complete.

Filling remaining NaNs in 58 metric columns with -1...
Filling with -1 complete.

Verifying NaN counts after final filling...
NaNs in 'LEG_DURATION_MEAN_180_ASSUNTO_IF' (Zero DF): 0
NaNs in 'LEG_DURATION_MEAN_180_ASSUNTO_IF' (Minus One DF): 0
NaNs in 'RPC_MEAN_180_TOTAL' (Zero DF): 0
NaNs in 'RPC_MEAN_180_TOTAL' (Minus One DF): 0
NaNs in 'FTR_CALCULATED_MEAN_365_ASSUNTO_TV' (Zero DF): 0
NaNs in 'FTR_CALCULATED_MEAN_365_ASSUNTO_TV' (Minus One DF): 0
NaNs in 'LEG_DURATION_MEAN_365_TOTAL' (Zero DF): 0
NaNs in 'LEG_DURATION_MEAN_365_TOTAL' (Minus One DF): 0

Sample data from client_df_filled_zero:
        LEG_DURATION_MEAN_180_ASSUNTO_IF  RPC_MEAN_180_TOTAL  \
376658                               0.0            0.000000   
456434                               0.0            1.000000   
350973                               0.0            0.777778   
239888              

In [61]:
client_df_filled_zero.shape

(1173226, 166)

In [62]:
# Check missing values in client_df
missing_values = client_df_filled_zero.isnull().sum()             # Count missing values per column
missing_values = missing_values[missing_values > 0]               # Filter to only columns with missing values
missing_values = missing_values.sort_values(ascending=False)      # Sort by descending count
missing_values


PF_CLIENT_MONTHS                      159617
NR_SAS                                 80264
FLG_BOX_2_HD_CABO                      80264
ARPU_CALCULATED                        80264
TECH_COPPER_FLG                        80264
SERVICES_QTY                           80264
TECH_FTTH_FLG                          80264
TECH_DTH_FLG                           80264
TECH_GSM_FLG                           80264
TECH_CABLE_FLG                         80264
TV_BTV_FLG                             80264
TV_11SPORTS_FLG                        80264
TV_TVCINE_FLG                          80264
IF_DOWNLOAD_SPEED_MBPS_QTY             80264
IF_UPLOAD_SPEED_KBPS_QTY               80264
FLG_BOX_1_HD_SAT                       80264
FLG_BOX_1_HD_PLUS_DVR_SAT_TDT          80264
TV_SPTV_FLG                            80264
FLG_INDEFINIDO                         80264
FLG_BOX_2_HD_PLUS_DVR_CABO             80264
FLG_NO_BOX                             80264
FLG_BOX_3_ULTRA_HD                     80264
FLG_CONSUM

In [63]:
# Check for rows where the Prop Calls TV is still NaN and the Call Count is also null
client_df[
    (client_df['PROP_CALLS_TV_180'].isna()) &
    (client_df['CALL_IF_ID_NUNIQUE_180_TOTAL'].isna())
][['CALL_IF_ID_NUNIQUE_180_ASSUNTO_TV','CALL_IF_ID_NUNIQUE_180_ASSUNTO_IF','CALL_IF_ID_NUNIQUE_180_ASSUNTO_VF','CALL_IF_ID_NUNIQUE_180_ASSUNTO_MOVEL','CALL_IF_ID_NUNIQUE_180_ASSUNTO_OTHER','CALL_IF_ID_NUNIQUE_180_TOTAL', 'PROP_CALLS_TV_180']].shape[0]

0

In [64]:
# Check for rows where the Prop Calls TV is still NaN and the Call Count is also null
client_df[
    (client_df['PROP_CALLS_TV_180'].isna()) &
    (client_df['CALL_IF_ID_NUNIQUE_180_TOTAL'].isna())
][['CALL_IF_ID_NUNIQUE_180_ASSUNTO_TV','CALL_IF_ID_NUNIQUE_180_ASSUNTO_IF','CALL_IF_ID_NUNIQUE_180_ASSUNTO_VF','CALL_IF_ID_NUNIQUE_180_ASSUNTO_MOVEL','CALL_IF_ID_NUNIQUE_180_ASSUNTO_OTHER','CALL_IF_ID_NUNIQUE_180_TOTAL', 'PROP_CALLS_TV_180']].shape[0]

0

In [65]:
# Investigate why FLAG_OT_SUM_180_TOTAL is still null   
# Check for rows where the Prop Calls TV is still NaN and the Call Count is also null
client_df_filled_zero[
    (client_df_filled_zero['LEG_IF_ID_COUNT_180_TOTAL'].isna())
][['CALL_IF_ID_NUNIQUE_180_ASSUNTO_TV','CALL_IF_ID_NUNIQUE_180_ASSUNTO_IF','CALL_IF_ID_NUNIQUE_180_ASSUNTO_VF','CALL_IF_ID_NUNIQUE_180_ASSUNTO_MOVEL','CALL_IF_ID_NUNIQUE_180_ASSUNTO_OTHER','CALL_IF_ID_NUNIQUE_180_TOTAL', 'PROP_CALLS_TV_180']].shape

(620, 7)

In [66]:
incident_columns = [
    'NR_INCIDENTS_PER_LEG_SUM_180_ASSUNTO_IF',
    'NR_INCIDENTS_PER_LEG_SUM_180_ASSUNTO_MOVEL',
    'NR_INCIDENTS_PER_LEG_SUM_180_ASSUNTO_OTHER',
    'NR_INCIDENTS_PER_LEG_SUM_180_ASSUNTO_TV',
    'NR_INCIDENTS_PER_LEG_SUM_180_ASSUNTO_VF',
    'NR_INCIDENTS_PER_LEG_SUM_180_TOTAL',
    'NR_INCIDENTS_PER_LEG_SUM_365_ASSUNTO_IF',
    'NR_INCIDENTS_PER_LEG_SUM_365_ASSUNTO_MOVEL',
    'NR_INCIDENTS_PER_LEG_SUM_365_ASSUNTO_OTHER',
    'NR_INCIDENTS_PER_LEG_SUM_365_ASSUNTO_TV',
    'NR_INCIDENTS_PER_LEG_SUM_365_ASSUNTO_VF',
    'NR_INCIDENTS_PER_LEG_SUM_365_TOTAL',
]

In [67]:
# Investigate why FLAG_OT_SUM_180_TOTAL is still null   
# Check for rows where the Prop Calls TV is still NaN and the Call Count is also null
client_df_filled_zero[
    (client_df_filled_zero['NR_INCIDENTS_PER_LEG_SUM_180_TOTAL'].isna())
][incident_columns].head()

,NR_INCIDENTS_PER_LEG_SUM_180_ASSUNTO_IF,NR_INCIDENTS_PER_LEG_SUM_180_ASSUNTO_MOVEL,NR_INCIDENTS_PER_LEG_SUM_180_ASSUNTO_OTHER,NR_INCIDENTS_PER_LEG_SUM_180_ASSUNTO_TV,NR_INCIDENTS_PER_LEG_SUM_180_ASSUNTO_VF,NR_INCIDENTS_PER_LEG_SUM_180_TOTAL,NR_INCIDENTS_PER_LEG_SUM_365_ASSUNTO_IF,NR_INCIDENTS_PER_LEG_SUM_365_ASSUNTO_MOVEL,NR_INCIDENTS_PER_LEG_SUM_365_ASSUNTO_OTHER,NR_INCIDENTS_PER_LEG_SUM_365_ASSUNTO_TV,NR_INCIDENTS_PER_LEG_SUM_365_ASSUNTO_VF,NR_INCIDENTS_PER_LEG_SUM_365_TOTAL
422143,0.0,0.0,0.0,0.0,0.0,NaN,0.0,0.0,0.0,0.0,0.0,NaN
597992,0.0,0.0,0.0,0.0,0.0,NaN,0.0,0.0,0.0,0.0,0.0,NaN
590730,4.0,0.0,0.0,1.0,0.0,NaN,4.0,0.0,0.0,1.0,0.0,NaN
15534,0.0,0.0,0.0,0.0,0.0,NaN,0.0,0.0,0.0,0.0,0.0,NaN
354752,4.0,0.0,0.0,0.0,0.0,NaN,4.0,0.0,0.0,0.0,0.0,NaN


In [68]:
flag_ot_sum_columns = [
    'FLAG_OT_SUM_180_ASSUNTO_IF',
    'FLAG_OT_SUM_180_ASSUNTO_MOVEL',
    'FLAG_OT_SUM_180_ASSUNTO_OTHER',
    'FLAG_OT_SUM_180_ASSUNTO_TV',
    'FLAG_OT_SUM_180_ASSUNTO_VF',
    'FLAG_OT_SUM_180_TOTAL',
    'FLAG_OT_SUM_365_ASSUNTO_IF',
    'FLAG_OT_SUM_365_ASSUNTO_MOVEL',
    'FLAG_OT_SUM_365_ASSUNTO_OTHER',
    'FLAG_OT_SUM_365_ASSUNTO_TV',
    'FLAG_OT_SUM_365_ASSUNTO_VF',
    'FLAG_OT_SUM_365_TOTAL'
]


In [69]:
# Investigate why FLAG_OT_SUM_180_TOTAL is still null   
# Check for rows where the Prop Calls TV is still NaN and the Call Count is also null
client_df_filled_zero[
    (client_df_filled_zero['FLAG_OT_SUM_180_TOTAL'].isna())
][flag_ot_sum_columns].head()

,FLAG_OT_SUM_180_ASSUNTO_IF,FLAG_OT_SUM_180_ASSUNTO_MOVEL,FLAG_OT_SUM_180_ASSUNTO_OTHER,FLAG_OT_SUM_180_ASSUNTO_TV,FLAG_OT_SUM_180_ASSUNTO_VF,FLAG_OT_SUM_180_TOTAL,FLAG_OT_SUM_365_ASSUNTO_IF,FLAG_OT_SUM_365_ASSUNTO_MOVEL,FLAG_OT_SUM_365_ASSUNTO_OTHER,FLAG_OT_SUM_365_ASSUNTO_TV,FLAG_OT_SUM_365_ASSUNTO_VF,FLAG_OT_SUM_365_TOTAL
422143,0.0,0.0,0.0,0.0,0.0,NaN,0.0,0.0,0.0,0.0,0.0,NaN
597992,0.0,0.0,0.0,0.0,0.0,NaN,0.0,0.0,0.0,0.0,0.0,NaN
590730,0.0,0.0,0.0,0.0,0.0,NaN,0.0,0.0,0.0,0.0,0.0,NaN
15534,0.0,0.0,0.0,0.0,0.0,NaN,0.0,0.0,0.0,0.0,0.0,NaN
354752,0.0,0.0,0.0,0.0,0.0,NaN,0.0,0.0,0.0,0.0,0.0,NaN


In [70]:
# Check to see rows where PF_CLIENT_MONTHS is null
client_df_filled_zero[client_df_filled_zero['PF_CLIENT_MONTHS'].isna()].head()

,Column1,PERSON_SK,FISCAL_NUM,START_DATE,PERSON_ID,CA_COD,SA_COD,LEG_START_TIME_DAY,DAT_CRI_SA,DT_NASC,CLIENT_AGE_YEARS,CLIENT_ANTIQUITY_YEARS,NR_SAS,ARPU_CALCULATED,SERVICES_QTY,TV_PROD_FLG,IF_PROD_FLG,VF_PROD_FLG,VM_PROD_FLG,IM_PROD_FLG,TECH_COPPER_FLG,TECH_CABLE_FLG,TECH_FTTH_FLG,TECH_DTH_FLG,TECH_GSM_FLG,TV_SPTV_FLG,TV_BTV_FLG,TV_11SPORTS_FLG,TV_TVCINE_FLG,IF_DOWNLOAD_SPEED_MBPS_QTY,IF_UPLOAD_SPEED_KBPS_QTY,FLG_BOX_1_HD_SAT,FLG_BOX_1_HD_PLUS_DVR_SAT_TDT,FLG_BOX_2_HD_CABO,FLG_BOX_2_HD_PLUS_DVR_CABO,FLG_BOX_3_ULTRA_HD,FLG_NO_BOX,FLG_UNKNOWN,FLG_CONSUMER,FLG_EMPRESARIAL,FLG_ILHAS,FLG_INDEFINIDO,ARPU_CLIENT,PF_CLIENT_MONTHS,LEG_DURATION_MEAN_180_ASSUNTO_IF,LEG_DURATION_MEAN_180_ASSUNTO_MOVEL,LEG_DURATION_MEAN_180_ASSUNTO_OTHER,LEG_DURATION_MEAN_180_ASSUNTO_TV,LEG_DURATION_MEAN_180_ASSUNTO_VF,LEG_DURATION_MEAN_180_TOTAL,LEG_DURATION_MEAN_365_ASSUNTO_IF,LEG_DURATION_MEAN_365_ASSUNTO_MOVEL,LEG_DURATION_MEAN_365_ASSUNTO_OTHER,LEG_DURATION_MEAN_365_ASSUNTO_TV,LEG_DURATION_MEAN_365_ASSUNTO_VF,LEG_DURATION_MEAN_365_TOTAL,RPC_MEAN_180_ASSUNTO_IF,RPC_MEAN_180_ASSUNTO_MOVEL,RPC_MEAN_180_ASSUNTO_OTHER,RPC_MEAN_180_ASSUNTO_TV,RPC_MEAN_180_ASSUNTO_VF,RPC_MEAN_180_TOTAL,RPC_MEAN_365_ASSUNTO_IF,RPC_MEAN_365_ASSUNTO_MOVEL,RPC_MEAN_365_ASSUNTO_OTHER,RPC_MEAN_365_ASSUNTO_TV,RPC_MEAN_365_ASSUNTO_VF,RPC_MEAN_365_TOTAL,FTR_CALCULATED_MEAN_180_ASSUNTO_IF,FTR_CALCULATED_MEAN_180_ASSUNTO_MOVEL,FTR_CALCULATED_MEAN_180_ASSUNTO_OTHER,FTR_CALCULATED_MEAN_180_ASSUNTO_TV,FTR_CALCULATED_MEAN_180_ASSUNTO_VF,FTR_CALCULATED_MEAN_180_TOTAL,FTR_CALCULATED_MEAN_365_ASSUNTO_IF,FTR_CALCULATED_MEAN_365_ASSUNTO_MOVEL,FTR_CALCULATED_MEAN_365_ASSUNTO_OTHER,FTR_CALCULATED_MEAN_365_ASSUNTO_TV,FTR_CALCULATED_MEAN_365_ASSUNTO_VF,FTR_CALCULATED_MEAN_365_TOTAL,LEG_IF_ID_COUNT_180_ASSUNTO_IF,LEG_IF_ID_COUNT_180_ASSUNTO_MOVEL,LEG_IF_ID_COUNT_180_ASSUNTO_OTHER,LEG_IF_ID_COUNT_180_ASSUNTO_TV,LEG_IF_ID_COUNT_180_ASSUNTO_VF,LEG_IF_ID_COUNT_180_TOTAL,LEG_IF_ID_COUNT_365_ASSUNTO_IF,LEG_IF_ID_COUNT_365_ASSUNTO_MOVEL,LEG_IF_ID_COUNT_365_ASSUNTO_OTHER,LEG_IF_ID_COUNT_365_ASSUNTO_TV,LEG_IF_ID_COUNT_365_ASSUNTO_VF,LEG_IF_ID_COUNT_365_TOTAL,NR_INCIDENTS_PER_LEG_SUM_180_ASSUNTO_IF,NR_INCIDENTS_PER_LEG_SUM_180_ASSUNTO_MOVEL,NR_INCIDENTS_PER_LEG_SUM_180_ASSUNTO_OTHER,NR_INCIDENTS_PER_LEG_SUM_180_ASSUNTO_TV,NR_INCIDENTS_PER_LEG_SUM_180_ASSUNTO_VF,NR_INCIDENTS_PER_LEG_SUM_180_TOTAL,NR_INCIDENTS_PER_LEG_SUM_365_ASSUNTO_IF,NR_INCIDENTS_PER_LEG_SUM_365_ASSUNTO_MOVEL,NR_INCIDENTS_PER_LEG_SUM_365_ASSUNTO_OTHER,NR_INCIDENTS_PER_LEG_SUM_365_ASSUNTO_TV,NR_INCIDENTS_PER_LEG_SUM_365_ASSUNTO_VF,NR_INCIDENTS_PER_LEG_SUM_365_TOTAL,CALL_IF_ID_NUNIQUE_180_ASSUNTO_IF,CALL_IF_ID_NUNIQUE_180_ASSUNTO_MOVEL,CALL_IF_ID_NUNIQUE_180_ASSUNTO_OTHER,CALL_IF_ID_NUNIQUE_180_ASSUNTO_TV,CALL_IF_ID_NUNIQUE_180_ASSUNTO_VF,CALL_IF_ID_NUNIQUE_180_TOTAL,CALL_IF_ID_NUNIQUE_365_ASSUNTO_IF,CALL_IF_ID_NUNIQUE_365_ASSUNTO_MOVEL,CALL_IF_ID_NUNIQUE_365_ASSUNTO_OTHER,CALL_IF_ID_NUNIQUE_365_ASSUNTO_TV,CALL_IF_ID_NUNIQUE_365_ASSUNTO_VF,CALL_IF_ID_NUNIQUE_365_TOTAL,FLAG_OT_SUM_180_ASSUNTO_IF,FLAG_OT_SUM_180_ASSUNTO_MOVEL,FLAG_OT_SUM_180_ASSUNTO_OTHER,FLAG_OT_SUM_180_ASSUNTO_TV,FLAG_OT_SUM_180_ASSUNTO_VF,FLAG_OT_SUM_180_TOTAL,FLAG_OT_SUM_365_ASSUNTO_IF,FLAG_OT_SUM_365_ASSUNTO_MOVEL,FLAG_OT_SUM_365_ASSUNTO_OTHER,FLAG_OT_SUM_365_ASSUNTO_TV,FLAG_OT_SUM_365_ASSUNTO_VF,FLAG_OT_SUM_365_TOTAL,PROP_CALLS_IF_180,PROP_CALLS_IF_365,PROP_OTS_IF_180,PROP_OTS_IF_365,PROP_CALLS_OTHER_180,PROP_CALLS_OTHER_365,PROP_OTS_OTHER_180,PROP_OTS_OTHER_365,PROP_CALLS_TV_180,PROP_CALLS_TV_365,PROP_OTS_TV_180,PROP_OTS_TV_365,PROP_CALLS_VF_180,PROP_CALLS_VF_365,PROP_OTS_VF_180,PROP_OTS_VF_365,PROP_CALLS_MOVEL_180,PROP_CALLS_MOVEL_365,PROP_OTS_MOVEL_180,PROP_OTS_MOVEL_365,PROP_OTS_TOTAL_180,PROP_OTS_TOTAL_365,Age_missing,Flag_CALL_IF_ID_NUNIQUE_180_ASSUNTO_IF,Flag_CALL_IF_ID_NUNIQUE_180_ASSUNTO_MOVEL,Flag_CALL_IF_ID_NUNIQUE_180_ASSUNTO_OTHER,Flag_CALL_IF_ID_NUNIQUE_180_ASSUNTO_TV,Flag_CALL_IF_ID_NUNIQUE_180_ASSUNTO_VF,Flag_CALL_IF_ID_NUNIQUE_180_TOTAL,Flag_CALL_IF_ID_N

In [71]:
# Check to see rows where PF_CLIENT_MONTHS is null
client_df_filled_zero[client_df_filled_zero['IF_DOWNLOAD_SPEED_MBPS_QTY'].isna()].head()

,Column1,PERSON_SK,FISCAL_NUM,START_DATE,PERSON_ID,CA_COD,SA_COD,LEG_START_TIME_DAY,DAT_CRI_SA,DT_NASC,CLIENT_AGE_YEARS,CLIENT_ANTIQUITY_YEARS,NR_SAS,ARPU_CALCULATED,SERVICES_QTY,TV_PROD_FLG,IF_PROD_FLG,VF_PROD_FLG,VM_PROD_FLG,IM_PROD_FLG,TECH_COPPER_FLG,TECH_CABLE_FLG,TECH_FTTH_FLG,TECH_DTH_FLG,TECH_GSM_FLG,TV_SPTV_FLG,TV_BTV_FLG,TV_11SPORTS_FLG,TV_TVCINE_FLG,IF_DOWNLOAD_SPEED_MBPS_QTY,IF_UPLOAD_SPEED_KBPS_QTY,FLG_BOX_1_HD_SAT,FLG_BOX_1_HD_PLUS_DVR_SAT_TDT,FLG_BOX_2_HD_CABO,FLG_BOX_2_HD_PLUS_DVR_CABO,FLG_BOX_3_ULTRA_HD,FLG_NO_BOX,FLG_UNKNOWN,FLG_CONSUMER,FLG_EMPRESARIAL,FLG_ILHAS,FLG_INDEFINIDO,ARPU_CLIENT,PF_CLIENT_MONTHS,LEG_DURATION_MEAN_180_ASSUNTO_IF,LEG_DURATION_MEAN_180_ASSUNTO_MOVEL,LEG_DURATION_MEAN_180_ASSUNTO_OTHER,LEG_DURATION_MEAN_180_ASSUNTO_TV,LEG_DURATION_MEAN_180_ASSUNTO_VF,LEG_DURATION_MEAN_180_TOTAL,LEG_DURATION_MEAN_365_ASSUNTO_IF,LEG_DURATION_MEAN_365_ASSUNTO_MOVEL,LEG_DURATION_MEAN_365_ASSUNTO_OTHER,LEG_DURATION_MEAN_365_ASSUNTO_TV,LEG_DURATION_MEAN_365_ASSUNTO_VF,LEG_DURATION_MEAN_365_TOTAL,RPC_MEAN_180_ASSUNTO_IF,RPC_MEAN_180_ASSUNTO_MOVEL,RPC_MEAN_180_ASSUNTO_OTHER,RPC_MEAN_180_ASSUNTO_TV,RPC_MEAN_180_ASSUNTO_VF,RPC_MEAN_180_TOTAL,RPC_MEAN_365_ASSUNTO_IF,RPC_MEAN_365_ASSUNTO_MOVEL,RPC_MEAN_365_ASSUNTO_OTHER,RPC_MEAN_365_ASSUNTO_TV,RPC_MEAN_365_ASSUNTO_VF,RPC_MEAN_365_TOTAL,FTR_CALCULATED_MEAN_180_ASSUNTO_IF,FTR_CALCULATED_MEAN_180_ASSUNTO_MOVEL,FTR_CALCULATED_MEAN_180_ASSUNTO_OTHER,FTR_CALCULATED_MEAN_180_ASSUNTO_TV,FTR_CALCULATED_MEAN_180_ASSUNTO_VF,FTR_CALCULATED_MEAN_180_TOTAL,FTR_CALCULATED_MEAN_365_ASSUNTO_IF,FTR_CALCULATED_MEAN_365_ASSUNTO_MOVEL,FTR_CALCULATED_MEAN_365_ASSUNTO_OTHER,FTR_CALCULATED_MEAN_365_ASSUNTO_TV,FTR_CALCULATED_MEAN_365_ASSUNTO_VF,FTR_CALCULATED_MEAN_365_TOTAL,LEG_IF_ID_COUNT_180_ASSUNTO_IF,LEG_IF_ID_COUNT_180_ASSUNTO_MOVEL,LEG_IF_ID_COUNT_180_ASSUNTO_OTHER,LEG_IF_ID_COUNT_180_ASSUNTO_TV,LEG_IF_ID_COUNT_180_ASSUNTO_VF,LEG_IF_ID_COUNT_180_TOTAL,LEG_IF_ID_COUNT_365_ASSUNTO_IF,LEG_IF_ID_COUNT_365_ASSUNTO_MOVEL,LEG_IF_ID_COUNT_365_ASSUNTO_OTHER,LEG_IF_ID_COUNT_365_ASSUNTO_TV,LEG_IF_ID_COUNT_365_ASSUNTO_VF,LEG_IF_ID_COUNT_365_TOTAL,NR_INCIDENTS_PER_LEG_SUM_180_ASSUNTO_IF,NR_INCIDENTS_PER_LEG_SUM_180_ASSUNTO_MOVEL,NR_INCIDENTS_PER_LEG_SUM_180_ASSUNTO_OTHER,NR_INCIDENTS_PER_LEG_SUM_180_ASSUNTO_TV,NR_INCIDENTS_PER_LEG_SUM_180_ASSUNTO_VF,NR_INCIDENTS_PER_LEG_SUM_180_TOTAL,NR_INCIDENTS_PER_LEG_SUM_365_ASSUNTO_IF,NR_INCIDENTS_PER_LEG_SUM_365_ASSUNTO_MOVEL,NR_INCIDENTS_PER_LEG_SUM_365_ASSUNTO_OTHER,NR_INCIDENTS_PER_LEG_SUM_365_ASSUNTO_TV,NR_INCIDENTS_PER_LEG_SUM_365_ASSUNTO_VF,NR_INCIDENTS_PER_LEG_SUM_365_TOTAL,CALL_IF_ID_NUNIQUE_180_ASSUNTO_IF,CALL_IF_ID_NUNIQUE_180_ASSUNTO_MOVEL,CALL_IF_ID_NUNIQUE_180_ASSUNTO_OTHER,CALL_IF_ID_NUNIQUE_180_ASSUNTO_TV,CALL_IF_ID_NUNIQUE_180_ASSUNTO_VF,CALL_IF_ID_NUNIQUE_180_TOTAL,CALL_IF_ID_NUNIQUE_365_ASSUNTO_IF,CALL_IF_ID_NUNIQUE_365_ASSUNTO_MOVEL,CALL_IF_ID_NUNIQUE_365_ASSUNTO_OTHER,CALL_IF_ID_NUNIQUE_365_ASSUNTO_TV,CALL_IF_ID_NUNIQUE_365_ASSUNTO_VF,CALL_IF_ID_NUNIQUE_365_TOTAL,FLAG_OT_SUM_180_ASSUNTO_IF,FLAG_OT_SUM_180_ASSUNTO_MOVEL,FLAG_OT_SUM_180_ASSUNTO_OTHER,FLAG_OT_SUM_180_ASSUNTO_TV,FLAG_OT_SUM_180_ASSUNTO_VF,FLAG_OT_SUM_180_TOTAL,FLAG_OT_SUM_365_ASSUNTO_IF,FLAG_OT_SUM_365_ASSUNTO_MOVEL,FLAG_OT_SUM_365_ASSUNTO_OTHER,FLAG_OT_SUM_365_ASSUNTO_TV,FLAG_OT_SUM_365_ASSUNTO_VF,FLAG_OT_SUM_365_TOTAL,PROP_CALLS_IF_180,PROP_CALLS_IF_365,PROP_OTS_IF_180,PROP_OTS_IF_365,PROP_CALLS_OTHER_180,PROP_CALLS_OTHER_365,PROP_OTS_OTHER_180,PROP_OTS_OTHER_365,PROP_CALLS_TV_180,PROP_CALLS_TV_365,PROP_OTS_TV_180,PROP_OTS_TV_365,PROP_CALLS_VF_180,PROP_CALLS_VF_365,PROP_OTS_VF_180,PROP_OTS_VF_365,PROP_CALLS_MOVEL_180,PROP_CALLS_MOVEL_365,PROP_OTS_MOVEL_180,PROP_OTS_MOVEL_365,PROP_OTS_TOTAL_180,PROP_OTS_TOTAL_365,Age_missing,Flag_CALL_IF_ID_NUNIQUE_180_ASSUNTO_IF,Flag_CALL_IF_ID_NUNIQUE_180_ASSUNTO_MOVEL,Flag_CALL_IF_ID_NUNIQUE_180_ASSUNTO_OTHER,Flag_CALL_IF_ID_NUNIQUE_180_ASSUNTO_TV,Flag_CALL_IF_ID_NUNIQUE_180_ASSUNTO_VF,Flag_CALL_IF_ID_NUNIQUE_180_TOTAL,Flag_CALL_IF_ID_N

In [72]:
# Dictionary mapping total columns to the respective category columns
total_mapping = {
    "LEG_IF_ID_COUNT_180_TOTAL": [
        "LEG_IF_ID_COUNT_180_ASSUNTO_MOVEL", 
        "LEG_IF_ID_COUNT_180_ASSUNTO_OTHER", 
        "LEG_IF_ID_COUNT_180_ASSUNTO_IF", 
        "LEG_IF_ID_COUNT_180_ASSUNTO_VF"
    ],
    "LEG_IF_ID_COUNT_365_TOTAL": [
        "LEG_IF_ID_COUNT_365_ASSUNTO_MOVEL", 
        "LEG_IF_ID_COUNT_365_ASSUNTO_OTHER", 
        "LEG_IF_ID_COUNT_365_ASSUNTO_IF", 
        "LEG_IF_ID_COUNT_365_ASSUNTO_VF"
    ],
    "NR_INCIDENTS_PER_LEG_SUM_180_TOTAL": [
        "NR_INCIDENTS_PER_LEG_SUM_180_ASSUNTO_MOVEL", 
        "NR_INCIDENTS_PER_LEG_SUM_180_ASSUNTO_OTHER", 
        "NR_INCIDENTS_PER_LEG_SUM_180_ASSUNTO_IF", 
        "NR_INCIDENTS_PER_LEG_SUM_180_ASSUNTO_VF", 
        "NR_INCIDENTS_PER_LEG_SUM_180_ASSUNTO_TV"
    ],
    "NR_INCIDENTS_PER_LEG_SUM_365_TOTAL": [
        "NR_INCIDENTS_PER_LEG_SUM_365_ASSUNTO_MOVEL", 
        "NR_INCIDENTS_PER_LEG_SUM_365_ASSUNTO_OTHER", 
        "NR_INCIDENTS_PER_LEG_SUM_365_ASSUNTO_IF", 
        "NR_INCIDENTS_PER_LEG_SUM_365_ASSUNTO_VF", 
        "NR_INCIDENTS_PER_LEG_SUM_365_ASSUNTO_TV"
    ],
    "FLAG_OT_SUM_180_TOTAL": [
        "FLAG_OT_SUM_180_ASSUNTO_MOVEL", 
        "FLAG_OT_SUM_180_ASSUNTO_OTHER", 
        "FLAG_OT_SUM_180_ASSUNTO_IF", 
        "FLAG_OT_SUM_180_ASSUNTO_TV"
    ],
    "FLAG_OT_SUM_365_TOTAL": [
        "FLAG_OT_SUM_365_ASSUNTO_MOVEL", 
        "FLAG_OT_SUM_365_ASSUNTO_OTHER", 
        "FLAG_OT_SUM_365_ASSUNTO_IF", 
        "FLAG_OT_SUM_365_ASSUNTO_TV"
    ]
}
 
# Iterate over each mapping entry and fill missing totals with the sum of their category columns.
for total_col, category_cols in total_mapping.items():
    print(f"Filling {total_col} using columns: {category_cols}")
    # Identify rows where the total column is missing (NaN)
    missing_mask = client_df_filled_zero[total_col].isna()
    # Replace missing values with the sum of the specified category columns for those rows
    client_df_filled_zero.loc[missing_mask, total_col] = client_df_filled_zero.loc[missing_mask, category_cols].sum(axis=1)

Filling LEG_IF_ID_COUNT_180_TOTAL using columns: ['LEG_IF_ID_COUNT_180_ASSUNTO_MOVEL', 'LEG_IF_ID_COUNT_180_ASSUNTO_OTHER', 'LEG_IF_ID_COUNT_180_ASSUNTO_IF', 'LEG_IF_ID_COUNT_180_ASSUNTO_VF']
Filling LEG_IF_ID_COUNT_365_TOTAL using columns: ['LEG_IF_ID_COUNT_365_ASSUNTO_MOVEL', 'LEG_IF_ID_COUNT_365_ASSUNTO_OTHER', 'LEG_IF_ID_COUNT_365_ASSUNTO_IF', 'LEG_IF_ID_COUNT_365_ASSUNTO_VF']
Filling NR_INCIDENTS_PER_LEG_SUM_180_TOTAL using columns: ['NR_INCIDENTS_PER_LEG_SUM_180_ASSUNTO_MOVEL', 'NR_INCIDENTS_PER_LEG_SUM_180_ASSUNTO_OTHER', 'NR_INCIDENTS_PER_LEG_SUM_180_ASSUNTO_IF', 'NR_INCIDENTS_PER_LEG_SUM_180_ASSUNTO_VF', 'NR_INCIDENTS_PER_LEG_SUM_180_ASSUNTO_TV']
Filling NR_INCIDENTS_PER_LEG_SUM_365_TOTAL using columns: ['NR_INCIDENTS_PER_LEG_SUM_365_ASSUNTO_MOVEL', 'NR_INCIDENTS_PER_LEG_SUM_365_ASSUNTO_OTHER', 'NR_INCIDENTS_PER_LEG_SUM_365_ASSUNTO_IF', 'NR_INCIDENTS_PER_LEG_SUM_365_ASSUNTO_VF', 'NR_INCIDENTS_PER_LEG_SUM_365_ASSUNTO_TV']
Filling FLAG_OT_SUM_180_TOTAL using columns: ['FLAG_OT_S

In [73]:
# Check missing values in client_df
missing_values = client_df_filled_zero.isnull().sum()             # Count missing values per column
missing_values = missing_values[missing_values > 0]               # Filter to only columns with missing values
missing_values = missing_values.sort_values(ascending=False)      # Sort by descending count
missing_values


PF_CLIENT_MONTHS                 159617
NR_SAS                            80264
SERVICES_QTY                      80264
ARPU_CALCULATED                   80264
TECH_GSM_FLG                      80264
TECH_CABLE_FLG                    80264
TECH_FTTH_FLG                     80264
TECH_DTH_FLG                      80264
TV_BTV_FLG                        80264
TV_SPTV_FLG                       80264
TV_11SPORTS_FLG                   80264
TECH_COPPER_FLG                   80264
FLG_BOX_3_ULTRA_HD                80264
TV_TVCINE_FLG                     80264
IF_UPLOAD_SPEED_KBPS_QTY          80264
IF_DOWNLOAD_SPEED_MBPS_QTY        80264
FLG_BOX_1_HD_PLUS_DVR_SAT_TDT     80264
FLG_BOX_2_HD_CABO                 80264
FLG_BOX_2_HD_PLUS_DVR_CABO        80264
FLG_BOX_1_HD_SAT                  80264
FLG_EMPRESARIAL                   80264
FLG_NO_BOX                        80264
FLG_CONSUMER                      80264
FLG_UNKNOWN                       80264
FLG_INDEFINIDO                    80264


In [74]:
missing_cols = [
    'IF_DOWNLOAD_SPEED_MBPS_QTY',
    'PF_CLIENT_MONTHS',
    'NR_SAS',
    'FLG_EMPRESARIAL',
    'FLG_CONSUMER',
    'FLG_UNKNOWN',
    'FLG_NO_BOX',
    'FLG_BOX_3_ULTRA_HD',
    'FLG_BOX_2_HD_PLUS_DVR_CABO',
    'FLG_BOX_2_HD_CABO',
    'FLG_BOX_1_HD_PLUS_DVR_SAT_TDT',
    'FLG_BOX_1_HD_SAT',
    'IF_UPLOAD_SPEED_KBPS_QTY',
    'TV_TVCINE_FLG',
    'FLG_INDEFINIDO',
    'TV_11SPORTS_FLG',
    'TV_BTV_FLG',
    'TV_SPTV_FLG',
    'TECH_GSM_FLG',
    'TECH_DTH_FLG',
    'TECH_FTTH_FLG',
    'TECH_CABLE_FLG',
    'TECH_COPPER_FLG',
    'SERVICES_QTY',
    'ARPU_CALCULATED',
    'FLG_ILHAS',
    'CLIENT_ANTIQUITY_YEARS',
    'ARPU_CLIENT'
]


In [75]:
# Do forward fill to fill in the rest of the missing values
client_df_filled_zero = client_df_filled_zero.sort_values(['PERSON_ID', 'START_DATE'])
 

for metric in missing_cols:
    client_df_filled_zero[metric] = client_df_filled_zero.groupby('PERSON_ID')[metric].ffill()

In [76]:
# Check missing values in client_df
missing_values = client_df_filled_zero.isnull().sum()             # Count missing values per column
missing_values = missing_values[missing_values > 0]               # Filter to only columns with missing values
missing_values = missing_values.sort_values(ascending=False)      # Sort by descending count
missing_values


PF_CLIENT_MONTHS                 151033
NR_SAS                            79264
SERVICES_QTY                      79264
ARPU_CALCULATED                   79264
TECH_GSM_FLG                      79264
TECH_CABLE_FLG                    79264
TECH_FTTH_FLG                     79264
TECH_DTH_FLG                      79264
TV_BTV_FLG                        79264
TV_SPTV_FLG                       79264
TV_11SPORTS_FLG                   79264
TECH_COPPER_FLG                   79264
FLG_BOX_3_ULTRA_HD                79264
TV_TVCINE_FLG                     79264
IF_UPLOAD_SPEED_KBPS_QTY          79264
IF_DOWNLOAD_SPEED_MBPS_QTY        79264
FLG_BOX_1_HD_PLUS_DVR_SAT_TDT     79264
FLG_BOX_2_HD_CABO                 79264
FLG_BOX_2_HD_PLUS_DVR_CABO        79264
FLG_BOX_1_HD_SAT                  79264
FLG_EMPRESARIAL                   79264
FLG_NO_BOX                        79264
FLG_CONSUMER                      79264
FLG_UNKNOWN                       79264
FLG_INDEFINIDO                    79264


In [77]:
# Create flag column for where PF_CLIENT_MONTHS is missing
missing_cols = [
    'IF_DOWNLOAD_SPEED_MBPS_QTY',
    'PF_CLIENT_MONTHS',
    'NR_SAS',
    'FLG_EMPRESARIAL',
    'FLG_CONSUMER',
    'FLG_UNKNOWN',
    'FLG_NO_BOX',
    'FLG_BOX_3_ULTRA_HD',
    'FLG_BOX_2_HD_PLUS_DVR_CABO',
    'FLG_BOX_2_HD_CABO',
    'FLG_BOX_1_HD_PLUS_DVR_SAT_TDT',
    'FLG_BOX_1_HD_SAT',
    'IF_UPLOAD_SPEED_KBPS_QTY',
    'TV_TVCINE_FLG',
    'FLG_INDEFINIDO',
    'TV_11SPORTS_FLG',
    'TV_BTV_FLG',
    'TV_SPTV_FLG',
    'TECH_GSM_FLG',
    'TECH_DTH_FLG',
    'TECH_FTTH_FLG',
    'TECH_CABLE_FLG',
    'TECH_COPPER_FLG',
    'SERVICES_QTY',
    'ARPU_CALCULATED',
    'FLG_ILHAS',
    'CLIENT_ANTIQUITY_YEARS',
    'ARPU_CLIENT'
]

# Check FLG_CONSUMER if it is blank, we take as a proxy for all columns. If this FLG_CONSUMER is blank, create a falg that all charachteristic columns are empty
flag_col = f"Flag_Charachteristics_Empty"
client_df_filled_zero[flag_col] = np.where((client_df_filled_zero['FLG_NO_BOX'].isna()), 1, 0) # Flag column should maybe be one if it is missing, and a 0 if not? my logic was if it was missing it would be a 0


# For each metric in call_count_columns, Create a new flag column, where 0 indicates a missing value and 1 indicates a non-missing value
for col in missing_cols:
    if col in ['PF_CLIENT_MONTHS', 'ARPU_CLIENT', 'CLIENT_ANTIQUITY_YEARS', 'ARPU_CALCULATED']:
      flag_col = f"Flag_{col}"
      client_df_filled_zero[flag_col] = np.where((client_df_filled_zero[col].isna()), 1, 0) # Flag column should maybe be one if it is missing, and a 0 if not? my logic was if it was missing it would be a 0
    client_df_filled_zero[col] = np.where((client_df_filled_zero[col].isna()), -1, client_df_filled_zero[col])


In [78]:
# Check missing values in client_df
missing_values = client_df_filled_zero.isnull().sum()             # Count missing values per column
missing_values = missing_values[missing_values > 0]               # Filter to only columns with missing values
missing_values = missing_values.sort_values(ascending=False)      # Sort by descending count
missing_values


Series([], dtype: int64)

In [ ]:
client_df_clean = client_df_filled_zero.copy()

### Call Data Cleaning

In [80]:
# Check calls_df for missing values
missing_values = call_df.isnull().sum()
missing_values = missing_values[missing_values > 0]
missing_values = missing_values.sort_values(ascending=False)
missing_values

FTR_1_SUM                   870103
TOPIC_INCIDENT_COD_COUNT    870103
FTR_CALCULATED              870103
NR_RPC_0                    254355
NR_RPC_1                    254355
NR_INCIDENTS_PER_LEG        254355
dtype: int64

In [81]:
call_df.shape

(2937781, 27)

In [82]:
# Investigate some of the rows where there are missing vlaues
call_df[call_df['TOPIC_INCIDENT_COD_COUNT'].isnull()].head(10)

,Column1,LEG_IF_ID,PERSON_ID,PERSON_SK,CALL_IF_ID,GROUP_KEY,RESOURCE_KEY,NR_RPC_0,NR_RPC_1,NR_INCIDENTS_PER_LEG,CALL_START_TIME_DAT,CALL_END_TIME_DAT,LEG_START_TIME_DAT,LEG_END_TIME_DAT,LEG_DURATION_SEC_QTY,CALL_DURATION_SEQ_QTY,TOPIC_TIPIFICATION_LVL_1_DSC,TOPIC_TIPIFICATION_LVL_2_DSC,TOPIC_TIPIFICATION_LVL_3_DSC,TOPIC_CLASSIFIC_ENTRY_AT_FT,WORK_ORDER_ID,TIBCO_DIFFUSION_DAT,TOPIC_INCIDENT_COD_COUNT,FTR_1_SUM,FTR_CALCULATED,WORK_ORDER_SCHEDULE_END_DAT,FLAG_OT
5,515472,8605128837,0UTanuzIZ5dPVFExXhtSF+xsWREW0F+qmaJ0wABC1iqM=,0VTHe2dyCX8baypGwUmaTqZUPgChdmYJz+yKGBb//rUM=,1755865115,633770001,14421150001,0.0,0.0,1.0,2024-02-27 09:29:31.000,2024-02-27 09:48:37.000,2024-02-27 09:31:04,2024-02-27 09:38:52,469,1146,,,,OTHER,,,NaN,NaN,NaN,,0
14,515481,8602683857,0HDvzhaXhult1TUgAhWGCD1+RhKFO2dAY047o3LHZPmc=,0VU8UQzQkSAuH2V1hz/qBEkhzWsIYTEW7qGFjsGr3vyU=,1755456799,572970001,12132090114,0.0,0.0,1.0,2024-02-26 12:56:22.000,2024-02-26 13:18:00.000,2024-02-26 13:00:37,2024-02-26 13:04:16,219,1297,,,,OTHER,,,NaN,NaN,NaN,,0
15,515482,8587666969,0cxbCvu7bE5Zw43zb1Fz939sDeURwSqmyqvZAKbRYrQY=,0AE2MWvielecZNBnIywkHI5if22sOBHdOO37YY7q72WE=,1752836503,625990001,14147430001,0.0,0.0,1.0,2024-02-20 10:24:24.000,2024-02-20 10:54:32.000,2024-02-20 10:27:08,2024-02-20 10:54:32,1644,1807,,,,OTHER,,,NaN,NaN,NaN,,0
16,515483,8558950713,0cxbCvu7bE5Zw43zb1Fz939sDeURwSqmyqvZAKbRYrQY=,0AE2MWvielecZNBnIywkHI5if22sOBHdOO37YY7q72WE=,1747897313,499330151,11804370024,1.0,0.0,1.0,2024-02-08 12:24:36.000,2024-02-08 12:54:53.000,2024-02-08 12:26:36,2024-02-08 12:54:53,1696,1817,DESATIVAÇÃO,SERVIÇO BASE,PEDIDO DE CLT,OTHER,,,NaN,NaN,NaN,,0
18,515485,8557715509,0cxbCvu7bE5Zw43zb1Fz939sDeURwSqmyqvZAKbRYrQY=,0AE2MWvielecZNBnIywkHI5if22sOBHdOO37YY7q72WE=,1747691343,588010001,13922370002,0.0,0.0,1.0,2024-02-07 19:54:14.000,2024-02-07 20:07:47.000,2024-02-07 19:55:56,2024-02-07 20:00:00,244,813,,,,OTHER,,,NaN,NaN,NaN,,0
19,515486,8557741677,0cxbCvu7bE5Zw43zb1Fz939sDeURwSqmyqvZAKbRYrQY=,0AE2MWvielecZNBnIywkHI5if22sOBHdOO37YY7q72WE=,1747691343,547380011,14442360001,0.0,0.0,1.0,2024-02-07 19:54:14.000,2024-02-07 20:07:47.000,2024-02-07 20:00:06,2024-02-07 20:07:47,461,813,,,,OTHER,,,NaN,NaN,NaN,,0
22,515489,8558753797,0cxbCvu7bE5Zw43zb1Fz939sDeURwSqmyqvZAKbRYrQY=,0AE2MWvielecZNBnIywkHI5if22sOBHdOO37YY7q72WE=,1747866099,499330151,14302590001,1.0,0.0,1.0,2024-02-08 12:07:13.000,2024-02-08 12:22:10.000,2024-02-08 12:10:06,2024-02-08 12:22:10,724,896,DESATIVAÇÃO,SERVIÇO BASE,PEDIDO DE CLT,OTHER,,,NaN,NaN,NaN,,0
36,515503,8467820361,0UXxO0QccTE/wiWZ3FP2EJXTWhJjoJoQxNOde6z8TltU=,0Tu/jLYt0L6uvJZt1Uf6Nuj0eh2XUK036wNa/vRC71Lc=,1732355365,593230001,14417930002,NaN,NaN,NaN,2024-01-03 12:26:08.000,2024-01-03 12:26:29.000,2024-01-03 12:26:16,2024-01-03 12:26:29,13,21,,,,OTHER,,,NaN,NaN,NaN,,0
38,515505,8562910401,0BmDkjwWRg8GwzFr6FhEsXNtziBB7ZJFUNNhhMlQLJ9Y=,0VnPKheQRBirzcDJh6Z1T6F1s7gpiSzGLQ+cZduyT1ik=,1748583295,593230001,14201200001,NaN,NaN,NaN,2024-02-09 15:39:51.000,2024-02-09 15:41:02.000,2024-02-09 15:40:05,2024-02-09 15:41:02,57,70,,,,OTHER,,,NaN,NaN,NaN,,0
39,515506,8573079009,0wOb/JBeAWBXiuImX9deh1Z3s85Flj5x974aaU9LLYfY=,0U9oPtBgfoi0/Wi43u8I7abauVae8qSrrvEncW36PuXA=,1750327285,631410001,13961960002,0.0,1.0,1.0,2024-02-14 10:46:09.000,2024-02-14 10:55:29.000,2024-02-14 10:47:49,2024-02-14 10:55:29,459,560,INTERVENÇÕES,REAGENDAR INTERVENÇÃO,PEDIDO DE CLT,OTHER,,,NaN,NaN,NaN,,0


In [ ]:
call_df = call_df.dropna()
call_df.drop(columns=['PERSON_ID', 'GROUP_KEY', 'Column1'], inplace=True)

### Merging Datasets

In [84]:
# Normalize the datetime values (set time to midnight) without changing the dtype
call_df['CALL_START_TIME_DAT_normalized'] = pd.to_datetime(call_df['CALL_START_TIME_DAT'], errors='coerce').dt.normalize()
client_df_clean['START_DATE_normalized'] = pd.to_datetime(client_df_clean['START_DATE'], errors='coerce').dt.normalize()


In [86]:
# First, ensure both DataFrames are sorted by their date columns
call_df.sort_values(['CALL_START_TIME_DAT_normalized'], inplace=True)
client_df_clean.sort_values(['START_DATE_normalized'], inplace=True)


In [88]:
for i in call_df.columns:
    if i not in ['RESOURCE_KEY', 'PERSON_SK']:
        call_df.rename(columns={i: f"call_{i}"}, inplace=True)

for i in client_df_clean.columns:
    if i not in ['PERSON_SK']:
        client_df_clean.rename(columns={i: f"client_{i}"}, inplace=True)

for i in gcs_unique.columns:
    if i not in ['RESOURCE_KEY']:
        gcs_unique.rename(columns={i: f"gc_{i}"}, inplace=True)


In [90]:
merged = pd.merge_asof(
    call_df,
    client_df_clean,
    left_on='call_CALL_START_TIME_DAT_normalized',
    right_on='client_START_DATE_normalized',
    by='PERSON_SK',
    direction='backward'
)

In [91]:
# Merge with GCS data on RESOURCE_KEY + Date
full_merged_df = merged.merge(
    gcs_unique, 
    how='left', 
    left_on=['RESOURCE_KEY'], 
    right_on=['RESOURCE_KEY']
)

In [92]:
full_merged_df.shape

(2067678, 238)

In [93]:
# Check for missing values in the merged DataFrame
missing_values = full_merged_df.isnull().sum()
missing_values = missing_values[missing_values > 0]  # Filter to only columns with missing values
missing_values = missing_values.sort_values(ascending=False)  # Sort by descending count
missing_values

gc_OTS_BY_CALL_TV                                    859714
gc_TOTAL_OTS_TV                                      859714
gc_MEAN_TMC_TV                                       859714
gc_MEAN_FTR_TV                                       859714
gc_MEAN_RPC_TV                                       859714
gc_COUNT_CALLS_TV                                    859714
gc_OTS_BY_CALL_VF                                    859714
gc_TOTAL_OTS_VF                                      859714
gc_MEAN_TMC_VF                                       859714
gc_MEAN_FTR_VF                                       859714
gc_MEAN_RPC_VF                                       859714
gc_COUNT_CALLS_VF                                    859714
gc_OTS_BY_CALL_OTHER                                 859714
gc_TOTAL_OTS_OTHER                                   859714
gc_MEAN_TMC_OTHER                                    859714
gc_MEAN_FTR_OTHER                                    859714
gc_MEAN_RPC_OTHER                       

In [94]:
# Drop rows where values are missing...
full_merged_df_filtered = full_merged_df.dropna()
# Check for missing values in the merged DataFrame
missing_values = full_merged_df_filtered.isnull().sum()
missing_values = missing_values[missing_values > 0]  # Filter to only columns with missing values
missing_values = missing_values.sort_values(ascending=False)  # Sort by descending count
missing_values

Series([], dtype: int64)

In [95]:
full_merged_df_filtered.shape

(1159470, 238)

In [ ]:
full_merged_df = full_merged_df_filtered.copy()

## Target variables and Cost

### Create dependent variables

In [6]:
# Filter out rows with call_LEG_DURATION_SEC_QTY above 10800
full_merged_df = full_merged_df[full_merged_df['call_LEG_DURATION_SEC_QTY'] <= 10800]

full_merged_df["call_CALL_START_TIME_DAT"].min(), full_merged_df["call_CALL_START_TIME_DAT"].max()

('2024-01-01 00:05:05.000', '2024-10-21 23:56:03.000')

In [7]:
# Create TMC_dependent
full_merged_df['TMC_dependent'] = full_merged_df['call_LEG_DURATION_SEC_QTY']
# Create FTR_dependent
full_merged_df['FTR_dependent'] = (full_merged_df['call_FTR_CALCULATED'] > 0).astype(int)
# Create OT_dependent
full_merged_df['OT_dependent'] = full_merged_df['call_FLAG_OT']

# Revert all other FTR colums as well
ftr_columns = [col for col in full_merged_df.columns if 'FTR' in col]
for col in ftr_columns:
    full_merged_df[col] = 1 - full_merged_df[col]

In [8]:
# cost calculation
full_merged_df["call_cost"] = full_merged_df["TMC_dependent"] / 60 * 0.35
full_merged_df["tech_cost"]  = full_merged_df["OT_dependent"] * 22
full_merged_df["repeat_cost"] = (1 - full_merged_df["FTR_dependent"]) * (full_merged_df["client_LEG_DURATION_MEAN_365_TOTAL"]/ 60 * 0.35)
full_merged_df["total_cost"]  = full_merged_df["call_cost"] + full_merged_df["tech_cost"] + full_merged_df["repeat_cost"]

## Evaluation_df: new aggregated dataset


The evaluation_df dataset summarizes call performance metrics per agent and topic.
Each row represents a unique combination of:

    RESOURCE_KEY → the individual GC (agent)

    call_TOPIC_TIPIFICATION_LVL_1_DSC → Level 1 topic category

    call_TOPIC_TIPIFICATION_LVL_2_DSC → Level 2 subtopic


*Steps performed*:

1) Removed duplicate calls:
Ensured each call (call_LEG_IF_ID) appears only once in full_merged_df.

2) Grouped by agent and topic:
Created groups for every (RESOURCE_KEY, topic_lvl_1, topic_lvl_2) combination.

3) Aggregated metrics within each group:

    count_calls: total number of calls handled by that GC in that topic

    mean_TMC_dependent: average handling time (TMC) per call

    mean_FTR_dependent: average first-time resolution (FTR) rate per call

    mean_OT_dependent: average on-time metric per call

    mean_total_cost: average cost per call


*Result*:

evaluation_df contains one row per GC–topic pair, summarizing how each GC performs across different call topics. 
This table serves as the foundation for further aggregations and visualizations (e.g., computing per-topic averages across GCs).

In [9]:
# Ensure one row per call (unique call_LEG_ID)
full_merged_df = full_merged_df.drop_duplicates(subset=['call_LEG_IF_ID']).copy()

# Group by the combination of agent and topic levels
group_cols = ['RESOURCE_KEY', 
              'call_TOPIC_TIPIFICATION_LVL_1_DSC', 
              'call_TOPIC_TIPIFICATION_LVL_2_DSC']

# Aggregate metrics
evaluation_df = (
    full_merged_df
    .groupby(group_cols)
    .agg(
        count_calls=('call_LEG_IF_ID', 'count'),
        mean_TMC_dependent=('TMC_dependent', 'mean'),
        mean_FTR_dependent=('FTR_dependent', 'mean'),
        mean_OT_dependent=('OT_dependent', 'mean'),
        mean_total_cost=('total_cost', 'mean')
    )
    .reset_index()
)


In [10]:
evaluation_df['topic_full'] = (
    evaluation_df['call_TOPIC_TIPIFICATION_LVL_1_DSC'].astype(str)
    + " | " +
    evaluation_df['call_TOPIC_TIPIFICATION_LVL_2_DSC'].astype(str)
)


In [11]:
evaluation_df.head()

,RESOURCE_KEY,call_TOPIC_TIPIFICATION_LVL_1_DSC,call_TOPIC_TIPIFICATION_LVL_2_DSC,count_calls,mean_TMC_dependent,mean_FTR_dependent,mean_OT_dependent,mean_total_cost,topic_full
0,11741170013,ADESÃO / ALTERAÇÃO SERVIÇOS,SERVIÇOS PREMIUM / ADICIONAIS,14,729.357143,0.857143,0.0,4.735758,ADESÃO / ALTERAÇÃO SERVIÇOS | SERVIÇOS PREMIUM / ADICIONAIS
1,11741170013,ALTERAÇÃO TITULAR,ADESÃO PRODUTOS EMPRESARIAIS,1,622.000000,1.000000,0.0,3.628333,ALTERAÇÃO TITULAR | ADESÃO PRODUTOS EMPRESARIAIS
2,11741170013,APLICATIVOS,APP MULTIPLATAFORMA - NOS TV,18,752.833333,0.888889,0.0,5.167199,APLICATIVOS | APP MULTIPLATAFORMA - NOS TV
3,11741170013,APLICATIVOS,APP MULTIPLATAFORMA - OUTRAS,4,310.250000,1.000000,0.0,1.809792,APLICATIVOS | APP MULTIPLATAFORMA - OUTRAS
4,11741170013,APLICATIVOS,APP TV,1,2758.000000,1.000000,0.0,16.088333,APLICATIVOS | APP TV


#### Filter out rows where count_calls < 3

In [12]:
evaluation_df["count_calls"].mean()

39.289290400726756

In [13]:
evaluation_df["topic_full"].nunique()

183

In [14]:
evaluation_df = evaluation_df[evaluation_df['count_calls'] >= 3].copy()

In [15]:
evaluation_df["topic_full"].nunique()

102

In [16]:
evaluation_df["count_calls"].mean()


63.36019791094008

In [17]:
topic_perc = (102/183)*100
print(f"Percentage of topics covered: {topic_perc:.2f}%")

Percentage of topics covered: 55.74%


## Add Ratings

In [18]:
def rate_series(series, higher_is_better=True, df=None, topic_col='topic_full'):
    """
    Assign ratings 1–5 based on standardized deviation from the median (σ bands),
    computed separately within each topic.

    Bands (default for lower-is-better metrics like cost):
        < median - 1σ           → 5 (excellent)
        [median - 1σ, -⅓σ)     → 4 (above average)
        [median - ⅓σ, +⅓σ)     → 3 (average)
        [median + ⅓σ, +1σ)     → 2 (below average)
        > median + 1σ           → 1 (poor)
    If higher_is_better=True, the scale is flipped.

    Parameters
    ----------
    series : pd.Series
        The numeric values to rate.
    higher_is_better : bool
        Whether higher values indicate better performance.
    df : pd.DataFrame
        The DataFrame containing the series and topic column.
    topic_col : str
        Name of the topic column (default 'topic_full').

    Returns
    -------
    pd.Series of integer ratings (1–5), same index as input.
    """
    if df is None or topic_col not in df.columns:
        raise ValueError("You must provide the full DataFrame with a 'topic_full' column for topic-level normalization.")

    ratings = pd.Series(index=series.index, dtype="Int64")

    # Compute per-topic z-score-based ratings
    for topic, idx in df.groupby(topic_col).groups.items():
        sub_series = series.loc[idx].dropna()
        if sub_series.empty:
            continue

        mean = sub_series.mean()
        std = sub_series.std(ddof=0)
        if std == 0 or np.isnan(std):
            ratings.loc[sub_series.index] = 3
            continue

        # σ-based cutoffs
        bins = [-np.inf, mean - 1*std, mean - (1/3)*std,
                mean + (1/3)*std, mean + 1*std, np.inf]
        labels = [5, 4, 3, 2, 1] if not higher_is_better else [1, 2, 3, 4, 5]

        topic_ratings = pd.cut(sub_series, bins=bins, labels=labels, include_lowest=True).astype(int)
        ratings.loc[sub_series.index] = topic_ratings

    return ratings.astype(int)


In [19]:
evaluation_df['rating_experience'] = rate_series(
    evaluation_df['count_calls'], higher_is_better=True, df=evaluation_df
)

evaluation_df['rating_FTR'] = rate_series(
    evaluation_df['mean_FTR_dependent'], higher_is_better=True, df=evaluation_df
)

evaluation_df['rating_TMC'] = rate_series(
    evaluation_df['mean_TMC_dependent'], higher_is_better=False, df=evaluation_df
)

evaluation_df['rating_OT'] = rate_series(
    evaluation_df['mean_OT_dependent'], higher_is_better=False, df=evaluation_df
)

evaluation_df['rating_cost'] = rate_series(
    evaluation_df['mean_total_cost'], higher_is_better=False, df=evaluation_df
)


## Models Tested

### Evaluation Re-usable functions

In [20]:
# ============================================================
# 0. MAIN SPLIT (random stratified by GC) – REQUIRED
# ============================================================
def split_gc_stratified(df, test_frac=0.2, seed=42):
    rs = np.random.RandomState(seed)
    parts = []
    for gc, g in df.groupby("RESOURCE_KEY"):
        m = len(g)
        if m <= 1:
            g = g.copy(); g["__split__"] = "train"; parts.append(g); continue
        idx = rs.choice(m, size=max(1, int(test_frac*m)), replace=False)
        g = g.copy(); g["__split__"] = "train"
        g.iloc[idx, g.columns.get_loc("__split__")] = "test"
        parts.append(g)
    out = pd.concat(parts)
    return (
        out[out["__split__"]=="train"].drop(columns="__split__"),
        out[out["__split__"]=="test"].drop(columns="__split__")
    )

# ============================================================
# 1. MATRIX + SIMILARITIES – REQUIRED
# ============================================================
def build_matrix(df):
    return df.pivot_table(
        index="RESOURCE_KEY",
        columns="topic_full",
        values="rating_cost"
    ).fillna(0)

def compute_sims(matrix):
    user_sim = pd.DataFrame(
        cosine_similarity(matrix),
        index=matrix.index,
        columns=matrix.index
    )
    item_sim = pd.DataFrame(
        cosine_similarity(matrix.T),
        index=matrix.columns,
        columns=matrix.columns
    )
    np.fill_diagonal(user_sim.values, 0)
    np.fill_diagonal(item_sim.values, 0)
    return user_sim, item_sim

# ============================================================
# 2. USER + ITEM KNN PREDICTORS – REQUIRED
# ============================================================
def predict_user_rating(gc, topic, user_sim, train_matrix, k=10):
    if gc not in user_sim.index or topic not in train_matrix.columns:
        return None
    sims = user_sim.loc[gc].sort_values(ascending=False).head(k)
    denom = sims.sum()
    if denom == 0: return None
    ratings = train_matrix.loc[sims.index, topic]
    return float((ratings * sims).sum() / denom)

def predict_item_rating(gc_id, topic, item_sim, train_matrix, k=10):
    if gc_id not in train_matrix.index or topic not in train_matrix.columns:
        return None
    user_row = train_matrix.loc[gc_id]
    rated = user_row[user_row > 0]
    if rated.empty:
        return None
    sims = item_sim.loc[topic, rated.index].sort_values(ascending=False).iloc[:k]
    denom = sims.sum()
    if denom == 0:
        return None
    # normalize by denom
    return float(np.dot(sims.values, rated.loc[sims.index].values) / denom)

# ============================================================
# 3. MATRIX FACTORIZATION – REQUIRED
# ============================================================
class MFModelV2:
    def __init__(self, P, Q, user_ids, item_ids):
        self.P = P
        self.Q = Q
        self.user_ids = user_ids
        self.item_ids = item_ids

    def predict(self, gc, topic):
        if gc not in self.user_ids or topic not in self.item_ids:
            return None
        return float(self.P[self.user_ids[gc]].dot(self.Q[self.item_ids[topic]]))

def train_mf_v2(train_df, n_factors=10, n_epochs=10, lr=0.01, reg=0.02, seed=42):
    users = train_df["RESOURCE_KEY"].unique()
    topics = train_df["topic_full"].unique()
    user_ids = {u:i for i,u in enumerate(users)}
    topic_ids = {t:i for i,t in enumerate(topics)}

    P = 0.1 * np.random.randn(len(users), n_factors)
    Q = 0.1 * np.random.randn(len(topics), n_factors)

    triples = [
        (user_ids[u], topic_ids[t], float(r))
        for u, t, r in train_df[["RESOURCE_KEY", "topic_full", "rating_cost"]].values
    ]

    for _ in range(n_epochs):
        np.random.shuffle(triples)
        for u, i, r in triples:
            pred = P[u].dot(Q[i])
            err = r - pred
            P[u] += lr * (err * Q[i] - reg * P[u])
            Q[i] += lr * (err * P[u] - reg * Q[i])

    return MFModelV2(P, Q, user_ids, topic_ids)

def evaluate_mf_model_v2(train_df, test_df, *args, **kwargs):
    mf_model = train_mf_v2(train_df, *args, **kwargs)
    preds = []
    for _, row in test_df.iterrows():
        gc, topic, true = row["RESOURCE_KEY"], row["topic_full"], float(row["rating_cost"])
        if gc in mf_model.user_ids and topic in mf_model.item_ids:
            preds.append((gc, topic, true, mf_model.predict(gc, topic)))
    return pd.DataFrame(preds, columns=["gc","topic","true","pred"]), mf_model

# ============================================================
# 4. EVALUATE MODEL (POINT PREDICTIONS INTO pred_df)
# ============================================================
def evaluate_model(train_df, test_df, model_name, user_sim, item_sim, mf_model, train_matrix, k=10):
    """
    Produces a DataFrame:
        gc, topic, true, pred
    for all GC–topic pairs in the test set.
    """

    out = []

    for _, row in test_df.iterrows():
        gc = row["RESOURCE_KEY"]
        topic = row["topic_full"]
        true = float(row["rating_cost"])
        pred = None

        if model_name == "user":
            pred = predict_user_rating(gc, topic, user_sim, train_matrix, k)

        elif model_name == "item":
            pred = predict_item_rating(gc, topic, item_sim, train_matrix, k)

        elif model_name == "mf":
            # MF only predicts if both GC + topic appear in training map
            if gc in mf_model.user_ids and topic in mf_model.item_ids:
                pred = mf_model.predict(gc, topic)

        # ignore non-predicted entries
        if pred is None:
            continue

        # clip to rating scale
        pred = float(np.clip(pred, 1.0, 5.0))

        out.append((gc, topic, true, pred))

    return pd.DataFrame(out, columns=["gc", "topic", "true", "pred"])


# ============================================================
# 5. POINTWISE AND PAIRWISE METRICS – REQUIRED
# ============================================================
def metrics_pointwise(pred_df):
    if pred_df.empty: return None, None
    return (
        float(mean_absolute_error(pred_df["true"], pred_df["pred"])),
        float(np.sqrt(mean_squared_error(pred_df["true"], pred_df["pred"])))
    )

def pairwise_accuracy(pred_df):
    scores = []
    for gc, g in pred_df.groupby("gc"):
        rows = g[["true","pred"]].values
        correct = total = 0
        for i in range(len(rows)):
            for j in range(i+1, len(rows)):
                t1, p1 = rows[i]
                t2, p2 = rows[j]
                if t1 == t2: continue
                total += 1
                if (t1 < t2 and p1 < p2) or (t1 > t2 and p1 > p2):
                    correct += 1
        if total > 0:
            scores.append(correct/total)
    return float(np.mean(scores)) if scores else None

# ============================================================
# 6. PREDICT-ALL FUNCTIONS FOR NDCG – REQUIRED
# ============================================================
def _predict_all_user(gc, train_matrix, user_sim):
    if gc not in user_sim.index: return pd.Series()
    sims = user_sim.loc[gc].sort_values(ascending=False).head(10)
    return train_matrix.loc[sims.index].T.dot(sims) / sims.sum()

def _predict_all_item(gc_id, train_matrix, item_sim, k_neighbors=10):
    if gc_id not in train_matrix.index:
        return pd.Series(dtype=float)
    user_row = train_matrix.loc[gc_id]
    rated = user_row[user_row > 0]
    if rated.empty:
        return pd.Series(dtype=float)

    preds = {}
    for topic in train_matrix.columns:
        sims = item_sim.loc[topic, rated.index].sort_values(ascending=False).iloc[:k_neighbors]
        denom = sims.sum()
        preds[topic] = np.nan if denom == 0 else float(np.dot(sims.values, rated.loc[sims.index].values) / denom)
    return pd.Series(preds)


def _predict_all_mf(gc, train_matrix, mf_model):
    if gc not in mf_model.user_ids: return pd.Series()
    return pd.Series({
        t: mf_model.predict(gc, t) for t in train_matrix.columns if t in mf_model.item_ids
    })

# ============================================================
# 7. TOP-K LOW/HIGH FUNCTIONS – REQUIRED
# ============================================================
def top_k_user_low(gc, train_matrix, user_sim, item_sim, mf_model, k=5):
    s = _predict_all_user(gc, train_matrix, user_sim).dropna()
    return list(s.sort_values(ascending=True).head(k).index) if not s.empty else []

def top_k_item_low(gc, train_matrix, user_sim, item_sim, mf_model, k=5):
    s = _predict_all_item(gc, train_matrix, item_sim).dropna()
    return list(s.sort_values(ascending=True).head(k).index) if not s.empty else []

def top_k_mf_low(gc, train_matrix, user_sim, item_sim, mf_model, k=5):
    s = _predict_all_mf(gc, train_matrix, mf_model).dropna()
    return list(s.sort_values(ascending=True).head(k).index) if not s.empty else []

def top_k_user_top(gc, train_matrix, user_sim, item_sim, mf_model, k=5):
    s = _predict_all_user(gc, train_matrix, user_sim).dropna()
    return list(s.sort_values(ascending=False).head(k).index) if not s.empty else []

def top_k_item_top(gc, train_matrix, user_sim, item_sim, mf_model, k=5):
    s = _predict_all_item(gc, train_matrix, item_sim).dropna()
    return list(s.sort_values(ascending=False).head(k).index) if not s.empty else []

def top_k_mf_top(gc, train_matrix, user_sim, item_sim, mf_model, k=5):
    s = _predict_all_mf(gc, train_matrix, mf_model).dropna()
    return list(s.sort_values(ascending=False).head(k).index) if not s.empty else []


In [21]:
# ============================================================
# SPLIT FUNCTIONS
# ============================================================

def split_gc_stratified (df, test_frac=0.2, seed=42):
    """Main split: stratified by GC."""
    rs = np.random.RandomState(seed)
    parts = []
    for gc, g in df.groupby("RESOURCE_KEY"):
        m = len(g)
        if m <= 1:
            g = g.copy(); g["__split__"] = "train"; parts.append(g); continue
        idx = rs.choice(m, size=max(1, int(test_frac*m)), replace=False)
        g = g.copy(); g["__split__"] = "train"
        g.iloc[idx, g.columns.get_loc("__split__")] = "test"
        parts.append(g)
    out = pd.concat(parts)
    return (out[out["__split__"]=="train"].drop(columns="__split__"),
            out[out["__split__"]=="test"].drop(columns="__split__"))


# ---- OPTIONAL: additional splits for robustness ----

def split_topic_stratified(df, test_frac=0.2, seed=42):
    """Stratify by topic."""
    np.random.seed(seed)
    parts = []
    for topic, g in df.groupby("topic_full"):
        m = len(g)
        if m <= 1:
            g = g.copy(); g["__split__"] = "train"; parts.append(g); continue
        idx = np.random.choice(m, size=max(1, int(test_frac*m)), replace=False)
        g = g.copy(); g["__split__"] = "train"
        g.iloc[idx, g.columns.get_loc("__split__")] = "test"
        parts.append(g)
    out = pd.concat(parts)
    return (out[out["__split__"]=="train"].drop(columns="__split__"),
            out[out["__split__"]=="test"].drop(columns="__split__"))


def split_random(df, test_frac=0.2, seed=42):
    """Fully random split: rows sampled uniformly across the entire dataset."""
    rs = np.random.RandomState(seed)
    df = df.copy()
    df["__split__"] = "train"

    m = len(df)
    test_size = max(1, int(test_frac * m))
    idx = rs.choice(m, size=test_size, replace=False)

    df.iloc[idx, df.columns.get_loc("__split__")] = "test"

    return (df[df["__split__"]=="train"].drop(columns="__split__"),
            df[df["__split__"]=="test"].drop(columns="__split__"))



def split_rare_topics(df, rarity_threshold=0.02, test_frac=0.5, seed=42):
    """Stress-test: rare topics go to test."""
    np.random.seed(seed)
    df = df.copy()
    df["__split__"] = "train"

    topic_counts = df["topic_full"].value_counts(normalize=True)
    rare_topics = topic_counts[topic_counts < rarity_threshold].index

    rare_df = df[df["topic_full"].isin(rare_topics)]
    if not rare_df.empty:
        idx = np.random.choice(len(rare_df), size=max(1, int(test_frac*len(rare_df))), replace=False)
        df.loc[rare_df.index[idx], "__split__"] = "test"

    return (df[df["__split__"]=="train"].drop(columns="__split__"),
            df[df["__split__"]=="test"].drop(columns="__split__"))


def split_leave_one_topic(df, seed=42):
    """Leave-one-topic-out per GC."""
    np.random.seed(seed)
    parts = []
    for gc, g in df.groupby("RESOURCE_KEY"):
        m = len(g)
        if m <= 1:
            g = g.copy(); g["__split__"] = "train"; parts.append(g); continue
        idx = np.random.choice(m, size=1, replace=False)
        g = g.copy(); g["__split__"] = "train"
        g.iloc[idx, g.columns.get_loc("__split__")] = "test"
        parts.append(g)
    out = pd.concat(parts)
    return (out[out["__split__"]=="train"].drop(columns="__split__"),
            out[out["__split__"]=="test"].drop(columns="__split__"))


In [22]:
# ================================================================
# 1. GC-LEVEL CLASSIFICATION METRICS (NEW)
# ================================================================

def classify_gc_topics(values):
    """
    Given a vector of ratings for ONE GC (true or predicted),
    returns a dict: topic -> {"low", "mid", "high"} based on GC-specific percentiles.
    """
    if len(values) < 3:
        # not enough topics to compute percentiles
        return {t: "mid" for t in values.index}

    q20 = np.nanpercentile(values, 20)
    q80 = np.nanpercentile(values, 80)

    out = {}
    for t, v in values.items():
        if v <= q20:
            out[t] = "low"
        elif v >= q80:
            out[t] = "high"
        else:
            out[t] = "mid"
    return out


def gc_classification_metrics(pred_df, test_df):
    """
    For each GC:
      - classify TRUE test ratings into low/mid/high
      - classify PREDICTED ratings (for same topics)
      - compare classes → confusion matrix
    Output:
      macro precision, macro recall, macro F1 across GCs.
    """

    gcs = pred_df["gc"].unique()

    precisions, recalls, f1s = [], [], []

    for gc in gcs:
        g_true = test_df[test_df["RESOURCE_KEY"] == gc]
        g_pred = pred_df[pred_df["gc"] == gc]

        # Align topics
        merged = pd.merge(
            g_true, g_pred, left_on="topic_full", right_on="topic",
            how="inner"
        )

        if merged.empty:
            continue

        # true and predicted ratings
        true_r = merged.set_index("topic")["rating_cost"]
        pred_r = merged.set_index("topic")["pred"]

        # classify each into {low,mid,high}
        true_classes = classify_gc_topics(true_r)
        pred_classes = classify_gc_topics(pred_r)

        labels = ["low", "mid", "high"]

        # build confusion counts
        cm = pd.DataFrame(0, index=labels, columns=labels)
        for t in true_classes:
            true_c = true_classes[t]
            pred_c = pred_classes[t]
            cm.loc[true_c, pred_c] += 1

        # compute precision, recall, f1 PER CLASS (macro-average)
        per_class_prec, per_class_rec, per_class_f1 = [], [], []

        for c in labels:
            TP = cm.loc[c, c]
            FP = cm[c].sum() - TP
            FN = cm.loc[c].sum() - TP

            prec = TP / (TP + FP) if (TP + FP) > 0 else np.nan
            rec = TP / (TP + FN) if (TP + FN) > 0 else np.nan
            f1 = (2 * prec * rec / (prec + rec)) if prec and rec and (prec + rec) > 0 else np.nan

            if not np.isnan(prec): per_class_prec.append(prec)
            if not np.isnan(rec): per_class_rec.append(rec)
            if not np.isnan(f1): per_class_f1.append(f1)

        if per_class_prec:
            precisions.append(np.mean(per_class_prec))
        if per_class_rec:
            recalls.append(np.mean(per_class_rec))
        if per_class_f1:
            f1s.append(np.mean(per_class_f1))

    # macro across GCs
    return (
        float(np.mean(precisions)) if precisions else None,
        float(np.mean(recalls)) if recalls else None,
        float(np.mean(f1s)) if f1s else None
    )


In [23]:
# ==============================================================
# NDCG@K for LOW and TOP true relevance
# ==============================================================

def dcg_at_k(relevances):
    """Compute DCG given a list of relevance values in rank order."""
    return sum(rel / np.log2(idx + 2) for idx, rel in enumerate(relevances))


def ndcg_at_k_low(gc, pred_series, test_df, k=5):
    """
    NDCG@K for LOW-performing topics (rating <= 2).
    Higher score = model ranks true weak topics earlier.
    """
    # True relevance: 1 for low-topics, 0 otherwise
    true_lows = set(
        test_df[(test_df["RESOURCE_KEY"] == gc) & (test_df["rating_cost"] <= 2)]["topic_full"]
    )
    if len(true_lows) == 0:
        return None  # no low topics → not evaluatable

    # Predicted ranking (ascending = low first)
    ranked_topics = pred_series.sort_values(ascending=True).index.tolist()

    # relevance vector for top-K predicted
    rel_vector = [1 if t in true_lows else 0 for t in ranked_topics[:k]]

    dcg = dcg_at_k(rel_vector)

    # Ideal DCG: all low topics ranked first
    ideal_rel_vector = sorted(rel_vector, reverse=True)
    idcg = dcg_at_k(ideal_rel_vector)

    return dcg / idcg if idcg > 0 else None


def ndcg_at_k_top(gc, pred_series, test_df, k=5):
    """
    NDCG@K for TOP-performing topics (rating >= 4).
    Checks if model ranks the strong topics early (descending).
    """
    true_tops = set(
        test_df[(test_df["RESOURCE_KEY"] == gc) & (test_df["rating_cost"] >= 4)]["topic_full"]
    )
    if len(true_tops) == 0:
        return None

    # Predicted ranking (descending = high first)
    ranked_topics = pred_series.sort_values(ascending=False).index.tolist()

    rel_vector = [1 if t in true_tops else 0 for t in ranked_topics[:k]]

    dcg = dcg_at_k(rel_vector)

    ideal_rel_vector = sorted(rel_vector, reverse=True)
    idcg = dcg_at_k(ideal_rel_vector)

    return dcg / idcg if idcg > 0 else None


def evaluate_ndcg_metrics(pred_df, train_matrix, user_sim, item_sim, mf_model, test_df, model_name, k=5):
    """
    Wrapper that computes mean NDCG@K_low and NDCG@K_top across all GCs.
    """
    ndcg_low_scores = []
    ndcg_top_scores = []

    # choose prediction generator
    if model_name == "user":
        predict_all = lambda gc: _predict_all_user(gc, train_matrix, user_sim)
    elif model_name == "item":
        predict_all = lambda gc: _predict_all_item(gc, train_matrix, item_sim)
    else:  # mf
        predict_all = lambda gc: _predict_all_mf(gc, train_matrix, mf_model)

    for gc in pred_df["gc"].unique():
        pred_series = predict_all(gc)
        if pred_series is None or pred_series.empty:
            continue

        n_low = ndcg_at_k_low(gc, pred_series, test_df, k)
        n_top = ndcg_at_k_top(gc, pred_series, test_df, k)

        if n_low is not None:
            ndcg_low_scores.append(n_low)
        if n_top is not None:
            ndcg_top_scores.append(n_top)

    return (
        np.mean(ndcg_low_scores) if ndcg_low_scores else None,
        np.mean(ndcg_top_scores) if ndcg_top_scores else None,
    )


In [24]:
# ================================================================
# MASTER EXPERIMENT RUNNER (UPDATED WITH GC-LEVEL CLASSIFICATION + NDCG)
# ================================================================
def run_experiment_v2(split_name, train_df, test_df,
                      n_factors=10, n_epochs=10, lr=0.01, reg=0.02):

    print(f"\n=== Running experiment: {split_name} ===")

    # 1) matrix + similarities
    train_matrix = build_matrix(train_df)
    user_sim, item_sim = compute_sims(train_matrix)

    # 2) train MF
    mf_pred_df, mf_model = evaluate_mf_model_v2(
        train_df, test_df, n_factors, n_epochs, lr, reg
    )

    # 3) model functions
    models = {
        "user": (top_k_user_low, top_k_user_top),
        "item": (top_k_item_low, top_k_item_top),
        "mf":   (top_k_mf_low,   top_k_mf_top),
    }

    results = []

    for model_name, (topk_low, topk_top) in models.items():

        # point predictions
        pred_df = evaluate_model(
            train_df, test_df, model_name,
            user_sim, item_sim, mf_model, train_matrix
        )

        mae, rmse = metrics_pointwise(pred_df)
        pw = pairwise_accuracy(pred_df)

        # GC classification metrics
        class_prec, class_rec, class_f1 = gc_classification_metrics(pred_df, test_df)

        # NEW: NDCG metrics (low and top)
        ndcg_low, ndcg_top = evaluate_ndcg_metrics(
            pred_df, train_matrix, user_sim, item_sim, mf_model, test_df, model_name, k=5
        )

        results.append({
            "split": split_name,
            "model": model_name,
            "MAE": mae,
            "RMSE": rmse,
            "PairwiseAcc": pw,

            "GC_Class_Precision": class_prec,
            "GC_Class_Recall": class_rec,
            "GC_Class_F1": class_f1,

            "NDCG@5_low": ndcg_low,
            "NDCG@5_top": ndcg_top,

            "n_preds": len(pred_df)
        })

    return results


### Original dataset filtered ≥ 3 calls per pair


In [25]:
train_df, test_df = split_gc_stratified(evaluation_df)
results_random = run_experiment_v2("gc_stratified", train_df, test_df)
pd.DataFrame(results_random)



=== Running experiment: gc_stratified ===


,split,model,MAE,RMSE,PairwiseAcc,GC_Class_Precision,GC_Class_Recall,GC_Class_F1,NDCG@5_low,NDCG@5_top,n_preds
0,gc_stratified,user,1.554462,1.887839,0.481857,0.510572,0.497168,0.652030,0.732669,0.493482,3413
1,gc_stratified,item,0.878906,1.125585,0.512634,0.525274,0.502989,0.661875,0.594319,0.544872,3413
2,gc_stratified,mf,0.895327,1.129098,0.516566,0.509372,0.491164,0.657029,NaN,0.601497,3413


In [26]:
# ENSURE THE CORRECT SPLIT
train_df, test_df = split_gc_stratified(evaluation_df)

# NOW RUN TUNING
param_grid = [
    (10, 10),
    (20, 20),
    (40, 40),
    (20, 40),
    (40, 20),
]

all_tuning_results = []

for f, e in param_grid:
    print(f"\nRunning MF tuning: factors={f}, epochs={e}")
    res = run_experiment_v2(
        "gc_strat",
        train_df, test_df,
        n_factors=f,
        n_epochs=e
    )

    mf_row = [r for r in res if r["model"] == "mf"][0]
    all_tuning_results.append(mf_row)

tuning_df = pd.DataFrame(all_tuning_results)
tuning_df



Running MF tuning: factors=10, epochs=10

=== Running experiment: gc_strat ===

Running MF tuning: factors=20, epochs=20

=== Running experiment: gc_strat ===

Running MF tuning: factors=40, epochs=40

=== Running experiment: gc_strat ===

Running MF tuning: factors=20, epochs=40

=== Running experiment: gc_strat ===

Running MF tuning: factors=40, epochs=20

=== Running experiment: gc_strat ===


,split,model,MAE,RMSE,PairwiseAcc,GC_Class_Precision,GC_Class_Recall,GC_Class_F1,NDCG@5_low,NDCG@5_top,n_preds
0,gc_strat,mf,0.898730,1.131283,0.527395,0.516110,0.494416,0.646551,NaN,0.611281,3413
1,gc_strat,mf,0.952519,1.198203,0.528893,0.522935,0.501435,0.655793,0.482820,0.583629,3413
2,gc_strat,mf,0.986535,1.240312,0.508410,0.513850,0.493206,0.653256,0.528377,0.548006,3413
3,gc_strat,mf,1.038112,1.309695,0.501877,0.505712,0.488944,0.647795,0.645505,0.652992,3413
4,gc_strat,mf,0.944060,1.191017,0.526207,0.520057,0.499177,0.652776,NaN,0.546053,3413


### Original dataset filtered ≥ 7 calls per pair

In [27]:
evaluation_df = evaluation_df[evaluation_df['count_calls'] >= 7].copy()

In [28]:
evaluation_df['count_calls'].sum()

1126044

In [29]:
evaluation_df["count_calls"].mean()

95.38703939008894

In [30]:
evaluation_df["topic_full"].nunique()

78

In [31]:
topic_perc = (78/183)*100
print(f"Percentage of topics covered: {topic_perc:.2f}%")

Percentage of topics covered: 42.62%


#### Re-compute Ratings

In [32]:
evaluation_df['rating_experience'] = rate_series(
    evaluation_df['count_calls'], higher_is_better=True, df=evaluation_df
)

evaluation_df['rating_FTR'] = rate_series(
    evaluation_df['mean_FTR_dependent'], higher_is_better=True, df=evaluation_df
)

evaluation_df['rating_TMC'] = rate_series(
    evaluation_df['mean_TMC_dependent'], higher_is_better=False, df=evaluation_df
)

evaluation_df['rating_OT'] = rate_series(
    evaluation_df['mean_OT_dependent'], higher_is_better=False, df=evaluation_df
)

evaluation_df['rating_cost'] = rate_series(
    evaluation_df['mean_total_cost'], higher_is_better=False, df=evaluation_df
)


#### Test Models again

In [33]:
train_df, test_df = split_gc_stratified(evaluation_df)
results_random = run_experiment_v2("gc_stratified", train_df, test_df)
pd.DataFrame(results_random)



=== Running experiment: gc_stratified ===


,split,model,MAE,RMSE,PairwiseAcc,GC_Class_Precision,GC_Class_Recall,GC_Class_F1,NDCG@5_low,NDCG@5_top,n_preds
0,gc_stratified,user,1.658750,2.019057,0.460123,0.619817,0.612820,0.767588,0.641882,0.420319,2186
1,gc_stratified,item,0.872281,1.109777,0.546864,0.617426,0.606063,0.782682,0.587343,0.551962,2186
2,gc_stratified,mf,0.883475,1.119493,0.539226,0.621048,0.606439,0.771912,NaN,0.625396,2186


### Topic Re-aggregation

In [34]:
# --- Step 1. Load both datasets ---
mapping = pd.read_excel("topics_agg.xlsx")
calls = full_merged_df.copy() 

In [35]:
# Load Excel file
mapping_path = "topics_agg.xlsx"
mapping_df = pd.read_excel(mapping_path)

# Normalize column names
L1 = "call_TOPIC_TIPIFICATION_LVL_1_DSC"
L2 = "call_TOPIC_TIPIFICATION_LVL_2_DSC"
L3 = "call_TOPIC_TIPIFICATION_LVL_3_DSC"

# Replace NaN with None for clean tuple keys and targets
mapping_df[[L1, L2, L3]] = mapping_df[[L1, L2, L3]].replace({np.nan: None})
mapping_df[["final_lvl_1", "final_lvl_2", "final_lvl_3"]] = mapping_df[
    ["final_lvl_1", "final_lvl_2", "final_lvl_3"]
].replace({np.nan: None})

# Initialize dictionary
veredict_dict = {"level_1": {}, "level_2": {}, "level_3": {}}

# --- Helper function ---
def build_entry(row):
    """Return dict entry for a given row. For 'in' and 'join to:' store the destination tuple."""
    verdict_raw = str(row["final"]).strip() if pd.notna(row["final"]) else None
    verdict = verdict_raw.lower() if verdict_raw else None

    # destination tuple (even if it's all None, we still store it)
    join_tuple = (row["final_lvl_1"], row["final_lvl_2"], row["final_lvl_3"])

    if verdict is None:
        return {"veredict": None, "join_to": None}

    if verdict.startswith("join to"):
        return {"veredict": "join to:", "join_to": join_tuple}

    if verdict == "in":
        # >>> CHANGE: 'in' also carries the final tuple <<<
        return {"veredict": "in", "join_to": join_tuple}

    # for 'out' and 'delete' (and anything else), no destination
    return {"veredict": verdict_raw, "join_to": None}

# --- LEVEL 1 ---
lvl1_rows = mapping_df[mapping_df[L1].notna() & mapping_df[L2].isna() & mapping_df[L3].isna()]
for _, row in lvl1_rows.iterrows():
    veredict_dict["level_1"][row[L1]] = build_entry(row)

# --- LEVEL 2 ---
lvl2_rows = mapping_df[mapping_df[L1].notna() & mapping_df[L2].notna() & mapping_df[L3].isna()]
for _, row in lvl2_rows.iterrows():
    key = (row[L1], row[L2])
    veredict_dict["level_2"][key] = build_entry(row)

# --- LEVEL 3 ---
lvl3_rows = mapping_df[mapping_df[L1].notna() & mapping_df[L2].notna() & mapping_df[L3].notna()]
for _, row in lvl3_rows.iterrows():
    key = (row[L1], row[L2], row[L3])
    veredict_dict["level_3"][key] = build_entry(row)

# Sanity check
print("✅ Verdict dictionary created successfully")
print(f"Level 1 entries: {len(veredict_dict['level_1'])}")
print(f"Level 2 entries: {len(veredict_dict['level_2'])}")
print(f"Level 3 entries: {len(veredict_dict['level_3'])}")


✅ Verdict dictionary created successfully
Level 1 entries: 51
Level 2 entries: 182
Level 3 entries: 640


In [36]:
# Assumes `calls` exists and has L1/L2/L3 columns
L1 = "call_TOPIC_TIPIFICATION_LVL_1_DSC"
L2 = "call_TOPIC_TIPIFICATION_LVL_2_DSC"
L3 = "call_TOPIC_TIPIFICATION_LVL_3_DSC"

# Normalize None/empty consistently in calls so lookups work
def _none_if_empty(x):
    if pd.isna(x):
        return None
    sx = str(x).strip()
    if sx == "" or sx.lower() == "none":
        return None
    return sx

for c in (L1, L2, L3):
    calls[c] = calls[c].apply(_none_if_empty)

def _join_path(parts):
    """Build 'L1 | L2 | L3' but skip Nones/empties."""
    parts = [p for p in parts if p is not None and str(p).strip() != "" and str(p).strip().lower() != "none"]
    return " | ".join(parts) if parts else None

def _verdict_for_level(level, key):
    """Fetch verdict record dict or None if not present."""
    try:
        return veredict_dict[level].get(key)
    except KeyError:
        return None

def _handle_join_to(rec):
    """Build final_category string from the stored tuple (for both 'in' and 'join to:')."""
    jt = rec.get("join_to") if isinstance(rec, dict) else None
    # If tuple is missing or all None, return None -> caller can fallback
    if not isinstance(jt, (tuple, list)):
        return None
    fc = _join_path(jt)
    return fc if fc else None

def _decide_row(row):
    """
    L1 -> L2 -> L3 cascade:
      - 'delete'  => mark delete
      - 'in'      => build from stored tuple (same as 'join to:')
      - 'join to:'=> build from stored tuple
      - 'out'     => move to next level
    If no rule at a level, behave like 'out' (check next).
    Fallback (no decision at any level): full path 'L1 | L2 | L3'.
    """
    l1, l2, l3 = row[L1], row[L2], row[L3]

    # ----- Level 1 -----
    rec1 = _verdict_for_level("level_1", l1)
    if rec1:
        v1 = (rec1.get("veredict") or "").strip().lower()
        if v1 == "delete":
            return {"delete": True, "final_category": None}
        if v1 in ("in", "join to:") or v1.startswith("join to"):
            fc = _handle_join_to(rec1)
            # If tuple not provided, fallback to current level path name
            return {"delete": False, "final_category": (fc if fc else _join_path((l1,)))}
        # if "out" -> continue

    # ----- Level 2 -----
    rec2 = _verdict_for_level("level_2", (l1, l2))
    if rec2:
        v2 = (rec2.get("veredict") or "").strip().lower()
        if v2 == "delete":
            return {"delete": True, "final_category": None}
        if v2 in ("in", "join to:") or v2.startswith("join to"):
            fc = _handle_join_to(rec2)
            return {"delete": False, "final_category": (fc if fc else _join_path((l1, l2)))}
        # if "out" -> continue

    # ----- Level 3 -----
    rec3 = _verdict_for_level("level_3", (l1, l2, l3))
    if rec3:
        v3 = (rec3.get("veredict") or "").strip().lower()
        if v3 == "delete":
            return {"delete": True, "final_category": None}
        if v3 in ("in", "join to:") or v3.startswith("join to"):
            fc = _handle_join_to(rec3)
            return {"delete": False, "final_category": (fc if fc else _join_path((l1, l2, l3)))}
        # if "out" -> fall through

    # ----- Fallback -----
    return {"delete": False, "final_category": _join_path((l1, l2, l3))}

# Apply
decisions = calls[[L1, L2, L3]].apply(_decide_row, axis=1, result_type="expand")
calls["final_category"] = decisions.apply(lambda d: d["final_category"], axis=1)
to_delete_mask = decisions.apply(lambda d: d["delete"], axis=1).astype(bool)
calls = calls.loc[~to_delete_mask].copy()

# Optional checks
# print(calls["final_category"].isna().mean())
# print(calls["final_category"].head())


In [37]:
# 1) Count calls per final_category
cat_counts = calls["final_category"].value_counts(dropna=False)

# 2) Identify categories to drop (< 500 calls)
threshold = 500
small_cats = cat_counts[cat_counts < threshold].index

# 3) How many rows (calls) will be deleted?
rows_to_delete = int(cat_counts.loc[small_cats].sum()) if len(small_cats) else 0
num_small_categories = int(len(small_cats))

print(f"Categories below {threshold}: {num_small_categories}")
print(f"Rows to delete (calls in those categories): {rows_to_delete}")

# 4) Filter them out
before_rows = len(calls)
before_categories = calls["final_category"].nunique()

calls = calls[~calls["final_category"].isin(small_cats)].copy()

after_rows = len(calls)
after_categories = calls["final_category"].nunique()

print(f"Deleted rows: {before_rows - after_rows}")
print(f"Deleted categories: {before_categories - after_categories}")

# 5) Optional sanity checks
new_counts = calls["final_category"].value_counts()
print("New min count per category:", new_counts.min() if not new_counts.empty else "N/A")

# Optional: show distribution summary after cleanup
print(new_counts.describe())

Categories below 500: 8
Rows to delete (calls in those categories): 795
Deleted rows: 795
Deleted categories: 8
New min count per category: 636
count        76.000000
mean      15338.460526
std       53220.753124
min         636.000000
25%        2119.500000
50%        3566.000000
75%        6700.750000
max      376979.000000
Name: count, dtype: float64


In [38]:
category_counts = calls["final_category"].value_counts().rename_axis("final_category").reset_index(name="count")
category_counts.head(10)

,final_category,count
0,FALHA DE SERVIÇO | INTERNET FIXA | EU PRECISO DE AJUDA COM O MEU SERVIÇO,376979
1,FALHA DE SERVIÇO | TELEVISÃO,275517
2,FALHA DE SERVIÇO | BOX | APARECE UM ALERTA/ERRO,57599
3,FALHA DE SERVIÇO | TV | EU NÃO CONSIGO VER TELEVISÃO,57460
4,FALHA DE SERVIÇO | VOZ FIXA | EU PRECISO DE AJUDA COM O MEU SERVIÇO,47642
5,SEM VOZ DE CLIENTE,27987
6,PRODUTOS E SERVIÇOS | ADESÃO/ALTERAÇÃO | EU QUERO ADERIR/ALTERAR O PRODUTO/SERVIÇO,17264
7,FALHA DE SERVIÇO | INTERNET FIXA | EU NÃO CONSIGO ACEDER À INTERNET,17181
8,FALHA DE SERVIÇO | VOZ MÓVEL | EU NÃO CONSIGO FAZER E/OU RECEBER CHAMADAS,15311
9,FALHA DE SERVIÇO | TV | A TELEVISÃO ESTÁ COM MÁ QUALIDADE IMAGEM/ÁUDIO,14711


#### New Evaluation df

In [39]:
# Group by the combination of agent and topic levels
group_cols = ['RESOURCE_KEY',  
              'final_category']

# Aggregate metrics
evaluation_df = (
    calls
    .groupby(group_cols)
    .agg(
        count_calls=('call_LEG_IF_ID', 'count'),
        mean_TMC_dependent=('TMC_dependent', 'mean'),
        mean_FTR_dependent=('FTR_dependent', 'mean'),
        mean_OT_dependent=('OT_dependent', 'mean'),
        mean_total_cost=('total_cost', 'mean')
    )
    .reset_index()
)

In [40]:
evaluation_df["count_calls"].sum()

1165723

In [41]:
evaluation_df["count_calls"].mean()

32.602164671663495

In [42]:
evaluation_df["final_category"].nunique()

76

In [43]:
evaluation_df = evaluation_df[evaluation_df['count_calls'] >= 7].copy()

In [44]:
evaluation_df["final_category"].nunique()

76

In [45]:
evaluation_df.rename(columns={'final_category': 'topic_full'}, inplace=True)

#### Add Rating Again

In [46]:
evaluation_df['rating_experience'] = rate_series(
    evaluation_df['count_calls'], higher_is_better=True, df=evaluation_df
)

evaluation_df['rating_FTR'] = rate_series(
    evaluation_df['mean_FTR_dependent'], higher_is_better=True, df=evaluation_df
)

evaluation_df['rating_TMC'] = rate_series(
    evaluation_df['mean_TMC_dependent'], higher_is_better=False, df=evaluation_df
)

evaluation_df['rating_OT'] = rate_series(
    evaluation_df['mean_OT_dependent'], higher_is_better=False, df=evaluation_df
)

evaluation_df['rating_cost'] = rate_series(
    evaluation_df['mean_total_cost'], higher_is_better=False, df=evaluation_df
)


#### New matrix Sparsity

In [47]:
R = evaluation_df.pivot_table(
    index="RESOURCE_KEY",
    columns="topic_full",
    values="rating_cost"
)

print(f"Matrix shape: {R.shape}, Sparsity: {R.isna().mean().mean():.2%}")

R_filled = R.fillna(0)

Matrix shape: (636, 76), Sparsity: 65.80%


#### Test Models Again

In [48]:
train_df, test_df = split_gc_stratified(evaluation_df)
results_random = run_experiment_v2("gc_strat", train_df, test_df)
pd.DataFrame(results_random)



=== Running experiment: gc_strat ===


,split,model,MAE,RMSE,PairwiseAcc,GC_Class_Precision,GC_Class_Recall,GC_Class_F1,NDCG@5_low,NDCG@5_top,n_preds
0,gc_strat,user,1.529453,1.866389,0.514743,0.596333,0.587375,0.695715,0.551007,0.479633,3127
1,gc_strat,item,0.900734,1.132452,0.532694,0.586810,0.567126,0.692411,0.531923,0.607159,3127
2,gc_strat,mf,0.897821,1.120240,0.543384,0.581980,0.564386,0.687409,0.544102,0.619445,3127


#### Test Different Splits

In [49]:
# ==========================================================
# SPLIT COLLECTION
# ==========================================================
splits = {
    "gc_strat": split_gc_stratified,
    "random": split_random,
    "topic_strat": split_topic_stratified,
    "rare_topic": split_rare_topics,
    "leave_one_topic": split_leave_one_topic
}


In [50]:
# ==========================================================
# RUN ALL SPLITS
# ==========================================================
all_results = []

for split_name, split_fn in splits.items():
    print(f"\n>>> Running split: {split_name}")

    train_df, test_df = split_fn(evaluation_df)
    split_results = run_experiment_v2(split_name, train_df, test_df)
    all_results.extend(split_results)

results_df = pd.DataFrame(all_results)
results_df



>>> Running split: gc_strat

=== Running experiment: gc_strat ===

>>> Running split: random

=== Running experiment: random ===

>>> Running split: topic_strat

=== Running experiment: topic_strat ===

>>> Running split: rare_topic

=== Running experiment: rare_topic ===

>>> Running split: leave_one_topic

=== Running experiment: leave_one_topic ===


,split,model,MAE,RMSE,PairwiseAcc,GC_Class_Precision,GC_Class_Recall,GC_Class_F1,NDCG@5_low,NDCG@5_top,n_preds
0,gc_strat,user,1.529453,1.866389,0.514743,0.596333,0.587375,0.695715,0.551007,0.479633,3127
1,gc_strat,item,0.900734,1.132452,0.532694,0.586810,0.567126,0.692411,0.531923,0.607159,3127
2,gc_strat,mf,0.898930,1.117978,0.550360,0.591525,0.571769,0.689019,0.527771,0.573467,3127
3,random,user,1.554037,1.895502,0.460292,0.541739,0.527248,0.652562,0.525456,0.454951,3298
4,random,item,0.900172,1.118624,0.530590,0.536387,0.513016,0.650964,0.561921,0.595558,3298
5,random,mf,0.906376,1.120622,0.515324,0.527087,0.510736,0.653330,0.475585,0.591149,3298
6,topic_strat,user,1.537734,1.881444,0.446911,0.513240,0.497157,0.632911,0.607455,0.500168,3274
7,topic_strat,item,0.916431,1.161565,0.514036,0.509607,0.488939,0.648681,0.552263,0.578168,3274
8,topic_strat,mf,0.918785,1.148221,0.516294,0.486201,0.463167,0.636009,0.480484,0.586434,3274
9,rare_topic,user,2.068484,2.413206,0.121227,0.494425,0.489581,0.576941,0.523205,NaN,4191


In [51]:
# ENSURE THE CORRECT SPLIT
train_df, test_df = split_gc_stratified(evaluation_df)

# NOW RUN TUNING
param_grid = [
    (10, 10),
    (20, 20),
    (40, 40),
    (20, 40),
    (40, 20),
]

all_tuning_results = []

for f, e in param_grid:
    print(f"\nRunning MF tuning: factors={f}, epochs={e}")
    res = run_experiment_v2(
        "topic_strat",
        train_df, test_df,
        n_factors=f,
        n_epochs=e
    )

    mf_row = [r for r in res if r["model"] == "mf"][0]
    all_tuning_results.append(mf_row)

tuning_df = pd.DataFrame(all_tuning_results)
tuning_df



Running MF tuning: factors=10, epochs=10

=== Running experiment: topic_strat ===

Running MF tuning: factors=20, epochs=20

=== Running experiment: topic_strat ===

Running MF tuning: factors=40, epochs=40

=== Running experiment: topic_strat ===

Running MF tuning: factors=20, epochs=40

=== Running experiment: topic_strat ===

Running MF tuning: factors=40, epochs=20

=== Running experiment: topic_strat ===


,split,model,MAE,RMSE,PairwiseAcc,GC_Class_Precision,GC_Class_Recall,GC_Class_F1,NDCG@5_low,NDCG@5_top,n_preds
0,topic_strat,mf,0.897260,1.115953,0.562699,0.587005,0.569409,0.691521,0.534424,0.583127,3127
1,topic_strat,mf,0.940008,1.182299,0.564698,0.598363,0.578438,0.701706,0.587037,0.596638,3127
2,topic_strat,mf,0.988094,1.232253,0.550845,0.590242,0.569636,0.688456,0.579748,0.562126,3127
3,topic_strat,mf,1.038604,1.300947,0.547134,0.587810,0.570726,0.696282,0.629488,0.654559,3127
4,topic_strat,mf,0.951092,1.189005,0.556584,0.594383,0.576820,0.698022,0.555294,0.545487,3127


## Final Model Deployment

In [52]:
# ===========================
# ONE-STOP TRAIN+RECOMMEND MF
# ===========================
import numpy as np
import pandas as pd

# ---- Minimal holder for the MF model ----
class MFModelV2:
    def __init__(self, P, Q, user_ids, topic_ids):
        self.P = P
        self.Q = Q
        self.user_ids = user_ids
        self.topic_ids = topic_ids

def train_mf_v2(train_df, user_col, item_col, rating_col,
                n_factors=20, n_epochs=20, lr=0.01, reg=0.02, seed=42):
    rng = np.random.RandomState(seed)

    users = train_df[user_col].astype(object).unique()
    items = train_df[item_col].astype(object).unique()

    user_ids  = {u:i for i,u in enumerate(users)}
    topic_ids = {t:i for i,t in enumerate(items)}

    P = 0.1 * rng.randn(len(users), n_factors)
    Q = 0.1 * rng.randn(len(items), n_factors)

    triples = [
        (user_ids[u], topic_ids[t], float(r))
        for u, t, r in train_df[[user_col, item_col, rating_col]].astype(object).values
    ]

    for _ in range(n_epochs):
        rng.shuffle(triples)
        for u, i, r in triples:
            pred = P[u].dot(Q[i])
            err  = r - pred
            P[u] += lr * (err * Q[i] - reg * P[u])
            Q[i] += lr * (err * P[u] - reg * Q[i])

    return MFModelV2(P, Q, user_ids, topic_ids)

def build_and_recommend_mf(
    evaluation_df,
    gc_id,
    user_col="RESOURCE_KEY",
    topic_col="topic_full",
    rating_col="rating_cost",
    cost_col="mean_total_cost",     # if missing, will fallback to per-topic mean of rating or zeros
    freq_col="count_calls",         # if missing, will fallback to per-topic frequency counts
    n_factors=20,
    n_epochs=20,
    lr=0.01,
    reg=0.02,
    n_recommend=5,
):
    """
    ONE CALL DOES IT ALL:
      1) builds R, costs, frequencies, and cost_by_topic_rating from evaluation_df
      2) trains MF
      3) predicts for unseen topics
      4) merges seen+predicted
      5) computes business priority
      6) prints diagnostics and returns top n rows (DataFrame)
    """

    # ---------- 0) Clean types ----------
    df = evaluation_df.copy()
    df[user_col]  = df[user_col].astype(object)
    df[topic_col] = df[topic_col].astype(object)

    # ---------- 1) Ratings matrix R ----------
    R = df.pivot_table(index=user_col, columns=topic_col, values=rating_col, aggfunc="mean")
    if gc_id not in R.index:
        raise ValueError(f"gc_id {gc_id!r} not found in {user_col}.")
    true_r = R.loc[gc_id]

    # ---------- 2) Business inputs ----------
    # Topic cost
    if cost_col in df.columns:
        topic_cost = df.groupby(topic_col)[cost_col].mean()
    else:
        # Fallback: no explicit cost -> use topic mean rating as proxy; else zero.
        topic_cost = df.groupby(topic_col)[rating_col].mean()

    # Topic frequency
    if freq_col in df.columns:
        topic_frequency = df.groupby(topic_col)[freq_col].mean()
    else:
        topic_frequency = df.groupby(topic_col).size()
    topic_frequency = topic_frequency.reindex(R.columns).fillna(0.0)

    # Cost by topic and (discrete) rating 1..5
    rate_int = np.clip(np.rint(df[rating_col]).astype("Int64"), 1, 5)
    tmp = df.assign(_rint=rate_int)
    if cost_col in df.columns:
        cvals = cost_col
    else:
        cvals = rating_col  # graceful fallback

    cost_by_topic_rating = (
        tmp.groupby([topic_col, "_rint"], observed=True)[cvals]
           .mean()
           .unstack("_rint")
           .reindex(R.columns)
    )

    # ---------- 3) Train MF on observed triples ----------
    mf_model = train_mf_v2(
        train_df=df[[user_col, topic_col, rating_col]],
        user_col=user_col,
        item_col=topic_col,
        rating_col=rating_col,
        n_factors=n_factors,
        n_epochs=n_epochs,
        lr=lr,
        reg=reg,
    )

    # ---------- 4) MF predictor ----------
    def mf_predict(m, u_id, t):
        if u_id not in m.user_ids or t not in m.topic_ids:
            return np.nan
        u = m.user_ids[u_id]
        i = m.topic_ids[t]
        return float(m.P[u].dot(m.Q[i]))

    # ---------- 5) Scoring loop (seen + predicted) ----------
    rows = []
    for topic in R.columns:
        # rating (seen or predicted)
        if pd.notna(true_r[topic]):
            rhat = float(true_r[topic])
            source = "Seen"
        else:
            rhat = mf_predict(mf_model, gc_id, topic)
            source = "Predicted"
        if np.isnan(rhat):
            continue

        rhat_int = int(np.clip(round(rhat), 1, 5))

        # business inputs
        cost = float(topic_cost.get(topic, 0.0))
        freq = float(topic_frequency.get(topic, 0.0))

        C_rating = None
        C_3 = None
        # pull from the cost_by_topic_rating table if available
        if topic in cost_by_topic_rating.index:
            row = cost_by_topic_rating.loc[topic]
            C_rating = row.get(rhat_int, np.nan)
            C_3      = row.get(3, np.nan)

        # fallbacks if sparse
        if pd.isna(C_rating):
            C_rating = cost
        if pd.isna(C_3):
            C_3 = cost

        priority = (C_rating - C_3) * freq

        rows.append({
            "topic_full": topic,
            "rating": rhat,
            "rating_int": rhat_int,
            "source": source,
            "frequency": freq,
            "C_rating": float(C_rating),
            "C_3": float(C_3),
            "priority": float(priority),
        })

    out = pd.DataFrame(rows).sort_values("priority", ascending=False)

    # ---------- 6) Report ----------
    print(f"\n===============================")
    print(f" RECOMMENDATION REPORT – GC {gc_id}")
    print(f"===============================\n")

    print("🎯 Top Training Priorities:")
    for _, row in out.head(n_recommend).iterrows():
        print(f"\n🔸 {row['topic_full']}")
        print(f"   rating = {row['rating']:.2f} ({row['source']}, r̂_int={row['rating_int']})")
        print(f"   C̄(topic, r̂) = {row['C_rating']:.2f} | C̄(topic,3) = {row['C_3']:.2f}")
        print(f"   frequency = {row['frequency']:.2f}")
        print(f"   PRIORITY = {row['priority']:.2f}")

    return out.head(n_recommend), out, R, mf_model, cost_by_topic_rating, topic_cost, topic_frequency


# ===========================
# EXAMPLE CALL (JUST RUN THIS)
# ===========================
gc_example = evaluation_df["RESOURCE_KEY"].iloc[0]  # or set a specific ID you want
top5, full_ranked, R_built, mf_model, c_by_rt, t_cost, t_freq = build_and_recommend_mf(
    evaluation_df=evaluation_df,
    gc_id=gc_example,
    user_col="RESOURCE_KEY",
    topic_col="topic_full",
    rating_col="rating_cost",
    cost_col="mean_total_cost",   # if you don't have this column, it's handled
    freq_col="count_calls",       # if you don't have this column, it's handled
    n_factors=10,
    n_epochs=10,
    lr=0.01,
    reg=0.02,
    n_recommend=5,
)

# Inspect the top 5 table
top5



 RECOMMENDATION REPORT – GC 11741170013

🎯 Top Training Priorities:

🔸 SEM VOZ DE CLIENTE
   rating = 1.00 (Seen, r̂_int=1)
   C̄(topic, r̂) = 7.19 | C̄(topic,3) = 4.12
   frequency = 55.41
   PRIORITY = 169.87

🔸 FALHA DE SERVIÇO | OTHERS
   rating = 1.00 (Seen, r̂_int=1)
   C̄(topic, r̂) = 12.25 | C̄(topic,3) = 6.20
   frequency = 12.48
   PRIORITY = 75.53

🔸 FALHA DE SERVIÇO | INTERNET FIXA | EU NÃO CONSIGO ACEDER À INTERNET
   rating = 2.00 (Seen, r̂_int=2)
   C̄(topic, r̂) = 9.29 | C̄(topic,3) = 7.88
   frequency = 35.53
   PRIORITY = 49.89

🔸 FALHA DE SERVIÇO | INTERNET FIXA | OTHERS
   rating = 2.00 (Seen, r̂_int=2)
   C̄(topic, r̂) = 7.85 | C̄(topic,3) = 6.67
   frequency = 21.46
   PRIORITY = 25.39

🔸 EQUIPAMENTOS | CARTÃO SIM/ESIM/TWIN | EU QUERO UMA 2A VIA
   rating = 2.00 (Seen, r̂_int=2)
   C̄(topic, r̂) = 6.40 | C̄(topic,3) = 5.45
   frequency = 13.33
   PRIORITY = 12.64


,topic_full,rating,rating_int,source,frequency,C_rating,C_3,priority
74,SEM VOZ DE CLIENTE,1.0,1,Seen,55.406814,7.185984,4.120200,169.865323
36,FALHA DE SERVIÇO | OTHERS,1.0,1,Seen,12.482759,12.254127,6.203311,75.530880
30,FALHA DE SERVIÇO | INTERNET FIXA | EU NÃO CONSIGO ACEDER À INTERNET,2.0,2,Seen,35.530655,9.288825,7.884579,49.893773
32,FALHA DE SERVIÇO | INTERNET FIXA | OTHERS,2.0,2,Seen,21.457801,7.848978,6.665906,25.386129
13,EQUIPAMENTOS | CARTÃO SIM/ESIM/TWIN | EU QUERO UMA 2A VIA,2.0,2,Seen,13.334572,6.395088,5.446959,12.642893


In [53]:
top5, full_ranked, R_built, mf_model, c_by_rt, t_cost, t_freq = build_and_recommend_mf(
    evaluation_df=evaluation_df,
    gc_id = evaluation_df["RESOURCE_KEY"].unique()[7],   # your GC id
    n_recommend=5
)



 RECOMMENDATION REPORT – GC 11792920033

🎯 Top Training Priorities:

🔸 FALHA DE SERVIÇO | TELEVISÃO
   rating = 2.00 (Seen, r̂_int=2)
   C̄(topic, r̂) = 12.74 | C̄(topic,3) = 11.42
   frequency = 439.33
   PRIORITY = 582.43

🔸 FALHA DE SERVIÇO | BOX | APARECE UM ALERTA/ERRO
   rating = 1.00 (Seen, r̂_int=1)
   C̄(topic, r̂) = 15.97 | C̄(topic,3) = 11.88
   frequency = 107.89
   PRIORITY = 441.93

🔸 FALHA DE SERVIÇO | TV | A TELEVISÃO ESTÁ COM MÁ QUALIDADE IMAGEM/ÁUDIO
   rating = 1.00 (Seen, r̂_int=1)
   C̄(topic, r̂) = 20.26 | C̄(topic,3) = 15.13
   frequency = 34.56
   PRIORITY = 177.10

🔸 FALHA DE SERVIÇO | INTERNET FIXA | EU NÃO CONSIGO ACEDER À INTERNET
   rating = 1.00 (Seen, r̂_int=1)
   C̄(topic, r̂) = 11.51 | C̄(topic,3) = 7.88
   frequency = 35.53
   PRIORITY = 128.77

🔸 FALHA DE SERVIÇO | GRAVAÇÕES/RESTART TV | OTHERS
   rating = 2.48 (Predicted, r̂_int=2)
   C̄(topic, r̂) = 16.90 | C̄(topic,3) = 13.71
   frequency = 13.60
   PRIORITY = 43.45


In [54]:
full_ranked.head(10)


,topic_full,rating,rating_int,source,frequency,C_rating,C_3,priority
38,FALHA DE SERVIÇO | TELEVISÃO,2.000000,2,Seen,439.330144,12.742706,11.416989,582.427264
23,FALHA DE SERVIÇO | BOX | APARECE UM ALERTA/ERRO,1.000000,1,Seen,107.892857,15.974508,11.878512,441.928691
40,FALHA DE SERVIÇO | TV | A TELEVISÃO ESTÁ COM MÁ QUALIDADE IMAGEM/ÁUDIO,1.000000,1,Seen,34.556627,20.255873,15.131048,177.096662
30,FALHA DE SERVIÇO | INTERNET FIXA | EU NÃO CONSIGO ACEDER À INTERNET,1.000000,1,Seen,35.530655,11.508899,7.884579,128.774459
28,FALHA DE SERVIÇO | GRAVAÇÕES/RESTART TV | OTHERS,2.481436,2,Predicted,13.601990,16.904822,13.710740,43.445879
27,FALHA DE SERVIÇO | GRAVAÇÕES/RESTART TV | EU NÃO CONSIGO VER O PROGRAMA,2.000000,2,Seen,15.020979,15.294736,12.777091,37.817489
44,FALHA DE SERVIÇO | VIDEOCLUBE,2.497464,2,Predicted,14.940000,20.326340,17.954283,35.438539
21,FALHA DE SERVIÇO | BOX | A BOX BLOQUEIA,1.730024,2,Predicted,16.295203,8.955790,7.358053,26.035451
62,PRODUTOS E SERVIÇOS | ADESÃO/ALTERAÇÃO | EU QUERO ALTERAR/CORRIGIR A MORADA DE INSTALAÇÃO,1.000000,1,Seen,11.058511,7.088539,4.935877,23.805237
17,EQUIPAMENTOS | OTHERS,2.277913,2,Predicted,10.800000,3.982648,3.415814,6.121807
